# Reliability- and Uncertainty-Aware Multimodal Learning for Disaster Understanding

*CrisisMMD-based multimodal disaster-tweet understanding with DeBERTa-v3, CLIP ViT-B/32, and an uncertainty-aware fusion pipeline (E-REM → UASG → UG-CMA).*


## Journal Upgrade Notes — read before running

This notebook (`FINAL_CRISISMMD_JOURNAL.ipynb`) is a scientific-rigor upgrade of
`FINAL_CRISISMMD_NOTEBOOK.ipynb`. It keeps the same dataset, split, and architecture
(DeBERTa-v3 + CLIP ViT-B/32 → E-REM → UASG → UG-CMA → shared representation → 5 task heads).

**Important limitation of the environment this upgrade was authored in:** the audit/edit
session had no GPU, no `torch`/`transformers`, no network access to model-weight hosts, and
no copy of the CrisisMMD dataset — only the original `.ipynb` file. Consequently:

- Every **new** cell added for this upgrade (leakage checks, the corrected uncertainty loss,
  extra diagnostics) is **untested code**, clearly marked `# NOT EXECUTED IN THIS SESSION`.
  Run it yourself once you have your GPU/dataset environment back and treat its first run as
  a normal code review, not a verified result.
- Every **existing** result/figure kept from the original notebook is a genuine prior run
  (Kaggle, 2× Tesla T4, `torch==2.10.0+cu128`, real CrisisMMD data) — these are **not**
  regenerated here, only reorganized, re-labeled, and in a few places corrected for a bug
  found during static code review.
- Nothing below has been invented, estimated, or numerically adjusted. Where a claim from the
  task brief (multi-seed runs, retrained ablations) could not be verified in the saved outputs,
  it is reported as a limitation, not asserted.

See the final "Journal Audit Report" cell at the end of the notebook for the itemized list of
bugs found, bugs fixed, and what still needs to be re-executed by you.


## Section Map

This notebook's cells are numbered 1–26 below in **execution order**, which is the order
they must run in for variables to be defined correctly. This differs slightly from a purely
logical journal outline in one place: **Section 14 (Calibration and Test-Time Augmentation)
appears before Section 16 (Main Test Results)** and before Section 21 (Ablation Study),
because the ablation study's post-hoc technique table (Section 21) reuses the calibration
bias tensor computed in Section 14 (`use_calib=True` reads `CALIB_BIAS`, which only exists
after Section 14 runs). Reordering these cells to a strict "results, then ablation, then
calibration" sequence would break that dependency and was not done, per the requirement to
keep the notebook runnable top-to-bottom. All other sections follow the requested outline.

| # | Section |
|---|---|
| 1 | Experimental Configuration and Reproducibility |
| 2 | CrisisMMD Dataset |
| 3 | Data Preprocessing |
| 4 | Model Components: E-REM, UASG, and UG-CMA (overview) |
| 5 | Text Encoder: DeBERTa-v3 |
| 6 | Image Encoder: CLIP ViT-B/32 |
| 7 | Evidence-based Reliability and Uncertainty Module (E-REM) |
| 8 | Uncertainty-Aware Soft Gating (UASG) |
| 9 | Contrastive Alignment with InfoNCE |
| 10 | Uncertainty-Guided Cross-Modal Attention (UG-CMA) |
| 11 | Shared Multimodal Representation and Multi-Task Prediction Heads |
| 12 | Multi-Task Training Objective |
| 13 | Model Training |
| 14 | Calibration and Test-Time Augmentation |
| 15 | Training and Validation Curves |
| 16 | Main Test Results |
| 17 | Confusion Matrices |
| 18 | Uncertainty Analysis |
| 19 | Explainability with Grad-CAM |
| 20 | Representation Visualization (t-SNE) |
| 21 | Ablation Study |
| 22 | Per-Class and Per-Task Performance |
| 23 | Statistical Significance |
| 24 | Single-Sample Inference |
| 25 | Final Experimental Summary |
| 26 | Experimental Reproduction Status |


## 1. Experimental Configuration and Reproducibility

Installs `transformers`, `open_clip_torch`, and `grad-cam` from PyPI. **Prerequisite:** enable Internet access in the Kaggle sidebar before running.

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

pip_install('sentencepiece')
pip_install('protobuf')
pip_install('transformers==4.40.1')
pip_install('open_clip_torch')
pip_install('grad-cam')

print('✅ All packages installed successfully.')

import gc
gc.collect()

### 1.1 Library Imports

Organizes all imports into logical blocks: Core/OS, Data Science, Visualization, Deep Learning, Pre-trained Models (DeBERTa-v3 + CLIP), Image Processing, Evaluation, and Explainability (Grad-CAM).

In [ ]:
import os, sys, warnings, logging, random, json, time, copy, math
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.ERROR)

import numpy  as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy

import matplotlib
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

import torch
import torch.nn            as nn
import torch.nn.functional as F
import torch.optim         as optim
from torch.utils.data      import Dataset, DataLoader
from torch.cuda.amp        import GradScaler, autocast

from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
import open_clip

from PIL                import Image
import cv2
from torchvision        import transforms

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_recall_fscore_support,
)
from sklearn.preprocessing import LabelEncoder

def free_memory():
    """Release Python + CUDA memory. Call after large intermediate objects
    (dataframes, batches, hooks) are no longer needed."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print('✅ All libraries imported | Seed fixed to 42')

import gc
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass

### 1.2 Device Configuration (CPU/GPU-safe)

Detects the available accelerator (CUDA / MPS / CPU) and fixes the global random seed for reproducibility.

In [ ]:
if torch.cuda.is_available():
    DEVICE     = torch.device('cuda')
    N_GPUS     = torch.cuda.device_count()
    GPU_NAMES  = [torch.cuda.get_device_name(i) for i in range(N_GPUS)]
    VRAM_GB    = [torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(N_GPUS)]
else:
    DEVICE    = torch.device('cpu')
    N_GPUS    = 0
    GPU_NAMES = []
    VRAM_GB   = []

USE_AMP = N_GPUS > 0

print('=' * 60)
print('  🖥️  Hardware Configuration')
print('=' * 60)
print(f'  PyTorch Version  : {torch.__version__}')
print(f'  CUDA Available   : {torch.cuda.is_available()}')
print(f'  Device           : {DEVICE}')
print(f'  Number of GPUs   : {N_GPUS}')
for i, (name, vram) in enumerate(zip(GPU_NAMES, VRAM_GB)):
    print(f'    GPU {i}: {name}  ({vram:.1f} GB VRAM)')
print(f'  Mixed Precision  : {USE_AMP}')
print('=' * 60)
if N_GPUS == 0:
    print('GPU not available. Full model training may be computationally expensive on CPU.')
    print('The notebook remains functionally CPU-safe; no CUDA-only code path is required to run it.')

def gpu_memory_report():
    """Per-GPU allocated / reserved / total memory (GB). No-op message on CPU-only runs.
    Call between large experiments (Grad-CAM, t-SNE, ablation, checkpoint reloads) to check
    headroom on each T4 individually -- with 2xT4, each GPU has its OWN ~15 GB limit; the pair
    is NOT a pooled 32 GB device, and nn.DataParallel still replicates activations per-GPU."""
    if not torch.cuda.is_available():
        print('  [gpu_memory_report] No CUDA device available.')
        return
    print('-' * 60)
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        resv  = torch.cuda.memory_reserved(i)  / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i} ({torch.cuda.get_device_name(i)}): '
              f'allocated={alloc:5.2f} GB | reserved={resv:5.2f} GB | total={total:5.2f} GB')
    print('-' * 60)

gpu_memory_report()

free_memory()

### 1.3 Configuration Parameters

Defines `CFG`, the single source of truth for dataset paths, pre-trained checkpoint names (`microsoft/deberta-v3-base`, CLIP **ViT-B/32**), dimensions, training hyperparameters, loss weights, and task label maps.

In [ ]:
import os
import gc
from pathlib import Path

def _find_dataset_root():
    env = os.environ.get('CRISISMD_ROOT')
    if env and Path(env).exists():
        return Path(env)

    kaggle_slugs = [
        '/kaggle/input/datasets/seaninggg/crisismmd-multimodal-crisis-dataset',
        '/kaggle/input/datasets/seaninggg/crisismmd-multimodal-crisis-dataset/CrisisMMD_v2.0',
        '/kaggle/input/crisismmd-multimodal-crisis-dataset',
        '/kaggle/input/crisismmdv2',
        '/kaggle/input/crisis-mmd',
    ]
    for s in kaggle_slugs:
        if Path(s).exists():
            return Path(s)

    local_tries = [
        Path('/content/CrisisMMD_v2.0'),
        Path('/content/CrisisMMD'),
        Path('/workspace/CrisisMMD'),
        Path('/data/CrisisMMD'),
        Path.home() / 'datasets' / 'CrisisMMD',
        Path.home() / 'data' / 'CrisisMMD',
        Path('./CrisisMMD_v2.0'),
        Path('./CrisisMMD'),
    ]
    for p in local_tries:
        if p.exists():
            return p

    raise FileNotFoundError(
        '\n' + '=' * 65 + '\n'
        'CrisisMMD dataset NOT FOUND.\n\n'
        'Check the right-hand sidebar under "Data" and ensure the dataset is attached.\n'
        + '=' * 65
    )

def _find_data_dir(root):
    sub = root / 'CrisisMMD_v2.0'
    return sub if sub.exists() else root

def _find_output_dir():
    for candidate in [Path('/kaggle/working'), Path('/content'), Path('./outputs')]:
        if candidate.parent.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
    return Path('.')

try:
    _ROOT = _find_dataset_root()
except FileNotFoundError as e:
    print(e)
    _ROOT = Path('.')

_DATA    = _find_data_dir(_ROOT)
_ANN_DIR = _DATA / 'annotations'
_IMG_DIR = _DATA / 'data_image'
_OUT_DIR = _find_output_dir()

class CFG:

    ROOT      = _ROOT
    DATA_DIR  = _DATA
    ANN_DIR   = _ANN_DIR
    IMG_DIR   = _IMG_DIR
    OUT_DIR   = _OUT_DIR

    ALL_TSVS  = [
        f for f in _ANN_DIR.iterdir()
        if f.suffix == '.tsv' and not f.name.startswith('._')
    ] if _ANN_DIR.exists() else []

    ROBERTA_CKPT  = 'microsoft/deberta-v3-base'
    CLIP_MODEL    = 'ViT-B-32'
    CLIP_PRETRAIN = 'openai'

    TEXT_DIM      = 768
    VIS_DIM       = 768
    FUSE_DIM      = 768
    GRAPH_HIDDEN  = 512
    GRAPH_OUT     = 384
    HEAD_DIM      = 256

    BATCH_SIZE    = 32
    EPOCHS        = 45
    LR_ROBERTA    = 1e-5
    LR_CLIP       = 1e-5
    LR_HEAD       = 3e-4
    WEIGHT_DECAY  = 1e-2
    WARMUP_RATIO  = 0.1
    MAX_GRAD_NORM = 1.0
    MAX_LEN       = 128
    IMG_SIZE      = 224
    PATIENCE      = 10

    CKPT_TASKS       = ['human', 'damage']
    FOCAL_GAMMA_HUD    = 1.5

    FOCAL_GAMMA_HUMAN  = 1.5
    FOCAL_GAMMA_DAMAGE = 1.2
    USE_SAMPLER      = True

    SOFTEN_CB_WITH_SAMPLER = True

    USE_CLASS_BALANCED_LOSS = True

    CB_BETA                 = 0.999 if SOFTEN_CB_WITH_SAMPLER and USE_SAMPLER else 0.9999
    USE_MANIFOLD_MIXUP      = True
    MIXUP_ALPHA             = 0.2
    MIXUP_PROB              = 0.5
    USE_LOGIT_CALIBRATION   = True
    CALIB_SEARCH_ITERS      = 300
    USE_TTA                 = True
    USE_TEXT_CLEANING       = True
    USE_HIERARCHICAL_GATING = True
    RUN_GENUINE_ABLATION    = True
    ABLATION_RETRAIN_EPOCHS = 15

    ABLATION_SEEDS          = [42, 43, 44]

    FREEZE_ENCODER_EPOCHS = 2
    HUMAN_DAMAGE_LABEL_SMOOTHING = 0.08
    USE_NO_DECAY_GROUPS   = True
    LOG_VAR_CLAMP         = (-3.0, 3.0)
    MIXUP_LOSS_BLEND      = 0.3
    GRAD_ACCUM_STEPS      = 2

    EFFECTIVE_SCHEDULE_EPOCHS = 22
    SWA_PER_TASK_TOLERANCE    = 0.01

    LAMBDA_EVENT   = 1.0
    LAMBDA_INFO    = 1.2
    LAMBDA_HUMAN   = 2.0
    LAMBDA_DAMAGE  = 2.0
    LAMBDA_VERIF   = 1.2

    LAMBDA_UNCERT  = 0.3

    FOCAL_GAMMA_VERIF = 1.5
    CB_BETA_VERIF     = 0.99
    VERIF_LABEL_SMOOTHING = 0.05

    EREM_N_PROTOTYPES = 8

    USE_FILM_SE_GATING = True

    USE_SUPCON      = True
    SUPCON_TEMP     = 0.1
    SUPCON_TASK     = 'event_type'
    SUPCON_WEIGHT_INIT = 0.15

    GRADCAM_N_EVAL_SAMPLES = 100
    GRADCAM_METHODS        = ['gradcam', 'gradcam++', 'eigencam']
    CALIBRATION_METHOD     = 'temperature_and_bias'

    EVENT_LABELS  = ['earthquake', 'fire', 'flood', 'hurricane']
    INFO_LABELS   = ['informative', 'not_informative']
    HUMAN_LABELS  = [
        'affected_individuals', 'infrastructure_and_utility_damage',
        'not_humanitarian', 'other_relevant_information',
        'rescue_volunteering_or_donation_effort',
        'vehicle_damage', 'missing_or_found_people'
    ]
    DAMAGE_LABELS = ['little_or_no_damage', 'mild_damage', 'severe_damage']
    VERIF_LABELS  = ['authentic', 'fabricated']

    N_EVENT   = len(EVENT_LABELS)
    N_INFO    = len(INFO_LABELS)
    N_HUMAN   = len(HUMAN_LABELS)
    N_DAMAGE  = len(DAMAGE_LABELS)
    N_VERIF   = len(VERIF_LABELS)

    CKPT_DIR  = _OUT_DIR / 'checkpoints'
    PLOTS_DIR = _OUT_DIR / 'plots'
    METRICS_DIR = _OUT_DIR / 'metrics'
    PRED_DIR  = _OUT_DIR / 'predictions'
    LOG_DIR   = _OUT_DIR / 'logs'
    for _d in (CKPT_DIR, PLOTS_DIR, METRICS_DIR, PRED_DIR, LOG_DIR):
        _d.mkdir(parents=True, exist_ok=True)

    CKPT_PATH = str(CKPT_DIR / 'best_model.pt')

cfg = CFG()

print('📋 Configuration Summary')
print('─' * 70)
print(f'  Root Base   : {cfg.ROOT}')
print(f'  Annotations : {cfg.ANN_DIR}')
print(f'  Images      : {cfg.IMG_DIR}')
print(f'  TSVs Found  : {len(cfg.ALL_TSVS)}')
print(f'  Checkpoints : {cfg.CKPT_DIR}')
print(f'  Plots       : {cfg.PLOTS_DIR}')
print(f'  Metrics     : {cfg.METRICS_DIR}')
print(f'  Predictions : {cfg.PRED_DIR}')
print('─' * 70)

_n_gpus_eff = max(N_GPUS, 1)
_effective_batch = cfg.BATCH_SIZE * cfg.GRAD_ACCUM_STEPS * _n_gpus_eff
print('\n📦 Batch / Multi-GPU Configuration')
print('─' * 70)
print(f'  Per-device batch size     : {cfg.BATCH_SIZE}')
print(f'  Gradient accumulation     : {cfg.GRAD_ACCUM_STEPS} steps')
print(f'  GPUs used (DataParallel)  : {N_GPUS if N_GPUS > 0 else "0 (CPU)"}')
print(f'  Multi-GPU active          : {N_GPUS > 1}')
print(f'  Effective global batch    : {cfg.BATCH_SIZE} x {cfg.GRAD_ACCUM_STEPS} x {_n_gpus_eff} = {_effective_batch}')
print('  Note: with 2xT4, DataParallel splits each batch across GPUs; each GPU still holds')
print('        its own activations/gradients within its own ~15 GB, memory does NOT pool.')
print('─' * 70)

free_memory()

## 2. CrisisMMD Dataset

Loads all CrisisMMD `.tsv` annotation files, merges tweet text with image paths, and builds the `train / dev / test` splits.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print('Loading and combining all event TSVs...')
dfs = []
for tsv_path in cfg.ALL_TSVS:
    try:
        df = pd.read_csv(tsv_path, sep='\t', on_bad_lines='skip')
        df.columns = [c.strip().lower() for c in df.columns]

        if 'event_name' not in df.columns and 'event_type' not in df.columns:
            name = tsv_path.stem.lower()
            if 'earthquake' in name:
                df['event_type'] = 'earthquake'
            elif 'fire' in name:
                df['event_type'] = 'fire'
            elif 'flood' in name:
                df['event_type'] = 'flood'
            else:
                df['event_type'] = 'hurricane'

        dfs.append(df)
    except Exception as e:
        pass

full_df = pd.concat(dfs, ignore_index=True)

df_train, temp_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df['event_type'])
df_dev, df_test   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['event_type'])

print(f'  [TRAIN] shape : {df_train.shape}')
print(f'  [DEV]   shape : {df_dev.shape}')
print(f'  [TEST]  shape : {df_test.shape}')

RENAME_MAP = {
    'tweet_text'         : 'text',
    'image'              : 'image_path',
    'label_image'        : 'label_image',
    'label_text'         : 'label_text',
    'label'              : 'label_text',
    'event_type'         : 'event_type',
    'event_name'         : 'event_type',
    'humanitarian_class' : 'humanitarian',
    'text_human'         : 'humanitarian',
    'damage_type'        : 'damage',
    'image_damage'       : 'damage',
    'text_info'          : 'informative',
    'informativeness'    : 'informative',
}

def normalize_cols(df):
    df = df.copy()
    df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})

    if 'informative' not in df.columns and 'label_text' in df.columns:
        df['informative'] = df['label_text']

    if 'image_path' in df.columns:
        def resolve_path(p):
            p_str = str(p).strip()
            if p_str.startswith('/'):
                return p_str
            if p_str.startswith('data_image/'):

                return str(cfg.DATA_DIR / p_str)
            return str(cfg.IMG_DIR / p_str)

        df['image_path'] = df['image_path'].apply(resolve_path)

    if 'humanitarian' in df.columns:
        df['humanitarian'] = df['humanitarian'].replace({
            'missing_or_found_people': 'other_relevant_information',
            'vehicle_damage': 'other_relevant_information'
        })

    return df

df_train = normalize_cols(df_train)
df_dev   = normalize_cols(df_dev)
df_test  = normalize_cols(df_test)

import re

_DIR_TAG_RE = re.compile(r'(?:[\w\-]+/)+[\w\-.]+')
_SOURCE_TAG_RE = re.compile(
    r'\b(?:' + '|'.join([
        'california_wildfires', 'hurricane_(?:harvey|irma|maria|florence|michael)',
        'mexico_earthquake', 'iraq_iran_earthquake', 'sri_lanka_floods',
        'srilanka_floods', 'puebla_mexico_earthquake', 'ecuador_earthquake',
        'data_image', 'crisismmd', 'annotations_final', 'consolidated'
    ]) + r')\b', flags=re.IGNORECASE
)

def clean_text(text):
    """Strip dataset-artifact directory/source tags out of raw tweet text
    so the text encoder can't shortcut on dataset provenance."""
    if not isinstance(text, str):
        return text
    t = text
    t = _DIR_TAG_RE.sub(' ', t)
    t = _SOURCE_TAG_RE.sub(' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

for _df in (df_train, df_dev, df_test):
    if 'text' in _df.columns:
        _df['text_raw'] = _df['text']
        if getattr(cfg, 'USE_TEXT_CLEANING', True):
            _df['text'] = _df['text'].apply(clean_text)

print('✅ Stripped directory names / source tags from text column (train/dev/test).')

def encode_label_col(df_t, df_d, df_te, col):
    le = LabelEncoder()
    all_vals = pd.concat([df_t[col], df_d[col], df_te[col]]).dropna().unique()
    le.fit(all_vals)
    for df in [df_t, df_d, df_te]:
        df[f'{col}_idx'] = le.transform(df[col].fillna(le.classes_[0]))
    return le

LABEL_ENCODERS = {}
for col in ['event_type', 'informative', 'humanitarian', 'damage']:
    if col in df_train.columns:
        LABEL_ENCODERS[col] = encode_label_col(df_train, df_dev, df_test, col)
        print(f'  {col:15s} → classes: {list(LABEL_ENCODERS[col].classes_)}')

_IMAGE_INFO_CANDIDATES = ['image_info', 'img_info', 'image_informative', 'label_image_info']

def _find_image_info_col(df):
    for c in _IMAGE_INFO_CANDIDATES:
        if c in df.columns:
            return c
    return None

_img_info_col = _find_image_info_col(df_train)

if _img_info_col is not None:
    print(f"\n✅ Found independent image-side informativeness column: '{_img_info_col}' — "
          "building REAL cross-modal Verification label from text/image agreement.")

    def _build_verif_label(df):
        df = df.copy()
        txt = df['informative'].astype(str).str.strip().str.lower()
        img = df[_img_info_col].astype(str).str.strip().str.lower()
        both_present = txt.notna() & img.notna() & (txt != 'nan') & (img != 'nan')
        agree = (txt == img)

        df['verif_idx'] = np.where(both_present, agree.astype(int), np.nan)
        return df

    import numpy as np
    df_train = _build_verif_label(df_train)
    df_dev   = _build_verif_label(df_dev)
    df_test  = _build_verif_label(df_test)

    _n_missing = int(df_train['verif_idx'].isna().sum())
    _n_total   = len(df_train)
    print(f"   Cross-modal agreement available for {_n_total - _n_missing}/{_n_total} train rows "
          f"({_n_missing} rows missing one side; verif label left NaN for those, handled at batch time).")

    _n_verified = int((df_train['verif_idx'] == 1).sum())
    _n_unverified = int((df_train['verif_idx'] == 0).sum())
    print(f"   Class balance (train): verified(agree)={_n_verified}  unverified(disagree)={_n_unverified}")
    VERIF_IS_REAL = True
else:
    print(f"\n⚠️  No independent image-side informativeness column found among {_IMAGE_INFO_CANDIDATES} — "
          "cannot build a genuine cross-modal Verification label from this dataset build. "
          "Falling back to the OLD proxy (verif = 1 when informative == 'not_informative'), "
          "clearly flagged as a proxy everywhere it's reported.")
    for _df in (df_train, df_dev, df_test):
        _df['verif_idx'] = (_df['informative'].astype(str).str.strip().str.lower() == 'not_informative').astype(int)
    VERIF_IS_REAL = False

if 'event_type' in LABEL_ENCODERS: cfg.N_EVENT = len(LABEL_ENCODERS['event_type'].classes_)
if 'informative' in LABEL_ENCODERS: cfg.N_INFO = len(LABEL_ENCODERS['informative'].classes_)
if 'humanitarian' in LABEL_ENCODERS: cfg.N_HUMAN = len(LABEL_ENCODERS['humanitarian'].classes_)
if 'damage' in LABEL_ENCODERS: cfg.N_DAMAGE = len(LABEL_ENCODERS['damage'].classes_)

def check_images(df, split_name):
    missing = df['image_path'].apply(lambda p: not Path(p).exists()).sum()
    total   = len(df)
    print(f'  [{split_name}] {total - missing}/{total} images found  |  {missing} missing')

print('\nChecking image paths...')
if 'image_path' in df_train.columns:
    check_images(df_train, 'TRAIN')
    check_images(df_dev,   'DEV  ')
    check_images(df_test,  'TEST ')

print('✅ Annotations combined, split, validated, and images mapped correctly.')
df_train.head(3)

free_memory()

### 2.1 Dataset Statistics and Class Distribution

Visualizes label balance across the four supervised tasks: Event Type, Informativeness, Humanitarian Need, and Damage Severity.

In [ ]:
TASK_COLS = [c for c in ['event_type','informative','humanitarian','damage'] if c in df_train.columns]

fig, axes = plt.subplots(1, len(TASK_COLS), figsize=(6 * len(TASK_COLS), 5))
if len(TASK_COLS) == 1:
    axes = [axes]

PALETTES = ['Blues_d','Greens_d','Oranges_d','Purples_d']

for ax, col, pal in zip(axes, TASK_COLS, PALETTES):
    counts  = df_train[col].value_counts()
    colors  = sns.color_palette(pal, len(counts))
    bars    = ax.barh(counts.index, counts.values, color=colors, edgecolor='white')
    ax.set_title(f'{col.replace("_"," ").title()}(Train Split)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Count', fontsize=11)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + max(counts.values)*0.01, bar.get_y() + bar.get_height()/2,
                f'{val}', va='center', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('📊 Multi-Task Class Distribution — CrisisMMD v2.0 Train Split',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'eda_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class distribution chart saved.')

free_memory()

### 2.2 Split Integrity and Leakage Checks

Static checks that the train/dev/test split created in Section 2 is leak-free. These do
**not** alter the split — they only assert properties of the split already made with
`random_state=42` in the original notebook. Marked `NOT EXECUTED IN THIS SESSION` because
this environment has no copy of `df_train`/`df_dev`/`df_test` (no dataset access). Run as-is
in the original training environment.


In [ ]:
# NOT EXECUTED IN THIS SESSION -- no dataset access in the audit environment.
# Run this cell in the original (Kaggle/GPU) environment right after Section 2's split.

def leakage_report(df_train, df_dev, df_test):
    report = {}

    # 1. Duplicate tweet IDs within each split
    id_col = 'tweet_id' if 'tweet_id' in df_train.columns else None
    if id_col:
        for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
            report[f'{name}_dup_tweet_id'] = int(df[id_col].duplicated().sum())
        train_ids, dev_ids, test_ids = set(df_train[id_col]), set(df_dev[id_col]), set(df_test[id_col])
        report['train_dev_id_overlap']  = len(train_ids & dev_ids)
        report['train_test_id_overlap'] = len(train_ids & test_ids)
        report['dev_test_id_overlap']   = len(dev_ids & test_ids)
    else:
        report['tweet_id_column_found'] = False

    # 2. Duplicate image paths within/across splits
    if 'image_path' in df_train.columns:
        for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
            report[f'{name}_dup_image_path'] = int(df['image_path'].duplicated().sum())
        tr_img, dv_img, te_img = set(df_train['image_path']), set(df_dev['image_path']), set(df_test['image_path'])
        report['train_dev_image_overlap']  = len(tr_img & dv_img)
        report['train_test_image_overlap'] = len(tr_img & te_img)
        report['dev_test_image_overlap']   = len(dv_img & te_img)

    # 3. Duplicate raw text within/across splits (exact match; near-dup would need embeddings)
    if 'text' in df_train.columns:
        for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
            report[f'{name}_dup_text'] = int(df['text'].duplicated().sum())
        tr_txt, dv_txt, te_txt = set(df_train['text']), set(df_dev['text']), set(df_test['text'])
        report['train_dev_text_overlap']  = len(tr_txt & dv_txt)
        report['train_test_text_overlap'] = len(tr_txt & te_txt)
        report['dev_test_text_overlap']   = len(dv_txt & te_txt)

    return report

_leak_report = leakage_report(df_train, df_dev, df_test)
import pandas as pd
print('=== Split integrity / leakage report ===')
for k, v in _leak_report.items():
    flag = '  <-- CHECK' if isinstance(v, int) and v > 0 and 'overlap' in k else ''
    print(f'  {k:30s}: {v}{flag}')

# Any nonzero *_overlap value means the same tweet/image/text appears in more than one split,
# which would leak test information into training. This does NOT modify the split; it only
# flags a problem if one exists so it can be reported honestly.


## 3. Data Preprocessing

Renders a random grid of tweet text + image pairs so the multimodal pairing can be sanity-checked visually before training.

In [ ]:
def show_sample_grid(df, n_rows=3, n_cols=4, title='Random Training Samples'):
    sample = df.dropna(subset=['image_path','text']).sample(
        min(n_rows * n_cols, len(df)), random_state=7
    ).reset_index(drop=True)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.8))
    fig.suptitle(f'🖼️  {title}', fontsize=15, fontweight='bold', y=1.01)

    for idx, ax in enumerate(axes.flat):
        if idx >= len(sample):
            ax.axis('off')
            continue
        row = sample.iloc[idx]
        img_path = Path(row['image_path'])
        if img_path.exists():
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
        else:
            ax.set_facecolor('#f0f0f0')
            ax.text(0.5, 0.5, 'ImageMissing', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='gray')

        text_snip = str(row.get('text', ''))[:80] + ('…' if len(str(row.get('text',''))) > 80 else '')
        ev = row.get('event_type', 'N/A')
        dm = row.get('damage',     'N/A')
        inf= row.get('informative','N/A')
        ax.set_title(f'{text_snip}\n▸ {ev} | {dm} | {inf}',
                     fontsize=6.5, color='#1a1a2e', pad=4,
                     wrap=True)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(str(cfg.PLOTS_DIR / 'eda_samples.png'), dpi=130, bbox_inches='tight')
    plt.show()

if 'image_path' in df_train.columns and 'text' in df_train.columns:
    show_sample_grid(df_train)
else:
    print('⚠️  image_path or text column not found; skipping sample grid.')

free_memory()

### 3.1 Raw Data Spot-Check

Prints 50 raw rows straight from the source `.tsv` / images, prior to any preprocessing, to confirm the loader is reading the correct files.

#### 50 Lines of Text from the Training Dataset


In [ ]:
import pprint

def print_texts(df, split_name, n=50):
    text_col = "text" if "text" in df.columns else next((c for c in df.columns if ("text" in c or "tweet" in c) and "id" not in c.lower()), None)
    if text_col:
        print(f"\n{'='*40}\n{split_name} Texts (First {n})\n{'='*40}")
        sample_texts = df[text_col].dropna().head(n).tolist()
        for i, text in enumerate(sample_texts, 1):
            print(f"{i}.")
            pprint.pprint(text)
    else:
        print(f"No text column found in {split_name}!")

print_texts(df_train, "Train")
print_texts(df_dev, "Validation/Dev")
print_texts(df_test, "Test")

free_memory()

#### 50 Original TSV Files (Rows)


In [ ]:
try:
    display(df_train.head(50))
except NameError:
    from IPython.display import display
    display(df_train.head(50))

free_memory()

#### 50 Original Annotations


In [ ]:
import pandas as pd
import gc

samples_per_event = 12
target_cols = ['event_type', 'damage', 'humanitarian', 'informative']
valid_cols = [c for c in target_cols if c in df_train.columns]

_src_df = df_train.dropna(subset=['event_type'])
sampled_df = pd.concat(
    [g.sample(min(len(g), samples_per_event)) for _, g in _src_df.groupby('event_type')]
)

mixed_df = sampled_df.sample(frac=1.0).reset_index(drop=True)

print("=" * 80)
print(f"MIXED SAMPLES ACROSS ALL EVENT TYPES (Total: {len(mixed_df)})")
print("=" * 80)

for j, (_, row) in enumerate(mixed_df[valid_cols].iterrows(), 1):
    print(f"[{j}] {row.to_dict()}")

print()

free_memory()

#### 50 Original Images


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import glob

img_col = next((c for c in df_train.columns if "image" in c or "img" in c), None)

if img_col:

    print("Scanning image folders...")

    image_files = glob.glob(
        os.path.join(cfg.DATA_DIR, "data_image", "**", "*.jpg"),
        recursive=True
    )
    image_files += glob.glob(
        os.path.join(cfg.DATA_DIR, "data_image", "**", "*.png"),
        recursive=True
    )

    image_map = {
        os.path.splitext(os.path.basename(f))[0]: f
        for f in image_files
    }

    print(f"Total images found: {len(image_map):,}")

    sample_images = df_train[img_col].dropna().head(50).tolist()

    fig, axes = plt.subplots(5, 10, figsize=(20, 10))
    axes = axes.flatten()

    for i, img_id in enumerate(sample_images):

        if img_id in image_map:
            img = Image.open(image_map[img_id]).convert("RGB")
            axes[i].imshow(img)
        else:
            axes[i].text(
                0.5,
                0.5,
                "Not Found",
                ha="center",
                va="center",
                fontsize=8,
                color="red"
            )

        axes[i].axis("off")
        axes[i].set_title(f"Img {i+1}", fontsize=8)

    plt.tight_layout()
    plt.show()

else:
    print("No image column found!")
    print(df_train.columns.tolist())

free_memory()

### 3.2 Feature Extraction (Illustrative Demo)
DeBERTa-v3 text embedding $H_t$, CLIP image embedding $H_v$, plus hand-crafted quality vectors $Q_{text}\in\mathbb{R}^3$, $Q_{image}\in\mathbb{R}^4$ for E-REM.

### E-REM feature fusion
Concatenates `h_text`(768) + `h_image`(768) + `q_text`(3) + `q_image`(4) = 1543-dim input to the E-REM network.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy as scipy_entropy

def get_text_metrics(text):
    if not isinstance(text, str) or not text.strip():
        return pd.Series({
            'Char Count': 0.0, 'Word Count': 0.0, 'Entropy': 0.0,
            'Norm Char': 0.0, 'Norm Word': 0.0, 'Norm Entropy': 0.0
        })

    char_count = float(len(text))
    word_count = float(len(text.split()))

    counts = pd.Series(list(text)).value_counts().values
    entropy_val = scipy_entropy(counts)

    norm_char = min(char_count / 500.0, 1.0)
    norm_word = min(word_count / 100.0, 1.0)
    norm_entropy = min(entropy_val / 5.0, 1.0)

    return pd.Series({
        'Char Count': char_count,
        'Word Count': word_count,
        'Entropy': round(entropy_val, 4),
        'Norm Char': round(norm_char, 4),
        'Norm Word': round(norm_word, 4),
        'Norm Entropy': round(norm_entropy, 4)
    })

df_metrics = df_train.dropna(subset=['text']).head(15)[['text']].copy()

df_metrics.rename(columns={'text': 'Sentence'}, inplace=True)

df_metrics['Sentence'] = df_metrics['Sentence'].apply(
    lambda x: x[:46] + "..." if len(x) > 46 else x
)

metrics_columns = df_train.dropna(subset=['text']).head(15)['text'].apply(get_text_metrics)

final_text_features_df = pd.concat([df_metrics, metrics_columns], axis=1)

print(final_text_features_df.to_string())

### Image Quality Metrics Function

Defines `get_image_metrics`, the image counterpart to the text quality function above: computes brightness, contrast, blur, and noise (plus their normalized versions) for a single image path, returning zeroed defaults if the path is invalid or unreadable.

In [ ]:
import pandas as pd
import numpy as np
import cv2
from scipy.stats import entropy as scipy_entropy

def get_image_metrics(img_path):

    default_metrics = {
        'Brightness': 0.0, 'Contrast': 0.0, 'Blur': 0.0, 'Noise': 0.0,
        'Norm Bright': 0.0, 'Norm Contrast': 0.0, 'Norm Blur': 0.0, 'Norm Noise': 0.0
    }

    if not isinstance(img_path, str):
        return pd.Series(default_metrics)

    cv_img = cv2.imread(img_path)
    if cv_img is None:
        return pd.Series(default_metrics)

    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)

    brightness = np.mean(gray)
    contrast = np.std(gray)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    hist, _ = np.histogram(gray, bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    img_entropy = scipy_entropy(hist)

    norm_brightness = min(brightness / 255.0, 1.0)
    norm_contrast = min(contrast / 128.0, 1.0)
    norm_blur = min(laplacian_var / 1000.0, 1.0)
    norm_noise = min(img_entropy / 8.0, 1.0)

    return pd.Series({
        'Brightness': round(brightness, 2),
        'Contrast': round(contrast, 2),
        'Blur': round(laplacian_var, 2),
        'Noise': round(img_entropy, 4),
        'Norm Bright': round(norm_brightness, 4),
        'Norm Contrast': round(norm_contrast, 4),
        'Norm Blur': round(norm_blur, 4),
        'Norm Noise': round(norm_noise, 4)
    })

df_img_metrics = df_train.dropna(subset=['image_path']).head(15)[['image_path']].copy()

df_img_metrics.rename(columns={'image_path': 'Image Link'}, inplace=True)

df_img_metrics['Image Link'] = df_img_metrics['Image Link'].apply(
    lambda x: "..." + x[-43:] if len(x) > 46 else x
)

original_paths = df_train.dropna(subset=['image_path']).head(15)['image_path']
img_metrics_columns = original_paths.apply(get_image_metrics)

final_image_features_df = pd.concat([df_img_metrics, img_metrics_columns], axis=1)

print(final_image_features_df.to_string())

### Live Demo: E-REM Input Construction on a Real Batch

A quick, self-contained demonstration (with its own `RealDisasterDataset`/`UG_CMA` mini-loop below) showing how a real batch of text + image quality features would flow into E-REM before the full production pipeline is built in the following sections.

In [ ]:
import torch
import torch.nn as nn
import sys
import pandas as pd
from PIL import Image
import cv2
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import open_clip
from scipy.stats import entropy as scipy_entropy

def extract_text_quality(text):
    if not isinstance(text, str) or not text.strip():
        return np.zeros(3, dtype=np.float32)

    norm_len = min(len(text) / 500.0, 1.0)

    upper_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)

    counts = pd.Series(list(text)).value_counts().values
    char_entropy = scipy_entropy(counts) / 5.0

    return np.array([norm_len, upper_ratio, char_entropy], dtype=np.float32)

def extract_image_quality(cv_img):
    if cv_img is None:
        return np.zeros(4, dtype=np.float32)

    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)

    brightness = np.mean(gray) / 255.0

    contrast = min(np.std(gray) / 128.0, 1.0)

    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    sharpness = min(laplacian_var / 1000.0, 1.0)

    hist, _ = np.histogram(gray, bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    img_entropy = scipy_entropy(hist) / 8.0

    return np.array([brightness, contrast, sharpness, img_entropy], dtype=np.float32)

torch.set_printoptions(threshold=10_000, sci_mode=False, linewidth=120)
np.set_printoptions(threshold=sys.maxsize, suppress=True, linewidth=120)

print("Loading models for demonstration (this may take a moment)...")
_tokenizer = AutoTokenizer.from_pretrained(cfg.ROBERTA_CKPT, use_fast=False)
_text_model = AutoModel.from_pretrained(cfg.ROBERTA_CKPT)
_text_model.eval()

_clip_model, _, _clip_preprocess = open_clip.create_model_and_transforms(cfg.CLIP_MODEL, pretrained=cfg.CLIP_PRETRAIN)
_clip_model.eval()

sample_row = df_train.dropna(subset=['text', 'image_path']).iloc[0]
sample_text = sample_row['text']
sample_image_path = sample_row['image_path']

print(f"\nSample Text: '{sample_text}'")
print(f"Sample Image Path: '{sample_image_path}'")

with torch.no_grad():
    _inputs = _tokenizer(sample_text, return_tensors='pt', padding='max_length', truncation=True, max_length=cfg.MAX_LEN)
    _text_outputs = _text_model(**_inputs)
    h_text_raw = _text_outputs.last_hidden_state[:, 0, :]

with torch.no_grad():
    _pil_img = Image.open(sample_image_path).convert('RGB')
    _img_tensor = _clip_preprocess(_pil_img).unsqueeze(0)
    h_image_raw = _clip_model.encode_image(_img_tensor)

text_proj = nn.Linear(h_text_raw.shape[1], 768)
image_proj = nn.Linear(h_image_raw.shape[1], 768)

with torch.no_grad():
    h_text = text_proj(h_text_raw)
    h_image = image_proj(h_image_raw)

q_text_np = extract_text_quality(sample_text)
_cv_img = cv2.imread(sample_image_path)
q_image_np = extract_image_quality(_cv_img)

q_text = torch.tensor(q_text_np, dtype=torch.float32).unsqueeze(0)
q_image = torch.tensor(q_image_np, dtype=torch.float32).unsqueeze(0)

q_text_norm = F.normalize(q_text, p=2, dim=-1)
q_image_norm = F.normalize(q_image, p=2, dim=-1)

def display_tensor_info(name, tensor, desc):
    print(f"\n🔹 {name} | Shape: {list(tensor.shape)} | {desc}")

    print(tensor.squeeze().tolist()[:10], "... [truncated for display]")

print('\n' + '═' * 80)
print(' 📊 REAL FEATURES BEFORE CONCATENATION')
print('═' * 80)
display_tensor_info('h_text', h_text, 'Projected Text Embedding (768 dims)')
display_tensor_info('h_image', h_image, 'Projected Image Embedding (768 dims)')
display_tensor_info('q_text_norm', q_text_norm, 'Normalized Text Quality (3 dims)')
display_tensor_info('q_image_norm', q_image_norm, 'Normalized Image Quality (4 dims)')

rem_input = torch.cat(
    [
        h_text,
        h_image,
        q_text_norm,
        q_image_norm,
    ],
    dim=-1,
)

print('\n' + '═' * 80)
print(' 🧬 FINAL CONCATENATED FEATURE VECTOR (AFTER)')
print('═' * 80)
print(f"🔹 rem_input | Shape: {list(rem_input.shape)} | Combined Multimodal REM Input")

text_dim = 768
image_dim = 768
quality_dim = q_text_norm.shape[1] + q_image_norm.shape[1]
expected_total_dim = text_dim + image_dim + quality_dim

extracted_text = rem_input[:, :text_dim]
extracted_image = rem_input[:, text_dim:text_dim+image_dim]

print(f"\n--- Extracted Text Features from REM (first {text_dim} dims) ---")
print(extracted_text.squeeze().tolist()[:10], "...")

print(f"\n--- Extracted Image Features from REM (next {image_dim} dims) ---")
print(extracted_image.squeeze().tolist()[:10], "...")

assert rem_input.shape[1] == 1543, f"Expected 1543 dims, but got {rem_input.shape[1]}"
print(f"\n✓ Successfully created the guaranteed 1543-dimensional REM feature vector ready for fusion.")

free_memory()

### 3.3 Tokenizer and Image Transform Pipelines

Instantiates the DeBERTa-v3 tokenizer, loads the CLIP ViT-B/32 visual tower, and defines the train-time augmentation / eval-time normalization transforms used by the `Dataset` class.

In [ ]:
print('Loading RoBERTa tokenizer …')
TOKENIZER = AutoTokenizer.from_pretrained(cfg.ROBERTA_CKPT, use_fast=False)
print(f'  Vocab size : {TOKENIZER.vocab_size:,}')

print('Loading CLIP model …')
CLIP_MODEL, _, CLIP_PREPROCESS = open_clip.create_model_and_transforms(
    cfg.CLIP_MODEL, pretrained=cfg.CLIP_PRETRAIN
)
CLIP_MODEL = CLIP_MODEL.visual
CLIP_MODEL.eval()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.RandomResizedCrop(cfg.IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.CenterCrop(cfg.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('✅ Text tokenizer and vision transforms ready.')

free_memory()

### 3.4 Dataset and DataLoader Construction

Defines `CrisisMMDDataset`, which returns per-sample token ids, attention mask, pixel values, the $Q_{text}$/$Q_{image}$ reliability feature vectors, and the four task labels.

In [ ]:
import torch
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

class CrisisMMDDataset(Dataset):
    """Multimodal Dataset for CrisisMMD v2.0."""

    LABEL_COLS = {
        'event_type'   : 'label_event',
        'informative'  : 'label_info',
        'humanitarian' : 'label_human',
        'damage'       : 'label_damage',
    }

    def __init__(self, df: pd.DataFrame, tokenizer, img_transform, is_train: bool = True):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = img_transform
        self.is_train  = is_train

        texts = self.df['text'].fillna('').astype(str).tolist()

        max_length = getattr(cfg, 'MAX_LEN', 128)

        enc   = self.tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        self.input_ids      = enc['input_ids']
        self.attention_mask = enc['attention_mask']

    def __len__(self):
        return len(self.df)

    def _load_image(self, path_str: str) -> torch.Tensor:
        path = Path(str(path_str))
        try:
            if path.exists():
                img = Image.open(path).convert('RGB')
            else:
                raise FileNotFoundError
        except Exception:

            img_size = getattr(cfg, 'IMG_SIZE', 224)
            img = Image.fromarray(np.zeros((img_size, img_size, 3), dtype=np.uint8))
        return self.transform(img)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]

        input_ids      = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]

        pixel_values = self._load_image(row.get('image_path', ''))

        q_text = torch.tensor([
            row.get('Norm Char',    0.0),
            row.get('Norm Word',    0.0),
            row.get('Norm Entropy', 0.0),
        ], dtype=torch.float32)

        q_image = torch.tensor([
            row.get('Norm Bright',   0.0),
            row.get('Norm Contrast', 0.0),
            row.get('Norm Blur',     0.0),
            row.get('Norm Noise',    0.0),
        ], dtype=torch.float32)

        def get_label(col):
            idx_col = f'{col}_idx'
            if idx_col in self.df.columns:
                return int(row[idx_col])
            return 0

        verif_val = int(row['verif_idx']) if 'verif_idx' in self.df.columns and not pd.isna(row['verif_idx']) else 0

        return {
            'input_ids'      : input_ids,
            'attention_mask' : attention_mask,
            'pixel_values'   : pixel_values,
            'q_text'         : q_text,
            'q_image'        : q_image,
            'label_event'    : torch.tensor(get_label('event_type'),   dtype=torch.long),
            'label_info'     : torch.tensor(get_label('informative'),  dtype=torch.long),
            'label_human'    : torch.tensor(get_label('humanitarian'), dtype=torch.long),
            'label_damage'   : torch.tensor(get_label('damage'),       dtype=torch.long),
            'label_verif'    : torch.tensor(verif_val,                 dtype=torch.long),
        }

print('CrisisMMDDataset class defined ✅')

print('\n' + '═' * 80)
print(' 🧪 TESTING THE DATASET PIPELINE (Fetching item 0)')
print('═' * 80)

demo_dataset = CrisisMMDDataset(
    df=df_train.head(5),
    tokenizer=_tokenizer,
    img_transform=_clip_preprocess,
    is_train=True
)

sample = demo_dataset[0]

for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            print(f"🔹 {key:<16} | Value: {value.item()} (Scalar Label)")
        elif len(value.shape) == 1 and value.shape[0] < 10:
            print(f"🔹 {key:<16} | Shape: {list(value.shape)} | Values: {value.tolist()}")
        else:
            print(f"🔹 {key:<16} | Shape: {list(value.shape)}")

print('\n✓ Dataset is correctly formatting tokens, images, quality vectors, and labels!')

free_memory()

### Populate `df_train` with Quality Features

Runs `get_text_metrics` / `get_image_metrics` over every row of the training split (with a progress bar) and concatenates the resulting quality-feature columns onto `df_train`.

In [ ]:
from tqdm import tqdm

tqdm.pandas(desc="Processing")

print("Extracting text quality metrics...")

text_metrics_df = df_train['text'].progress_apply(get_text_metrics)

print("Extracting image quality metrics (This will take a few minutes)...")
img_metrics_df = df_train['image_path'].progress_apply(get_image_metrics)

df_train = pd.concat([df_train, text_metrics_df, img_metrics_df], axis=1)
df_train = df_train.loc[:, ~df_train.columns.duplicated()]

print("\n✓ Quality columns successfully added to df_train!")

print('\n' + '═' * 80)
print(' 🧪 RE-TESTING DATASET (Now with populated quality features)')
print('═' * 80)

demo_dataset = CrisisMMDDataset(
    df=df_train.head(5),
    tokenizer=_tokenizer,
    img_transform=_clip_preprocess,
    is_train=True
)

sample = demo_dataset[0]

for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            print(f"🔹 {key:<16} | Value: {value.item()} (Scalar Label)")
        elif len(value.shape) == 1 and value.shape[0] < 10:
            rounded_vals = [round(v, 4) for v in value.tolist()]
            print(f"🔹 {key:<16} | Shape: {list(value.shape)} | Values: {rounded_vals}")
        else:
            print(f"🔹 {key:<16} | Shape: {list(value.shape)}")

print('\n✓ Success! The q_text and q_image vectors are now correctly populated.')

### Populate `df_dev` and `df_test` with Quality Features

Repeats the same text/image quality-metric extraction for the validation and test splits, so all three splits carry the same `Norm Char / Norm Word / Norm Entropy / Norm Bright / Norm Contrast / Norm Blur / Norm Noise` feature columns.

In [ ]:
from tqdm import tqdm
tqdm.pandas(desc="Processing")

print("Extracting metrics for Validation set (df_dev)...")
dev_text_metrics = df_dev['text'].progress_apply(get_text_metrics)
dev_img_metrics = df_dev['image_path'].progress_apply(get_image_metrics)

df_dev = pd.concat([df_dev, dev_text_metrics, dev_img_metrics], axis=1)
df_dev = df_dev.loc[:, ~df_dev.columns.duplicated()]

print("\nExtracting metrics for Test set (df_test)...")
test_text_metrics = df_test['text'].progress_apply(get_text_metrics)
test_img_metrics = df_test['image_path'].progress_apply(get_image_metrics)

df_test = pd.concat([df_test, test_text_metrics, test_img_metrics], axis=1)
df_test = df_test.loc[:, ~df_test.columns.duplicated()]

print("\n✓ Quality columns successfully added to df_dev and df_test!")

### 3.4.1 DataLoaders and Sanity Check

Wraps the `train / dev / test` splits in PyTorch `DataLoader`s and runs a one-batch shape/dtype sanity check.

In [ ]:
ds_train = CrisisMMDDataset(df_train, TOKENIZER, TRAIN_TRANSFORMS, is_train=True)
ds_dev   = CrisisMMDDataset(df_dev,   TOKENIZER, VAL_TRANSFORMS,   is_train=False)
ds_test  = CrisisMMDDataset(df_test,  TOKENIZER, VAL_TRANSFORMS,   is_train=False)

N_WORKERS = min(4, os.cpu_count() or 1)

from torch.utils.data import WeightedRandomSampler
import numpy as _np

sampler = None
if getattr(cfg, 'USE_SAMPLER', True) and {'humanitarian_idx','damage_idx'}.issubset(df_train.columns):
    h_idx = df_train['humanitarian_idx'].values
    d_idx = df_train['damage_idx'].values
    h_counts = _np.bincount(h_idx, minlength=cfg.N_HUMAN).clip(min=1)
    d_counts = _np.bincount(d_idx, minlength=cfg.N_DAMAGE).clip(min=1)

    w_h = 1.0 / h_counts[h_idx]
    w_d = 1.0 / d_counts[d_idx]
    sample_weights = (w_h * w_d)
    sample_weights = sample_weights / sample_weights.mean()
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    print(f'✅ WeightedRandomSampler built over (humanitarian × damage) joint rarity.')
else:
    print('⚠️  Sampler disabled or label columns missing -> falling back to shuffle=True.')

dl_train = DataLoader(
    ds_train,
    batch_size=32,
    shuffle=(sampler is None),
    sampler=sampler,
    num_workers=N_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(N_WORKERS > 0)
)

dl_dev = DataLoader(
    ds_dev,
    batch_size=32,
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=True,
    persistent_workers=(N_WORKERS > 0)
)

dl_test = DataLoader(
    ds_test,
    batch_size=32,
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=True,
    persistent_workers=(N_WORKERS > 0)
)

print(f'Train batches : {len(dl_train):,}  ({len(ds_train):,} samples)')
print(f'Dev   batches : {len(dl_dev):,}  ({len(ds_dev):,} samples)')
print(f'Test  batches : {len(dl_test):,}  ({len(ds_test):,} samples)')

print('\n── Sanity Check (one batch) ──────────────────────────────────────────')
batch = next(iter(dl_train))
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f'  {key:20s}  shape={tuple(val.shape)}  dtype={val.dtype}')
print('✅ DataLoaders ready.')

free_memory()

### 3.5 Preprocessed Data Spot-Check

Confirms nothing was corrupted while building `CrisisMMDDataset`: decodes tokenized text, prints the four task labels, and renders a thumbnail grid of images, for the same `N_SPOT_CHECK_SAMPLES` rows of every split.

*(Previously this was 24 separate cells — three near-identical copies of the same three checks, one hand-duplicated per split. Consolidated here into one set of reusable functions called once per split, with identical output.)*

In [ ]:
N_SPOT_CHECK_SAMPLES = 50

def print_sample_texts(dataset, split_name, n=N_SPOT_CHECK_SAMPLES):
    """Print the first n decoded tweet texts for a preprocessed split."""
    df = dataset.df
    text_col = 'text' if 'text' in df.columns else next(
        (c for c in df.columns if ('tweet' in c or 'text' in c) and 'label' not in c), None
    )
    if not text_col:
        print(f'No text column found in {split_name}!')
        return
    print(f"\n{'='*50}\nSample Texts from {split_name} (First {n})\n{'='*50}")
    for i in range(min(n, len(dataset))):
        print(f'[{i+1}] {df.iloc[i][text_col]}')

def print_sample_annotations(dataset, split_name, n=N_SPOT_CHECK_SAMPLES,
                              cols=('event_type', 'damage', 'humanitarian', 'informative')):
    """Print the raw task-label columns for the first n rows of a preprocessed split."""
    df = dataset.df
    valid_cols = [c for c in cols if c in df.columns]
    print(f"\n{'='*50}\nSample Annotations from {split_name} (First {n})\n{'='*50}")
    for i in range(min(n, len(dataset))):
        row = df.iloc[i]
        print(f'[{i+1}] ' + str({c: row[c] for c in valid_cols}))

def show_sample_images(dataset, split_name, n=N_SPOT_CHECK_SAMPLES, n_cols=10):
    """Render a thumbnail grid of the first n images of a preprocessed split."""
    df = dataset.df
    if 'image_path' not in df.columns:
        print('image_path column not found!')
        return
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 2 * n_rows))
    axes = axes.flatten()
    fig.suptitle(f'{split_name} — Preprocessed Image Sample', y=1.02)
    for i in range(len(axes)):
        if i >= min(n, len(dataset)):
            axes[i].axis('off')
            continue
        img_path = df.iloc[i]['image_path']
        if os.path.exists(img_path):
            axes[i].imshow(Image.open(img_path).convert('RGB'))
        else:
            axes[i].text(0.5, 0.5, 'Not Found', ha='center', va='center', color='red', fontsize=8)
        axes[i].axis('off')
        axes[i].set_title(f'Img {i+1}', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
for _dataset, _name in [(ds_train, 'Train'), (ds_dev, 'Dev'), (ds_test, 'Test')]:
    print_sample_texts(_dataset, _name)
    print_sample_annotations(_dataset, _name)
    show_sample_images(_dataset, _name)
    free_memory()

## 4. Model Components: E-REM, UASG, and UG-CMA

This section gives a brief overview of the three uncertainty-aware fusion components; each is implemented and detailed in its own section below (Sections 7, 8, and 10).

- **E-REM** (Evidence-based Reliability and Uncertainty Module): concat $[H_t,H_v,Q_t,Q_v]$ → Dirichlet evidence → uncertainty $u=K/S$
- **UASG** (Uncertainty-Aware Soft Gating): trust-gated features $H'_t=(e_t+(1-e_t)(1-u_t))\odot H_t$
- **UG-CMA** (Uncertainty-Guided Cross-Modal Attention): uncertainty-scaled cross-attention, fused via Linear→LayerNorm→GELU


## 5. Text Encoder: DeBERTa-v3


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

print("Fetching a real batch from dl_train...")
real_batch = next(iter(dl_train))
input_ids = real_batch['input_ids']
attention_mask = real_batch['attention_mask']
pixel_values = real_batch['pixel_values']
q_text = real_batch['q_text']
q_image = real_batch['q_image']

print(f"Batch Size: {input_ids.shape[0]}")

class DeBERTaEncoder(nn.Module):
    """Wraps microsoft/deberta-v3-base and returns the [CLS] pooled representation (H_t in R^768)."""
    def __init__(self, model_name: str = 'microsoft/deberta-v3-base', dropout: float = 0.1):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = self.roberta.config.hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.dropout(cls)

print("Loading Text Encoder (may take a moment to download/initialize)...")
text_encoder = DeBERTaEncoder()
text_encoder.eval()

with torch.no_grad():
    h_t = text_encoder(input_ids, attention_mask)

print("\n--- 🧪 TEXT ENCODER OUTPUT ---")
print(f"🔹 Input Tokens Shape : {list(input_ids.shape)}")
print(f"🔹 H_t Output Shape   : {list(h_t.shape)} -> (Batch, 768 text features)")
print(f"🔸 Sample Values (1st item, first 100 dims): {h_t[0, :100].tolist()}")

## 6. Image Encoder: CLIP ViT-B/32


In [ ]:
class CLIPVisionEncoder(nn.Module):
    """CLIP ViT-B/32 visual encoder, projects 512 → 768 to match text dim."""
    def __init__(self, clip_visual_model, text_dim: int = 768):
        super().__init__()
        self.encoder    = clip_visual_model
        self.projection = nn.Sequential(
            nn.Linear(512, text_dim),
            nn.LayerNorm(text_dim),
            nn.GELU(),
        )
        self.dropout = nn.Dropout(0.1)

    def forward(self, pixel_values):
        feats = self.encoder(pixel_values)
        if feats.dim() > 2:
            feats = feats.mean(dim=1)
        return self.dropout(self.projection(feats))

print("Initializing Vision Encoder...")
vision_encoder = CLIPVisionEncoder(clip_visual_model=_clip_model.visual)
vision_encoder.eval()

with torch.no_grad():
    h_v = vision_encoder(pixel_values)

print("\n--- 🧪 VISION ENCODER OUTPUT ---")
print(f"🔹 Input Images Shape : {list(pixel_values.shape)}")
print(f"🔹 H_v Output Shape   : {list(h_v.shape)} -> (Batch, 768 visual features)")
print(f"🔸 Sample Values (1st item, first 100 dims): {h_v[0, :100].tolist()}")

## 7. Evidence-based Reliability and Uncertainty Module (E-REM)


In [ ]:
class _EvidenceResBlock(nn.Module):
    """Pre-norm residual MLP block (GELU + LayerNorm)."""
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1  = nn.Linear(dim, dim * 2)
        self.fc2  = nn.Linear(dim * 2, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm(x)
        h = F.gelu(self.fc1(h))
        h = self.drop(h)
        h = self.fc2(h)
        return x + self.drop(h)

class _PrototypeAttention(nn.Module):
    """Shared memory of learned semantic prototypes. Both modalities attend to
    the SAME bank, which is what makes the resulting entropy/confidence scores
    comparable across text and image and lets them serve as a genuine, richer
    reliability signal (rather than a modality-private heuristic)."""
    def __init__(self, dim, n_prototypes=8):
        super().__init__()
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, dim) * 0.02)
        self.scale = dim ** -0.5

    def forward(self, x):
        logits = (x @ self.prototypes.t()) * self.scale
        attn = F.softmax(logits, dim=-1)
        entropy = -(attn * (attn.clamp_min(1e-8)).log()).sum(dim=-1, keepdim=True)
        entropy = entropy / math.log(attn.shape[-1])
        confidence = attn.max(dim=-1, keepdim=True).values
        return entropy, confidence

class E_REM(nn.Module):
    """Evidential Reliability Estimation Module (E-REM) -- v3.

    Whether this changes the ablation/reliability numbers requires a full
    retrain (Section 13/14) and a fresh Section 18 run, exactly as with prior
    revisions -- report whatever that actually shows.
    """
    def __init__(self, text_dim=768, vis_dim=768, q_text_dim=3, q_img_dim=4,
                 hidden=256, n_evidence_heads=3, cross_dim=64, n_prototypes=8):
        super().__init__()
        self.norm_t = nn.LayerNorm(text_dim)
        self.norm_v = nn.LayerNorm(vis_dim)
        self.n_heads = n_evidence_heads

        self.cross_proj_t = nn.Linear(text_dim, cross_dim)
        self.cross_proj_v = nn.Linear(vis_dim, cross_dim)

        self.proto_attn = _PrototypeAttention(text_dim, n_prototypes=n_prototypes)

        rich_q_text_dim = q_text_dim + 3
        rich_q_img_dim  = q_img_dim + 3

        text_in = text_dim + rich_q_text_dim + cross_dim
        vis_in  = vis_dim + rich_q_img_dim + cross_dim

        self.text_in_proj = nn.Linear(text_in, hidden)
        self.vis_in_proj  = nn.Linear(vis_in,  hidden)

        self.text_blocks = nn.ModuleList([_EvidenceResBlock(hidden) for _ in range(2)])
        self.vis_blocks  = nn.ModuleList([_EvidenceResBlock(hidden) for _ in range(2)])

        self.text_heads = nn.ModuleList([nn.Linear(hidden, 2) for _ in range(n_evidence_heads)])
        self.vis_heads  = nn.ModuleList([nn.Linear(hidden, 2) for _ in range(n_evidence_heads)])

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, h_t, h_v, q_text, q_image):
        q_text_n  = F.normalize(q_text.float(),  dim=-1)
        q_image_n = F.normalize(q_image.float(), dim=-1)

        h_t_n = self.norm_t(h_t)
        h_v_n = self.norm_v(h_v)

        ent_t, conf_t = self.proto_attn(h_t_n)
        ent_v, conf_v = self.proto_attn(h_v_n)

        consistency = F.cosine_similarity(h_t_n.detach(), h_v_n.detach(), dim=-1).unsqueeze(-1)
        consistency = (consistency + 1.0) / 2.0

        rich_q_text  = torch.cat([q_text_n,  ent_t, conf_t, consistency], dim=-1)
        rich_q_image = torch.cat([q_image_n, ent_v, conf_v, consistency], dim=-1)

        cross_from_v = self.cross_proj_v(h_v_n.detach())
        cross_from_t = self.cross_proj_t(h_t_n.detach())

        x_t = torch.cat([h_t_n, rich_q_text,  cross_from_v], dim=-1)
        x_v = torch.cat([h_v_n, rich_q_image, cross_from_t], dim=-1)

        z_t = self.text_in_proj(x_t)
        z_v = self.vis_in_proj(x_v)

        for blk in self.text_blocks:
            z_t = blk(z_t)
        for blk in self.vis_blocks:
            z_v = blk(z_v)

        evidence_t = torch.stack([F.softplus(h(z_t)) for h in self.text_heads], dim=0).mean(dim=0)
        evidence_v = torch.stack([F.softplus(h(z_v)) for h in self.vis_heads],  dim=0).mean(dim=0)

        alpha_t = evidence_t + 1.0
        alpha_v = evidence_v + 1.0

        S_t = alpha_t[:, 0:1] + alpha_t[:, 1:2]
        u_t = 2.0 / S_t

        S_v = alpha_v[:, 0:1] + alpha_v[:, 1:2]
        u_v = 2.0 / S_v

        return u_t, u_v, alpha_t, alpha_v

print("Initializing E-REM v3 (prototype-attention richer reliability features)...")
erem = E_REM(n_prototypes=getattr(cfg, 'EREM_N_PROTOTYPES', 8))
erem.eval()

with torch.no_grad():
    u_t, u_v, alpha_t, alpha_v = erem(h_t, h_v, q_text, q_image)

print("\n--- E-REM OUTPUT SHAPES ---")
print(f"alpha_t Shape (Text Dirichlet) : {list(alpha_t.shape)}")
print(f"alpha_v Shape (Image Dirichlet): {list(alpha_v.shape)}")
print(f"u_t Shape (Text Uncertainty)   : {list(u_t.shape)}")
print(f"u_v Shape (Image Uncertainty)  : {list(u_v.shape)}")

num_samples = min(10, u_t.shape[0])
for i in range(num_samples):
    print(f"Sample {i+1}: u_t={u_t[i].item():.4f}  u_v={u_v[i].item():.4f}")

### 7.1 Text Uncertainty Illustration (10 Random Samples)


In [ ]:
import pandas as pd

print("Analyzing df_train based on extracted quality metrics...")

df_analysis = df_train.dropna(subset=['text', 'Norm Char', 'Norm Word', 'Norm Entropy']).copy()

df_analysis['Dataset Quality'] = (df_analysis['Norm Char'] + df_analysis['Norm Word'] + df_analysis['Norm Entropy']) / 3.0

df_analysis['Dataset Uncertainty (Proxy)'] = 1.0 - df_analysis['Dataset Quality']

df_sorted = df_analysis.sort_values(by='Dataset Uncertainty (Proxy)')

df_sorted['Text (Preview)'] = df_sorted['text'].astype(str).apply(
    lambda x: x[:70] + "..." if len(x) > 70 else x
)

cols_to_show = ['Text (Preview)', 'Dataset Uncertainty (Proxy)', 'Norm Char', 'Norm Word', 'Norm Entropy']

print('\n' + '═' * 110)
print(' ✅ TOP 10 MOST CERTAIN TEXTS (Information-Rich, Long, High Entropy)')
print('═' * 110)

df_certain = df_sorted.head(10)[cols_to_show].copy()

df_certain.iloc[:, 1:] = df_certain.iloc[:, 1:].round(4)
print(df_certain.to_string(index=False))

print('\n' + '═' * 110)
print(' ⚠️ TOP 10 MOST UNCERTAIN TEXTS (Short, Noisy, or Low Information)')
print('═' * 110)

df_uncertain = df_sorted.tail(10)[cols_to_show].iloc[::-1].copy()
df_uncertain.iloc[:, 1:] = df_uncertain.iloc[:, 1:].round(4)
print(df_uncertain.to_string(index=False))

### 7.2 Image Uncertainty Illustration (10 Random Samples)


In [ ]:
import pandas as pd

print("Analyzing df_train based on extracted image quality metrics...")

img_cols = ['image_path', 'Norm Bright', 'Norm Contrast', 'Norm Blur', 'Norm Noise']
df_img_analysis = df_train.dropna(subset=img_cols).copy()

df_img_analysis['Image Quality'] = (
    df_img_analysis['Norm Bright'] +
    df_img_analysis['Norm Contrast'] +
    df_img_analysis['Norm Blur'] +
    df_img_analysis['Norm Noise']
) / 4.0

df_img_analysis['Image Uncertainty (Proxy)'] = 1.0 - df_img_analysis['Image Quality']

df_img_sorted = df_img_analysis.sort_values(by='Image Uncertainty (Proxy)')

df_img_sorted['Image Path (Preview)'] = df_img_sorted['image_path'].astype(str).apply(
    lambda x: "..." + x[-65:] if len(x) > 68 else x
)

cols_to_show_img = [
    'Image Path (Preview)', 'Image Uncertainty (Proxy)',
    'Norm Bright', 'Norm Contrast', 'Norm Blur', 'Norm Noise'
]

print('\n' + '═' * 125)
print(' ✅ TOP 10 MOST CERTAIN IMAGES (Sharp, High Contrast, Well-Lit)')
print('═' * 125)

df_img_certain = df_img_sorted.head(10)[cols_to_show_img].copy()

df_img_certain.iloc[:, 1:] = df_img_certain.iloc[:, 1:].round(4)
print(df_img_certain.to_string(index=False))

print('\n' + '═' * 125)
print(' ⚠️ TOP 10 MOST UNCERTAIN IMAGES (Blurry, Dark, or Low Detail)')
print('═' * 125)

df_img_uncertain = df_img_sorted.tail(10)[cols_to_show_img].iloc[::-1].copy()
df_img_uncertain.iloc[:, 1:] = df_img_uncertain.iloc[:, 1:].round(4)
print(df_img_uncertain.to_string(index=False))

## 8. Uncertainty-Aware Soft Gating (UASG)


### 8.1 Text and Image Gating


In [ ]:
import torch
import torch.nn as nn

class UASG(nn.Module):
    """FiLM + SE-style gating branch, conditioned on evidential uncertainty."""
    def __init__(self, dim: int = 768, reduction: int = 8):
        super().__init__()

        self.eps_t = nn.Parameter(torch.tensor([-2.197]))
        self.eps_v = nn.Parameter(torch.tensor([-2.197]))

        bottleneck = max(dim // reduction, 16)

        self.film_t = nn.Sequential(
            nn.Linear(dim + 1, bottleneck), nn.GELU(), nn.Linear(bottleneck, dim * 2)
        )
        self.film_v = nn.Sequential(
            nn.Linear(dim + 1, bottleneck), nn.GELU(), nn.Linear(bottleneck, dim * 2)
        )

        self.beta_scale = 0.1
        self._init_weights()

    def _init_weights(self):
        for m in [self.film_t[-1], self.film_v[-1]]:
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, h_t, u_t, h_v, u_v):
        eps_t = torch.sigmoid(self.eps_t)
        gate_scale_t = eps_t + (1.0 - eps_t) * (1.0 - u_t)

        eps_v = torch.sigmoid(self.eps_v)
        gate_scale_v = eps_v + (1.0 - eps_v) * (1.0 - u_v)

        gb_t = self.film_t(torch.cat([h_t, u_t], dim=-1))
        gamma_t, beta_t = gb_t.chunk(2, dim=-1)
        gamma_t = torch.sigmoid(gamma_t)

        gb_v = self.film_v(torch.cat([h_v, u_v], dim=-1))
        gamma_v, beta_v = gb_v.chunk(2, dim=-1)
        gamma_v = torch.sigmoid(gamma_v)

        h_t_prime = (gate_scale_t * gamma_t) * h_t + self.beta_scale * torch.tanh(beta_t)
        h_v_prime = (gate_scale_v * gamma_v) * h_v + self.beta_scale * torch.tanh(beta_v)

        return h_t_prime, h_v_prime, eps_t, eps_v, gate_scale_t, gate_scale_v

print("Initializing FiLM+SE Gating Branch (UASG v3)...")
uasg = UASG().to(h_t.device)
uasg.eval()

with torch.no_grad():
    h_t_prime, h_v_prime, eps_t, eps_v, gate_scale_t, gate_scale_v = uasg(h_t, u_t, h_v, u_v)

print("\n--- TEXT MODALITY ---")
print(f"h_t_prime shape: {list(h_t_prime.shape)}")
print(f"Global scalar gating threshold (eps_t): {eps_t.item():.4f}")

print("\n--- VISION MODALITY ---")
print(f"h_v_prime shape: {list(h_v_prime.shape)}")
print(f"Global scalar gating threshold (eps_v): {eps_v.item():.4f}")

## 9. Contrastive Alignment with InfoNCE


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class InfoNCE_Alignment(nn.Module):
    """Contrastive Alignment module to align multimodal representations."""
    def __init__(self, init_tau=0.07):
        super().__init__()
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / init_tau))

    def forward(self, h_t_prime, h_v_prime):

        h_t_norm = F.normalize(h_t_prime, dim=-1)
        h_v_norm = F.normalize(h_v_prime, dim=-1)

        sim_matrix = (h_t_norm @ h_v_norm.t()) * torch.exp(self.logit_scale)

        batch_size = h_t_prime.shape[0]
        labels = torch.arange(batch_size, device=h_t_prime.device)
        loss = (F.cross_entropy(sim_matrix, labels) + F.cross_entropy(sim_matrix.t(), labels)) / 2.0

        return loss, sim_matrix

infonce = InfoNCE_Alignment()
infonce.eval()

num_batches = 10
batch_size = 32
embedding_dim = 768

with torch.no_grad():
    for batch_idx in range(num_batches):

        h_t_prime = torch.randn(batch_size, embedding_dim)
        h_v_prime = torch.randn(batch_size, embedding_dim)

        contrastive_loss, sim_matrix = infonce(h_t_prime, h_v_prime)
        current_batch_size = h_t_prime.shape[0]

        print(f"\n{'='*75}")
        print(f"--- 🔗 InfoNCE DIAGNOSTIC | BATCH {batch_idx + 1:02d} / {num_batches} ---")
        print(f"{'='*75}")

        print("📏 TENSOR DIMENSIONS VERIFICATION:")
        print(f"   Text Input (h_t_prime)  : {list(h_t_prime.shape)} -> [Batch, Embedding]")
        print(f"   Image Input (h_v_prime) : {list(h_v_prime.shape)} -> [Batch, Embedding]")
        print(f"   Similarity Matrix       : {list(sim_matrix.shape)} -> [Batch, Batch]")
        print(f"{'-'*75}")

        print(f"Current Contrastive Loss: {contrastive_loss.item():.4f}\n")
        print(f"{'Sample':<8} | {'Diag Score (True Match)':<25} | {'Off-Diag Noise (Avg)':<20}")
        print("-" * 65)

        for i in range(min(10, current_batch_size)):
            true_match = sim_matrix[i, i].item()
            row_noise = (sim_matrix[i].sum() - true_match) / (current_batch_size - 1)
            print(f"Sample {i+1:<3} | {true_match:>23.4f} | {row_noise:>20.4f}")

        avg_diag = sim_matrix.diag().mean().item()
        avg_off_diag = (sim_matrix.sum() - sim_matrix.diag().sum()) / (sim_matrix.numel() - current_batch_size)

        print("-" * 65)
        print(f"Batch Mean Diagonal Strength: {avg_diag:.4f}")
        print(f"Batch Mean Off-Diagonal Noise : {avg_off_diag:.4f}")

        if avg_diag > avg_off_diag:
            print("✅ STATUS: System is showing early signs of semantic alignment.")
        else:
            print("⚠️ STATUS: Random initialization detected; awaiting training.")

        print("\n--- Similarity Matrix (Top 10x10 Subset) ---")
        top_10_sim = sim_matrix[:10, :10].cpu().numpy()
        np.set_printoptions(precision=2, suppress=True, linewidth=100)
        print(top_10_sim)
        np.set_printoptions(edgeitems=3, infstr='inf', linewidth=75, nanstr='nan', precision=8, suppress=False, threshold=1000, formatter=None)

## 10. Uncertainty-Guided Cross-Modal Attention (UG-CMA)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def _make_head(in_dim, out_dim, dropout_p=0.1):
    return nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(in_dim, in_dim // 2),
        nn.GELU(),
        nn.Dropout(p=dropout_p),
        nn.Linear(in_dim // 2, out_dim)
    )

print("✓ _make_head function defined.")

class UGCMALayer(nn.Module):
    """One block of uncertainty-guided, bidirectional cross-attention with a
    post-attention FFN (pre-norm transformer style), so each layer can refine
    the fused representation rather than only mixing it once."""
    def __init__(self, dim: int = 768, n_heads: int = 8, ffn_mult: int = 4, dropout: float = 0.1):
        super().__init__()
        self.text_to_img = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=dropout)
        self.img_to_text = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=dropout)
        self.tau_t = nn.Parameter(torch.tensor([1.0]))
        self.tau_v = nn.Parameter(torch.tensor([1.0]))

        self.norm_t_attn = nn.LayerNorm(dim)
        self.norm_v_attn = nn.LayerNorm(dim)
        self.ffn_t = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim * ffn_mult, dim),
        )
        self.ffn_v = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim * ffn_mult, dim),
        )
        self.norm_t_ffn = nn.LayerNorm(dim)
        self.norm_v_ffn = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, h_t, h_v, u_t, u_v):
        q_t, q_v = h_t.unsqueeze(1), h_v.unsqueeze(1)

        attn_t2v, attn_w_t2v = self.text_to_img(query=q_t, key=q_v, value=q_v)
        penalty_v = torch.exp(-F.softplus(self.tau_v) * u_v)
        f_t = self.norm_t_attn(h_t + self.drop(attn_t2v.squeeze(1) * penalty_v))
        f_t = self.norm_t_ffn(f_t + self.drop(self.ffn_t(f_t)))

        attn_v2t, _ = self.img_to_text(query=q_v, key=q_t, value=q_t)
        penalty_t = torch.exp(-F.softplus(self.tau_t) * u_t)
        f_v = self.norm_v_attn(h_v + self.drop(attn_v2t.squeeze(1) * penalty_t))
        f_v = self.norm_v_ffn(f_v + self.drop(self.ffn_v(f_v)))

        return f_t, f_v, penalty_t, penalty_v, attn_w_t2v

class UG_CMA(nn.Module):
    """Uncertainty-Guided Cross-Modal Attention. UPGRADED: `n_layers` stacked
    UGCMALayer blocks (default 2, was implicitly 1) progressively refine the
    text/image representations before a learnable-gated fusion combines them,
    instead of a single attention pass + plain concat->linear."""
    def __init__(self, dim: int = 768, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.dim = dim
        self.n_layers = n_layers
        self.layers = nn.ModuleList([
            UGCMALayer(dim, n_heads, dropout=dropout) for _ in range(n_layers)
        ])
        self.fusion_proj = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
        )

        self.gate_proj = nn.Sequential(nn.Linear(dim * 2 + 2, dim), nn.Sigmoid())
        self.fusion_dropout = nn.Dropout(dropout)

    def forward(self, h_t_prime, h_v_prime, u_t, u_v):
        f_t, f_v = h_t_prime, h_v_prime
        penalty_t = penalty_v = attn_w = None
        for layer in self.layers:
            f_t, f_v, penalty_t, penalty_v, attn_w = layer(f_t, f_v, u_t, u_v)

        concat_out = torch.cat([f_t, f_v], dim=-1)
        m_fuse_raw = self.fusion_proj(concat_out)
        gate       = self.gate_proj(torch.cat([concat_out, u_t, u_v], dim=-1))
        m_fuse     = gate * m_fuse_raw + (1.0 - gate) * 0.5 * (f_t + f_v)

        return self.fusion_dropout(m_fuse), penalty_t, penalty_v

class RealDisasterDataset(Dataset):
    def __init__(self):

        self.num_samples = 1000

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):

        h_t = torch.randn(768)
        h_v = torch.randn(768)
        u_t = torch.rand(1)
        u_v = torch.rand(1)

        label = torch.randint(0, 4, (1,)).item()
        return h_t, h_v, u_t, u_v, label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_dim = 768
num_classes = 4

ug_cma = UG_CMA(dim=embedding_dim).to(device)
classifier_head = _make_head(in_dim=embedding_dim, out_dim=num_classes).to(device)

dataset = RealDisasterDataset()
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

ug_cma.train()
classifier_head.train()

for batch_idx, (batch_h_t, batch_h_v, batch_u_t, batch_u_v, batch_labels) in enumerate(dataloader):

    batch_h_t = batch_h_t.to(device)
    batch_h_v = batch_h_v.to(device)
    batch_u_t = batch_u_t.to(device)
    batch_u_v = batch_u_v.to(device)
    batch_labels = batch_labels.to(device)

    m_fuse, _, _ = ug_cma(batch_h_t, batch_h_v, batch_u_t, batch_u_v)

    logits = classifier_head(m_fuse)

    if batch_idx == 0:
        print("\n--- 📏 FULL PIPELINE TENSOR DIMENSIONS ---")
        print(f"🔹 1. Inputs to UG-CMA : Text {list(batch_h_t.shape)} | Image {list(batch_h_v.shape)}")
        print(f"🔹 2. Fused Output     : {list(m_fuse.shape)} -> Passes into Classification Head")
        print(f"🔹 3. Final Logits     : {list(logits.shape)} -> Ready for CrossEntropyLoss")
        print(f"🔹 4. True Labels      : {list(batch_labels.shape)} -> Ground truth for batch")

        print("\n--- 🧠 CLASSIFICATION HEAD LAYER-BY-LAYER TRACE ---")
        x = m_fuse
        for i, layer in enumerate(classifier_head):
            x = layer(x)
            print(f"   Step {i+1:<2} | {layer.__class__.__name__:<10} | Shape: {list(x.shape)}")
        print("-" * 65)

        break

## 11. Shared Multimodal Representation and Multi-Task Prediction Heads
$M_{fuse}$ -> Linear->LayerNorm->GELU->Dropout -> $H_{disaster}$ -> 6 heads (event, info, human, damage, verification, uncertainty).

In [ ]:
class CrisisMultiModal(nn.Module):
    def __init__(self, cfg, clip_visual):
        super().__init__()

        self.text_encoder = DeBERTaEncoder(cfg.ROBERTA_CKPT)
        self.vis_encoder  = CLIPVisionEncoder(clip_visual, text_dim=cfg.TEXT_DIM)
        D = cfg.TEXT_DIM

        self.rem  = E_REM(text_dim=D, vis_dim=D, q_text_dim=3, q_img_dim=4)
        self.rasg = UASG()
        self.bcmf = UG_CMA(dim=D, n_heads=8)
        self.mm_dropout = nn.Dropout(p=0.15)

        self.disaster_proj = nn.Sequential(
            nn.Linear(D, D),
            nn.LayerNorm(D),
            nn.GELU(),
            nn.Dropout(0.2),
        )

        self.head_event  = _make_head(D, cfg.N_EVENT)
        self.head_info   = _make_head(D, cfg.N_INFO)
        self.head_human  = _make_head(D, cfg.N_HUMAN)
        self.head_damage = _make_head(D, cfg.N_DAMAGE)

        self.head_verif  = _make_head(D + 3, getattr(cfg, 'N_VERIF', 2))

        self.head_uncert = nn.Sequential(
            nn.Linear(D, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Softplus(),
        )
        self._init_heads()

        self.ablation_mode = 'full'

    def _init_heads(self):
        for name, mod in self.named_modules():
            if isinstance(mod, nn.Linear) and 'roberta' not in name and 'encoder' not in name:
                nn.init.xavier_uniform_(mod.weight)
                if mod.bias is not None:
                    nn.init.zeros_(mod.bias)

    def heads_forward(self, h_disaster, verif_extra=None):
        """Applies all 5 classification heads to an already-fused representation.
        verif_extra: optional (B,3) tensor [consistency, u_t, u_v]; zeros if not given
        (e.g. Manifold-Mixup calls this on a mixed h_disaster with no matching u_t/u_v)."""
        if verif_extra is None:
            verif_extra = torch.zeros(h_disaster.size(0), 3, device=h_disaster.device, dtype=h_disaster.dtype)
        verif_in = torch.cat([h_disaster, verif_extra], dim=-1)
        logits = {
            'logits_event'  : self.head_event(h_disaster),
            'logits_info'   : self.head_info(h_disaster),
            'logits_human'  : self.head_human(h_disaster),
            'logits_damage' : self.head_damage(h_disaster),
            'logits_verif'  : self.head_verif(verif_in),
        }
        for k in logits:
            logits[k].clamp_(-15.0, 15.0)
        return logits

    def forward(self, input_ids, attention_mask, pixel_values, q_text, q_image):
        mode = getattr(self, 'ablation_mode', 'full')

        h_t = self.text_encoder(input_ids, attention_mask)
        h_v = self.vis_encoder(pixel_values)

        if mode == 'text_only':
            h_v = torch.zeros_like(h_v)
        elif mode == 'image_only':
            h_t = torch.zeros_like(h_t)

        u_t, u_v, alpha_t, alpha_v = self.rem(h_t, h_v, q_text, q_image)

        if mode == 'no_erem':
            u_t = torch.full_like(u_t, 0.5)
            u_v = torch.full_like(u_v, 0.5)

        if mode == 'no_rasg':
            h_t_prime, h_v_prime = h_t, h_v
        else:

            h_t_prime, h_v_prime, *_ = self.rasg(h_t, u_t, h_v, u_v)

        h_t_prime = self.mm_dropout(h_t_prime)
        h_v_prime = self.mm_dropout(h_v_prime)

        if mode == 'no_bcmf':
            m_fuse = h_t_prime + h_v_prime
        else:
            m_fuse, p_t, p_v = self.bcmf(h_t_prime, h_v_prime, u_t, u_v)

        h_disaster = self.disaster_proj(m_fuse)

        consistency = F.cosine_similarity(h_t_prime.detach(), h_v_prime.detach(), dim=-1, eps=1e-6).unsqueeze(-1)
        consistency = (consistency + 1.0) / 2.0
        verif_extra = torch.cat([consistency, u_t, u_v], dim=-1)
        heads_out   = self.heads_forward(h_disaster, verif_extra=verif_extra)
        uncertainty = self.head_uncert(h_disaster)

        return {
            **heads_out,
            'uncertainty'   : uncertainty,
            'u_t'           : u_t,
            'u_v'           : u_v,
            'alpha_t'       : alpha_t,
            'alpha_v'       : alpha_v,
            'h_disaster'    : h_disaster,

            'h_t_prime'     : h_t_prime,
            'h_v_prime'     : h_v_prime
        }

print('✅ CrisisMultiModal model class updated and ready.')

## 12. Multi-Task Training Objective

InfoNCE aligns text/image embeddings with a learnable temperature; H-MTL combines all task losses using Kendall's homoscedastic-uncertainty weighting so the model auto-balances task difficulty.


### 12.1 Correction: Uncertainty Loss Term

**Bug found on static review of Section 8/12 (loss function cell):**

```python
sigma2      = outputs['uncertainty'].clamp(1e-6, 10.0)
loss_uncert = torch.log(sigma2).mean()
```

`log(sigma2)` has no accompanying error term, so its gradient always pushes `sigma2` toward
the lower clamp (`1e-6`) regardless of whether predictions are accurate — a well-known
degenerate case of heteroscedastic-uncertainty losses when the precision-weighted error term
(`0.5 * exp(-log_sigma2) * error^2`) is dropped. Combined with `DynamicMultiTaskLoss`'s
homoscedastic auto-weighting (`log_vars`), this term could silently collapse the `uncertainty`
output toward a near-constant floor value over training, independent of the model's actual
reliability — undermining any downstream claim that `uncertainty`/`sigma2` reflects real
predictive confidence.

**Fix applied:** `loss_uncert` is no longer backpropagated. E-REM's own `loss_erem` (the
evidential deep learning / EDL objective on the Dirichlet parameters `alpha_t`/`alpha_v`) is
kept as the sole uncertainty-training signal, since it already contains a proper error term,
a variance term, and a KL regularizer — i.e. it is the mathematically complete version of
what `loss_uncert` was trying to approximate. `sigma2`/`loss_uncert` are still computed and
logged for diagnostic visibility only.

**Honesty note:** the existing results elsewhere in this notebook (Sections 15, 15.2, 15.3,
18) come from the checkpoint trained **before** this fix, i.e. with the old (degenerate)
`loss_uncert` term included. The E-REM uncertainty-spread diagnostic later in this notebook
(`u_t`/`u_v` std ≈ 0.13–0.14, not collapsed) is about `outputs['u_t']`/`outputs['u_v']`
(E-REM's own uncertainty channel), which is a **different tensor** from `outputs['uncertainty']`
(the one `loss_uncert` operated on) — so that diagnostic does **not** by itself prove
`outputs['uncertainty']` avoided collapsing. This is a genuine open question this fix
addresses going forward, not one the existing run can retroactively answer. **Re-training with
this fix and re-checking `outputs['uncertainty']`'s spread is required** to confirm the fix
mattered in practice; that re-run was not executed in this session (no GPU/dataset access).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler
import numpy as np

def get_class_weights(df, col, n_classes):
    idx_col = f'{col}_idx'
    if idx_col not in df.columns:
        return None
    counts   = np.bincount(df[idx_col].values, minlength=n_classes)
    counts   = np.maximum(counts, 1)
    weights  = 1.0 / counts
    weights  = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)

def get_class_balanced_weights(df, col, n_classes, beta=0.9999):
    """Class-Balanced weighting via the 'effective number of samples'"""
    idx_col = f'{col}_idx'
    if idx_col not in df.columns:
        return None
    counts = np.bincount(df[idx_col].values, minlength=n_classes)
    counts = np.maximum(counts, 1)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)

_use_cb = getattr(cfg, 'USE_CLASS_BALANCED_LOSS', False)
_cb_beta = getattr(cfg, 'CB_BETA', 0.9999)

W_EVENT  = get_class_weights(df_train, 'event_type', cfg.N_EVENT)
W_INFO   = get_class_weights(df_train, 'informative', cfg.N_INFO)

if _use_cb:
    W_HUMAN  = get_class_balanced_weights(df_train, 'humanitarian', cfg.N_HUMAN, beta=_cb_beta)
    W_DAMAGE = get_class_balanced_weights(df_train, 'damage', cfg.N_DAMAGE, beta=_cb_beta)
else:
    W_HUMAN  = get_class_weights(df_train, 'humanitarian', cfg.N_HUMAN)
    W_DAMAGE = get_class_weights(df_train, 'damage', cfg.N_DAMAGE)

print(f'Class weights computed (humanitarian/damage using {"Class-Balanced (eff. num. samples)" if _use_cb else "plain inverse-frequency"}):')
for name, w in [('event',W_EVENT),('info',W_INFO),('human',W_HUMAN),('damage',W_DAMAGE)]:
    if w is not None:
        print(f'  {name:8s}: {w.numpy().round(3)}')

def get_verif_class_weights(df, beta=0.99):
    if 'verif_idx' not in df.columns:
        return None
    verif_idx = df['verif_idx'].fillna(0).values.astype(int)
    counts = np.bincount(verif_idx, minlength=2)
    counts = np.maximum(counts, 1)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / weights.sum() * 2
    return torch.tensor(weights, dtype=torch.float32)

W_VERIF = get_verif_class_weights(df_train, beta=getattr(cfg, 'CB_BETA_VERIF', 0.99))
if W_VERIF is not None:
    _verif_kind = 'real cross-modal label' if globals().get('VERIF_IS_REAL', False) else 'PROXY label (fallback)'
    print(f'  {"verif":8s}: {W_VERIF.numpy().round(3)}  ({_verif_kind})')

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2., label_smoothing=0.0):
        super().__init__()
        self.weight = weight

        self.gamma = gamma

        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none',
                                   label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

def _kl_dirichlet_to_uniform(alpha):
    K = alpha.shape[1]
    beta = torch.ones_like(alpha)
    S_alpha = alpha.sum(dim=1, keepdim=True)
    S_beta  = beta.sum(dim=1, keepdim=True)
    lnB_alpha = torch.lgamma(S_alpha) - torch.lgamma(alpha).sum(dim=1, keepdim=True)
    lnB_beta  = torch.lgamma(S_beta)  - torch.lgamma(beta).sum(dim=1, keepdim=True)
    dg_alpha  = torch.digamma(alpha)
    dg_S      = torch.digamma(S_alpha)
    kl = ((alpha - beta) * (dg_alpha - dg_S)).sum(dim=1, keepdim=True) + lnB_alpha - lnB_beta
    return kl.squeeze(-1)

def edl_loss(alpha, target_idx, kl_weight=0.05):
    """alpha: (B, 2) Dirichlet params from E-REM. target_idx: (B,) long, 0='true'/reliable, 1='false'/unreliable."""
    y = F.one_hot(target_idx, num_classes=alpha.shape[1]).float()
    S = alpha.sum(dim=1, keepdim=True)
    p = alpha / S
    err = ((y - p) ** 2).sum(dim=1)
    var = (alpha * (S - alpha) / (S * S * (S + 1))).sum(dim=1)

    alpha_tilde = y + (1.0 - y) * alpha
    kl = _kl_dirichlet_to_uniform(alpha_tilde)
    return (err + var + kl_weight * kl).mean()

def infonce_loss(features_a, features_b, temp):
    features_a = F.normalize(features_a, dim=-1)
    features_b = F.normalize(features_b, dim=-1)
    sim_matrix = torch.matmul(features_a, features_b.T) / temp
    labels = torch.arange(sim_matrix.size(0), dtype=torch.long, device=sim_matrix.device)
    loss_a = F.cross_entropy(sim_matrix, labels)
    loss_b = F.cross_entropy(sim_matrix.T, labels)
    return (loss_a + loss_b) / 2

def supcon_loss(features, labels, temperature=0.1):
    device = features.device
    features = F.normalize(features, dim=-1)
    batch_size = features.shape[0]
    if batch_size < 2:
        return torch.zeros((), device=device)
    sim = torch.matmul(features, features.T) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    labels = labels.view(-1, 1)
    mask_pos = (labels == labels.T).float().to(device)
    mask_self = torch.eye(batch_size, device=device)
    mask_pos = mask_pos - mask_self
    exp_sim = torch.exp(sim) * (1.0 - mask_self)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True).clamp_min(1e-8))
    n_pos = mask_pos.sum(dim=1)
    valid = n_pos > 0
    if valid.sum() == 0:
        return torch.zeros((), device=device)
    mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1)[valid] / n_pos[valid].clamp_min(1e-8)
    return -mean_log_prob_pos.mean()

class DynamicMultiTaskLoss(nn.Module):
    def __init__(self, cfg, w_event=None, w_info=None, w_human=None, w_damage=None, w_verif=None):
        super().__init__()
        self.cfg = cfg
        self.ce_event  = nn.CrossEntropyLoss(weight=w_event.to(DEVICE, non_blocking=True) if w_event is not None else None, label_smoothing=0.1)
        self.ce_info   = nn.CrossEntropyLoss(weight=w_info.to(DEVICE, non_blocking=True) if w_info is not None else None, label_smoothing=0.1)
        _gamma_human = getattr(cfg, 'FOCAL_GAMMA_HUMAN', getattr(cfg, 'FOCAL_GAMMA_HUD', 2.0))
        _gamma_damage = getattr(cfg, 'FOCAL_GAMMA_DAMAGE', getattr(cfg, 'FOCAL_GAMMA_HUD', 2.0))
        _hd_smooth = getattr(cfg, 'HUMAN_DAMAGE_LABEL_SMOOTHING', 0.0)
        self.ce_human  = FocalLoss(weight=w_human.to(DEVICE, non_blocking=True) if w_human is not None else None, gamma=_gamma_human, label_smoothing=_hd_smooth)
        self.ce_damage = FocalLoss(weight=w_damage.to(DEVICE, non_blocking=True) if w_damage is not None else None, gamma=_gamma_damage, label_smoothing=_hd_smooth)

        _gamma_verif = getattr(cfg, 'FOCAL_GAMMA_VERIF', 1.5)
        _verif_smooth = getattr(cfg, 'VERIF_LABEL_SMOOTHING', 0.05)
        self.ce_verif  = FocalLoss(weight=w_verif.to(DEVICE, non_blocking=True) if w_verif is not None else None, gamma=_gamma_verif, label_smoothing=_verif_smooth)

        self.infonce_temp = nn.Parameter(torch.tensor([-2.659]))
        self.use_infonce = True

        self.log_vars = nn.Parameter(torch.zeros(9))
        self._log_var_clamp = getattr(cfg, 'LOG_VAR_CLAMP', (-3.0, 3.0))
        self.use_supcon = getattr(cfg, 'USE_SUPCON', False)
        self.supcon_temp = getattr(cfg, 'SUPCON_TEMP', 0.1)
        self.supcon_task = getattr(cfg, 'SUPCON_TASK', 'event_type')

    @torch.no_grad()
    def clamp_log_vars(self):
        lo, hi = self._log_var_clamp
        self.log_vars.clamp_(lo, hi)

    def forward(self, outputs, batch):
        le = batch['label_event'].to(DEVICE, non_blocking=True)
        li = batch['label_info'].to(DEVICE, non_blocking=True)
        lh = batch['label_human'].to(DEVICE, non_blocking=True)
        ld = batch['label_damage'].to(DEVICE, non_blocking=True)

        loss_event  = self.ce_event(outputs['logits_event'],  le)
        loss_info   = self.ce_info(outputs['logits_info'],    li)
        loss_human  = self.ce_human(outputs['logits_human'],  lh)
        loss_damage = self.ce_damage(outputs['logits_damage'], ld)

        verif_label = batch['label_verif'].to(DEVICE, non_blocking=True)
        loss_verif  = self.ce_verif(outputs['logits_verif'], verif_label)

        temp = F.softplus(self.infonce_temp)
        loss_infonce = (infonce_loss(outputs['h_t_prime'], outputs['h_v_prime'], temp)
                        if getattr(self, 'use_infonce', True)
                        else torch.zeros((), device=outputs['h_t_prime'].device))

        # --- FIXED (journal upgrade) -------------------------------------------------
        # Original: loss_uncert = torch.log(sigma2).mean()
        # This term has NO error component, so gradient descent on it alone always wants
        # sigma2 -> its lower clamp (1e-6), regardless of whether the prediction is actually
        # reliable. It is a degenerate remainder of the standard heteroscedastic-uncertainty
        # objective (Kendall & Gal, 2017), which requires a precision-weighted error term:
        #   L = 0.5 * exp(-log_sigma2) * error^2 + 0.5 * log_sigma2
        # Minimizing log(sigma2) with no error term is not a valid uncertainty objective; it is
        # a one-directional pressure toward the clamp floor, which would make `uncertainty`
        # converge to a near-constant, uninformative value if this term's weight in the total
        # loss (log_vars[6], the 'uncert' task slot) were ever allowed to dominate.
        #
        # E-REM already has a principled, complete uncertainty objective: `loss_erem` below is
        # the evidential deep learning (EDL) loss (Sensoy et al., 2018) on the Dirichlet
        # parameters alpha_t/alpha_v, which DOES include an error term (`err`), a variance term
        # (`var`), and a KL-regularizer -- i.e. it already penalizes both wrong predictions and
        # miscalibrated confidence. `loss_uncert` was therefore redundant on top of `loss_erem`
        # as well as mathematically degenerate on its own.
        #
        # Fix: drop `loss_uncert` from the total loss (E-REM's `loss_erem` is kept as the sole,
        # principled uncertainty objective). We still compute and log sigma2 stats for
        # diagnostic visibility, but they no longer participate in backprop.
        sigma2 = outputs['uncertainty'].clamp(1e-6, 10.0)
        with torch.no_grad():
            loss_uncert = torch.log(sigma2).mean()  # diagnostic only, not backpropagated

        loss_erem = edl_loss(outputs['alpha_t'], verif_label) + edl_loss(outputs['alpha_v'], verif_label)

        if self.use_supcon:

            sc_labels = batch['label_event'].to(DEVICE, non_blocking=True) if 'label_event' in batch else le
            loss_supcon = supcon_loss(outputs['h_disaster'], sc_labels, temperature=self.supcon_temp)
        else:
            loss_supcon = torch.zeros((), device=outputs['h_disaster'].device)

        losses = {
            'event': loss_event,
            'info': loss_info,
            'human': loss_human,
            'damage': loss_damage,
            'verif': loss_verif,
            'infonce': loss_infonce,
            # 'uncert' intentionally excluded from backprop (see fix note above) -- kept only
            # in `breakdown` for diagnostic logging, using a zero-grad placeholder here so the
            # dynamic-weighting loop below still has 9 slots and old log_vars checkpoints load.
            'uncert': torch.zeros((), device=loss_event.device),
            'erem': loss_erem,
            'supcon': loss_supcon,
        }

        loss_total = 0.0
        for i, (name, l) in enumerate(losses.items()):
            loss_total = loss_total + torch.exp(-self.log_vars[i]) * l + self.log_vars[i]

        breakdown = {
            'event': loss_event.item(),
            'info': loss_info.item(),
            'human': loss_human.item(),
            'damage': loss_damage.item(),
            'verif': loss_verif.item(),
            'uncert': loss_uncert.item(),  # diagnostic (no-grad) value, see fix note above
            'infonce': loss_infonce.item(),
            'erem': loss_erem.item(),
            'supcon': loss_supcon.item(),
        }
        return loss_total, breakdown

    def mixup_forward(self, mixed_logits, batch, perm, lam):
        """Manifold-Mixup auxiliary loss (Verma et al., ICML 2019; building on"""
        le = batch['label_event'].to(DEVICE, non_blocking=True)
        li = batch['label_info'].to(DEVICE, non_blocking=True)
        lh = batch['label_human'].to(DEVICE, non_blocking=True)
        ld = batch['label_damage'].to(DEVICE, non_blocking=True)
        verif = batch['label_verif'].to(DEVICE, non_blocking=True)

        def mix_ce(loss_fn, logits, y):
            y_b = y[perm]
            return lam * loss_fn(logits, y) + (1.0 - lam) * loss_fn(logits, y_b)

        l_event  = mix_ce(self.ce_event,  mixed_logits['logits_event'],  le)
        l_info   = mix_ce(self.ce_info,   mixed_logits['logits_info'],   li)
        l_human  = mix_ce(self.ce_human,  mixed_logits['logits_human'],  lh)
        l_damage = mix_ce(self.ce_damage, mixed_logits['logits_damage'], ld)
        l_verif  = mix_ce(self.ce_verif,  mixed_logits['logits_verif'],  verif)

        total = (self.cfg.LAMBDA_EVENT  * l_event  + self.cfg.LAMBDA_INFO  * l_info +
                 self.cfg.LAMBDA_HUMAN  * l_human  + self.cfg.LAMBDA_DAMAGE * l_damage +
                 self.cfg.LAMBDA_VERIF  * l_verif)
        breakdown = {
            'mix_event': l_event.item(), 'mix_info': l_info.item(),
            'mix_human': l_human.item(), 'mix_damage': l_damage.item(),
            'mix_verif': l_verif.item(),
        }
        return total, breakdown

def set_encoder_trainable(model, trainable: bool):
    base = model.module if hasattr(model, 'module') else model
    for p in base.text_encoder.parameters():
        p.requires_grad = trainable
    for p in base.vis_encoder.parameters():
        p.requires_grad = trainable

MODEL = CrisisMultiModal(cfg, _clip_model.visual).to(DEVICE, non_blocking=True)
if N_GPUS > 1:
    MODEL = nn.DataParallel(MODEL)
print(f'Model parameters: {sum(p.numel() for p in MODEL.parameters()):,}')
print(f'Trainable params: {sum(p.numel() for p in MODEL.parameters() if p.requires_grad):,}')

def get_optimizer(model, criterion, cfg):
    """AdamW with automatic layer-wise LR decay (LLRD) for the two pretrained"""
    base = model.module if isinstance(model, nn.DataParallel) else model
    decay_rate = 0.9
    use_no_decay = getattr(cfg, 'USE_NO_DECAY_GROUPS', True)

    def _is_no_decay(name):

        lname = name.lower()
        return name.endswith('.bias') or 'layernorm' in lname or lname.endswith('norm.weight') or '.norm.' in lname

    def llrd_groups(encoder, base_lr, name_filter):
        named = [(n, p) for n, p in encoder.named_parameters() if p.requires_grad]

        groups = {}
        for n, p in named:
            depth = None
            for tok in n.split('.'):
                if tok.isdigit():
                    depth = int(tok)
                    break
            groups.setdefault(depth, []).append((n, p))
        max_depth = max([d for d in groups if d is not None], default=0)
        param_groups = []
        for depth, named_params in groups.items():
            if depth is None:
                lr = base_lr
            else:
                lr = base_lr * (decay_rate ** (max_depth - depth))
            if use_no_decay:
                decay_p    = [p for n, p in named_params if not _is_no_decay(n)]
                no_decay_p = [p for n, p in named_params if _is_no_decay(n)]
                if decay_p:
                    param_groups.append({'params': decay_p, 'lr': lr, 'weight_decay': cfg.WEIGHT_DECAY})
                if no_decay_p:
                    param_groups.append({'params': no_decay_p, 'lr': lr, 'weight_decay': 0.0})
            else:
                param_groups.append({'params': [p for _, p in named_params], 'lr': lr, 'weight_decay': cfg.WEIGHT_DECAY})
        return param_groups

    encoder_ids = {id(p) for p in base.text_encoder.parameters()} | \
                  {id(p) for p in base.vis_encoder.parameters()}

    text_groups = llrd_groups(base.text_encoder, cfg.LR_ROBERTA, 'text')
    vis_groups  = llrd_groups(base.vis_encoder,  cfg.LR_CLIP,    'vis')

    other_named = [(n, p) for n, p in model.named_parameters() if id(p) not in encoder_ids and p.requires_grad]
    other_named.extend([(n, p) for n, p in criterion.named_parameters() if p.requires_grad])

    if use_no_decay:
        other_decay    = [p for n, p in other_named if not _is_no_decay(n)]
        other_no_decay = [p for n, p in other_named if _is_no_decay(n)]
        other_groups = []
        if other_decay:
            other_groups.append({'params': other_decay, 'lr': cfg.LR_HEAD, 'weight_decay': cfg.WEIGHT_DECAY})
        if other_no_decay:
            other_groups.append({'params': other_no_decay, 'lr': cfg.LR_HEAD, 'weight_decay': 0.0})
    else:
        other_groups = [{'params': [p for _, p in other_named], 'lr': cfg.LR_HEAD, 'weight_decay': cfg.WEIGHT_DECAY}]

    param_groups = text_groups + vis_groups + other_groups
    return optim.AdamW(param_groups)

CRITERION  = DynamicMultiTaskLoss(cfg, W_EVENT, W_INFO, W_HUMAN, W_DAMAGE, W_VERIF).to(DEVICE, non_blocking=True)
OPTIMIZER  = get_optimizer(MODEL, CRITERION, cfg)

_schedule_epochs = getattr(cfg, 'EFFECTIVE_SCHEDULE_EPOCHS', cfg.EPOCHS)
total_steps  = len(dl_train) * _schedule_epochs
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
SCHEDULER    = get_cosine_schedule_with_warmup(OPTIMIZER, warmup_steps, total_steps)
SCALER       = GradScaler(enabled=USE_AMP)

print(f'\nOptimizer    : AdamW (differential LR)')
print(f'Scheduler    : Linear warmup ({warmup_steps} steps) + decay')
print(f'AMP Scaler   : {USE_AMP}')
print('o. Loss, optimizer, and scheduler ready.')

print('\n' + '═' * 80)
print(' 🧪 SANITY TESTING FULL PIPELINE GRADIENTS')
print('═' * 80)

test_batch = next(iter(dl_train))
MODEL.train()

outputs = MODEL(
    input_ids=test_batch['input_ids'].to(DEVICE),
    attention_mask=test_batch['attention_mask'].to(DEVICE),
    pixel_values=test_batch['pixel_values'].to(DEVICE),
    q_text=test_batch['q_text'].to(DEVICE),
    q_image=test_batch['q_image'].to(DEVICE)
)

loss_total, loss_breakdown = CRITERION(outputs, test_batch)

OPTIMIZER.zero_grad()
loss_total.backward()
OPTIMIZER.step()

print("\n--- 📊 DYNAMIC LOSS BREAKDOWN (1st Step) ---")
df_loss = pd.DataFrame([loss_breakdown])
print(df_loss.to_string(index=False))
print(f"\n🚀 Total Weighted Backpropagated Loss: {loss_total.item():.4f}")
print("✓ Multi-task optimization loop successfully validated on genuine distribution!")

## 13. Model Training
- **13:** builds `CrisisMultiModal`, `DynamicMultiTaskLoss`, AdamW (discriminative LR), scheduler, gradient checkpointing
- **14:** `run_epoch()` — forward → H-MTL loss → backward → clip → step; outer loop trains with early stopping on val F1


In [ ]:
import os
import time
import gc
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from torch.cuda.amp import autocast

def run_epoch(model, loader, criterion, optimizer=None, scaler=None, is_train=True):
    model.train() if is_train else model.eval()
    total_loss = 0.0

    tasks = ['event', 'info', 'human', 'damage']
    all_preds = {t: [] for t in tasks}
    all_trues = {t: [] for t in tasks}

    loss_breakdown = {k: 0.0 for k in ['event','info','human','damage','verif','uncert','erem','supcon']}

    if is_train and optimizer is not None:
        optimizer.zero_grad(set_to_none=True)

    ctx = torch.enable_grad() if is_train else torch.inference_mode()

    with ctx:
        for i, batch in enumerate(tqdm(loader, desc='Training' if is_train else 'Evaluating', leave=False)):
            ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
            mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
            pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
            qt   = batch['q_text'].to(DEVICE, non_blocking=True)
            qi   = batch['q_image'].to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                outputs = model(ids, mask, pix, qt, qi)
                loss, breakdown = criterion(outputs, batch)

                use_mix = (
                    is_train and getattr(cfg, 'USE_MANIFOLD_MIXUP', False)
                    and float(torch.rand(1)) < getattr(cfg, 'MIXUP_PROB', 0.5)
                    and outputs['h_disaster'].size(0) > 1
                )
                if use_mix:
                    base_model = model.module if hasattr(model, 'module') else model
                    h_dis = outputs['h_disaster']
                    perm  = torch.randperm(h_dis.size(0), device=h_dis.device)
                    lam   = float(np.random.beta(cfg.MIXUP_ALPHA, cfg.MIXUP_ALPHA))
                    mixed_h      = lam * h_dis + (1.0 - lam) * h_dis[perm]
                    mixed_logits = base_model.heads_forward(mixed_h)
                    mix_loss, mix_breakdown = criterion.mixup_forward(mixed_logits, batch, perm, lam)

                    _mix_blend = getattr(cfg, 'MIXUP_LOSS_BLEND', 0.3)
                    loss = loss + _mix_blend * mix_loss
                    breakdown = {**breakdown, **mix_breakdown}

            accum_steps = max(1, getattr(cfg, 'GRAD_ACCUM_STEPS', 1)) if is_train else 1
            step_now = is_train and (((i + 1) % accum_steps == 0) or (i + 1 == len(loader)))
            if is_train:
                loss_scaled = loss / accum_steps
                if scaler:
                    scaler.scale(loss_scaled).backward()
                    if step_now:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), getattr(cfg, 'MAX_GRAD_NORM', 1.0))
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad(set_to_none=True)

                        base_crit = criterion.module if hasattr(criterion, 'module') else criterion
                        if hasattr(base_crit, 'clamp_log_vars'):
                            base_crit.clamp_log_vars()
                        if SCHEDULER is not None:
                            SCHEDULER.step()
                else:
                    loss_scaled.backward()
                    if step_now:
                        nn.utils.clip_grad_norm_(model.parameters(), getattr(cfg, 'MAX_GRAD_NORM', 1.0))
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        base_crit = criterion.module if hasattr(criterion, 'module') else criterion
                        if hasattr(base_crit, 'clamp_log_vars'):
                            base_crit.clamp_log_vars()
                        if SCHEDULER is not None:
                            SCHEDULER.step()

            total_loss += loss.item()
            for k in loss_breakdown:
                if k in breakdown:
                    loss_breakdown[k] += breakdown[k]

            for t in tasks:
                preds = outputs[f'logits_{t}'].argmax(dim=-1).cpu().numpy()
                trues = batch[f'label_{t}'].cpu().numpy()
                all_preds[t].extend(preds)
                all_trues[t].extend(trues)

            del outputs, loss, breakdown

    n = len(loader)
    avg_loss = total_loss / n

    per_task_f1 = {t: f1_score(all_trues[t], all_preds[t], average='weighted', zero_division=0) for t in tasks}
    avg_f1 = sum(per_task_f1.values()) / len(per_task_f1)

    ckpt_tasks = getattr(cfg, 'CKPT_TASKS', tasks)
    target_f1  = sum(per_task_f1[t] for t in ckpt_tasks) / len(ckpt_tasks)

    for k in loss_breakdown:
        loss_breakdown[k] /= n

    return avg_loss, avg_f1, loss_breakdown, per_task_f1, target_f1

HISTORY = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[], 'val_f1_target':[], 'val_f1_per_task':[], 'log_vars':[], 'tau':[], 'eps':[]}
best_val_f1 = 0.0
best_target_f1 = 0.0
nobetter    = 0

print('🚀 Starting training…')
print(f'   Epochs: {cfg.EPOCHS}  |  Batch: {cfg.BATCH_SIZE}  |  Device: {DEVICE}')
print('─' * 80)

_freeze_epochs = getattr(cfg, 'FREEZE_ENCODER_EPOCHS', 0)
if _freeze_epochs > 0:
    set_encoder_trainable(MODEL, False)
    print(f'🧊 Encoders frozen for the first {_freeze_epochs} epoch(s) (heads/fusion only).')
else:
    print('Encoders trainable from epoch 1 (FREEZE_ENCODER_EPOCHS=0).')

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()

    if _freeze_epochs > 0 and epoch == _freeze_epochs + 1:
        set_encoder_trainable(MODEL, True)
        print(f'🔥 Unfreezing DeBERTa-v3-base + CLIP ViT-B/32 at epoch {epoch} (resuming normal LLRD fine-tuning).')

    tr_loss, tr_f1, tr_bd, tr_pt, tr_target  = run_epoch(MODEL, dl_train, CRITERION, OPTIMIZER, SCALER, is_train=True)
    va_loss, va_f1, va_bd, va_pt, va_target  = run_epoch(MODEL, dl_dev,   CRITERION, is_train=False)

    HISTORY['train_loss'].append(tr_loss)
    HISTORY['val_loss'].append(va_loss)
    HISTORY['train_f1'].append(tr_f1)
    HISTORY['val_f1'].append(va_f1)
    HISTORY['val_f1_target'].append(va_target)
    HISTORY['val_f1_per_task'].append(va_pt)

    HISTORY['log_vars'].append(CRITERION.log_vars.detach().cpu().numpy().copy())
    base_m = MODEL.module if hasattr(MODEL, 'module') else MODEL

    HISTORY['tau'].append([[l.tau_t.item(), l.tau_v.item()] for l in base_m.bcmf.layers])
    HISTORY['eps'].append([base_m.rasg.eps_t.item(), base_m.rasg.eps_v.item()])

    improved = '⭐' if va_target > best_target_f1 else '  '
    if va_target > best_target_f1:
        best_target_f1 = va_target
        best_val_f1    = va_f1
        nobetter       = 0

        torch.save(MODEL.state_dict(), cfg.CKPT_PATH)
    else:
        nobetter += 1

    TOP_K_CKPTS = globals().setdefault('TOP_K_CKPTS', [])
    ckpt_path_k = os.path.join(str(cfg.OUT_DIR), f'ckpt_epoch{epoch:02d}_f1{va_target:.4f}.pt')
    torch.save(MODEL.state_dict(), ckpt_path_k)
    TOP_K_CKPTS.append((va_target, ckpt_path_k))
    TOP_K_CKPTS.sort(key=lambda x: x[0], reverse=True)

    while len(TOP_K_CKPTS) > 3:
        _, stale_path = TOP_K_CKPTS.pop()
        if os.path.exists(stale_path):
            os.remove(stale_path)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{cfg.EPOCHS} {improved} | '
          f'Loss: {tr_loss:.4f}/{va_loss:.4f} | '
          f'wF1(4-avg): {tr_f1:.4f}/{va_f1:.4f} | '
          f'wF1(target={"+".join(cfg.CKPT_TASKS)}): {va_target:.4f} | '
          f'Best-target: {best_target_f1:.4f} | {elapsed:.0f}s')
    print(f'  Per-task val wF1 → event:{va_pt["event"]:.4f}  info:{va_pt["info"]:.4f}  '
          f'human:{va_pt["human"]:.4f}  damage:{va_pt["damage"]:.4f}')
    print(f'  Head losses → event:{tr_bd["event"]:.3f}  info:{tr_bd["info"]:.3f}  '
          f'human:{tr_bd["human"]:.3f}  damage:{tr_bd["damage"]:.3f}  '
          f'verif:{tr_bd["verif"]:.3f}  uncert:{tr_bd["uncert"]:.3f}')

    if getattr(cfg, 'PATIENCE', 5) and nobetter >= cfg.PATIENCE:
        print(f'\n⏹  Early stopping triggered (no improvement for {cfg.PATIENCE} epochs).')
        break

    gc.collect()
    torch.cuda.empty_cache()

print(f'\n✅ Training complete.  Best target wF1 (human+damage avg): {best_target_f1:.4f}  |  Best 4-task avg wF1: {best_val_f1:.4f}')

if 'TOP_K_CKPTS' in globals() and len(TOP_K_CKPTS) >= 2:
    print(f'\n🔀 Averaging top-{len(TOP_K_CKPTS)} checkpoints (SWA-style) ...')
    avg_state = None
    for _, p in TOP_K_CKPTS:
        sd = torch.load(p, map_location='cpu')
        if avg_state is None:
            avg_state = {k: v.clone().float() for k, v in sd.items()}
        else:
            for k in avg_state:
                avg_state[k] += sd[k].float()
    for k in avg_state:
        avg_state[k] /= len(TOP_K_CKPTS)
        avg_state[k] = avg_state[k].to(sd[k].dtype)

    _tmp_model = MODEL
    _tmp_model.load_state_dict(avg_state)
    _, swa_val_f1, _, swa_pt, swa_target = run_epoch(_tmp_model, dl_dev, CRITERION, is_train=False)
    print(f'   SWA dev target wF1: {swa_target:.4f}  (single-best target was {best_target_f1:.4f})')

    _tol = getattr(cfg, 'SWA_PER_TASK_TOLERANCE', 0.01)
    _best_pt = HISTORY['val_f1_per_task'][-1] if HISTORY['val_f1_per_task'] else {}
    _no_regression = all(swa_pt.get(t, 0.0) >= _best_pt.get(t, 0.0) - _tol for t in swa_pt)

    if swa_target > best_target_f1 and _no_regression:
        print('   ✅ SWA average improves on the single best checkpoint (no per-task regression) — saving as final best.')
        torch.save(avg_state, cfg.CKPT_PATH)
        best_target_f1 = swa_target
        best_val_f1 = swa_val_f1
    else:
        if swa_target > best_target_f1 and not _no_regression:
            print('   ↩️  SWA improved the target average but regressed an individual task beyond tolerance — keeping single-best checkpoint.')
        else:
            print('   ↩️  Single best checkpoint remains superior — reloading it.')
        MODEL.load_state_dict(torch.load(cfg.CKPT_PATH, map_location=DEVICE))

gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass

## 14. Calibration and Test-Time Augmentation
Applied on top of the trained model to boost humanitarian/damage F1:
1. **Logit-bias calibration** — per-class bias fit on DEV only, maximizes weighted-F1
2. **TTA** — averages predictions from image and its horizontal flip

Both are config-gated (`USE_LOGIT_CALIBRATION`, `USE_TTA`) and included as ablation rows in Section 18.


> **Status: Existing results from prior executed run; not regenerated in this environment.**

In [ ]:
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import f1_score

@torch.inference_mode()
def _collect_dev_logits(model, loader):

    model.eval()
    logits = {'human': [], 'damage': [], 'verif': []}
    trues  = {'human': [], 'damage': [], 'verif': []}
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
        qt   = batch['q_text'].to(DEVICE, non_blocking=True)
        qi   = batch['q_image'].to(DEVICE, non_blocking=True)
        out  = model(ids, mask, pix, qt, qi)
        logits['human'].append(out['logits_human'].float().cpu())
        logits['damage'].append(out['logits_damage'].float().cpu())
        logits['verif'].append(out['logits_verif'].float().cpu())
        trues['human'].extend(batch['label_human'].numpy())
        trues['damage'].extend(batch['label_damage'].numpy())
        trues['verif'].extend(batch['label_verif'].numpy())
    logits = {k: torch.cat(v, dim=0) for k, v in logits.items()}
    trues  = {k: np.array(v) for k, v in trues.items()}
    return logits, trues

def _fit_temperature(logits, y_true, n_iters=100, lr=0.05):
    """v3 NEW: scalar temperature scaling (Guo et al. 2017), fit on DEV by
    gradient descent on NLL. Applied BEFORE the per-class bias search, so the
    combination is a genuine 'vector+bias' calibration rather than bias-only.
    Improves probability calibration (not just argmax accuracy) which the
    original bias-only search did not directly target."""

    logits = logits.clone().detach().requires_grad_(False)
    logT = torch.zeros(1, requires_grad=True)
    y = torch.tensor(y_true, dtype=torch.long)
    opt = torch.optim.LBFGS([logT], lr=lr, max_iter=n_iters)

    def closure():
        opt.zero_grad()
        T = torch.exp(logT)
        loss = F.cross_entropy(logits / T, y)
        loss.backward()
        return loss
    opt.step(closure)
    return torch.exp(logT).detach()

def _fit_bias(logits, y_true, n_classes, n_iters=300, lr=0.05, seed=42):
    """Coordinate-ascent search for a per-class additive logit bias that"""
    rng = np.random.RandomState(seed)
    best_bias = torch.zeros(n_classes)
    best_f1   = f1_score(y_true, logits.argmax(dim=-1).numpy(), average='weighted', zero_division=0)
    for _ in range(n_iters):
        c    = rng.randint(n_classes)
        step = rng.choice([-lr, lr])
        trial = best_bias.clone()
        trial[c] += step
        preds = (logits + trial).argmax(dim=-1).numpy()
        f1 = f1_score(y_true, preds, average='weighted', zero_division=0)
        if f1 > best_f1:
            best_f1, best_bias = f1, trial
    return best_bias, best_f1

CALIB_BIAS = {'human': torch.zeros(cfg.N_HUMAN), 'damage': torch.zeros(cfg.N_DAMAGE), 'verif': torch.zeros(2)}
CALIB_TEMP = {'human': torch.ones(1), 'damage': torch.ones(1), 'verif': torch.ones(1)}
_calib_method = getattr(cfg, 'CALIBRATION_METHOD', 'bias')
if getattr(cfg, 'USE_LOGIT_CALIBRATION', False):
    print(f'Fitting calibration on the DEV set (human, damage, verif) -- method={_calib_method} ...')
    _dev_logits, _dev_true = _collect_dev_logits(MODEL, dl_dev)
    for _task, _n_cls in [('human', cfg.N_HUMAN), ('damage', cfg.N_DAMAGE), ('verif', 2)]:
        _raw_f1 = f1_score(_dev_true[_task], _dev_logits[_task].argmax(dim=-1).numpy(), average='weighted', zero_division=0)
        _work_logits = _dev_logits[_task]
        if _calib_method == 'temperature_and_bias':
            _T = _fit_temperature(_work_logits, _dev_true[_task])
            CALIB_TEMP[_task] = _T
            _work_logits = _work_logits / _T
        _bias, _calib_f1 = _fit_bias(_work_logits, _dev_true[_task], _n_cls,
                                      n_iters=getattr(cfg, 'CALIB_SEARCH_ITERS', 300))
        CALIB_BIAS[_task] = _bias
        print(f'  {_task:8s}: dev wF1 {_raw_f1:.4f} -> {_calib_f1:.4f} after calibration  '
              f'(T={CALIB_TEMP[_task].item():.3f}, bias={_bias.numpy().round(3)})')
else:
    print('Logit calibration disabled (cfg.USE_LOGIT_CALIBRATION=False) -- CALIB_BIAS/CALIB_TEMP left at identity.')

def _tta_forward(model, ids, mask, pix, qt, qi):
    """Standard test-time augmentation: average softmax probabilities from"""
    out1 = model(ids, mask, pix, qt, qi)
    if not getattr(cfg, 'USE_TTA', False):
        return out1
    pix_flip = torch.flip(pix, dims=[-1])
    out2 = model(ids, mask, pix_flip, qt, qi)
    out = dict(out1)
    for k in ['logits_event', 'logits_info', 'logits_human', 'logits_damage', 'logits_verif']:
        p1 = F.softmax(out1[k], dim=-1)
        p2 = F.softmax(out2[k], dim=-1)
        avg_p = (p1 + p2) / 2.0
        out[k] = torch.log(avg_p.clamp_min(1e-8))
    return out

def apply_calibration(out):
    """Applies DEV-fit temperature scaling then per-class bias to
    humanitarian/damage/verif logits. No-op when calibration was disabled."""
    out = dict(out)
    for _task, _key in [('human', 'logits_human'), ('damage', 'logits_damage'), ('verif', 'logits_verif')]:
        dev = out[_key].device
        out[_key] = out[_key] / CALIB_TEMP[_task].to(dev) + CALIB_BIAS[_task].to(dev)
    return out

print('✅ Calibration + TTA helpers ready (_tta_forward, apply_calibration, CALIB_BIAS).')

free_memory()

## 15. Training and Validation Curves

Plots training/validation loss and F1 curves saved in `HISTORY` across all epochs.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

eps = range(1, len(HISTORY['train_loss']) + 1)

ax1.plot(eps, HISTORY['train_loss'], 'o-', color='#e74c3c', label='Train Loss', linewidth=2)
ax1.plot(eps, HISTORY['val_loss'],   's--',color='#3498db', label='Val Loss',   linewidth=2)

best_ep = HISTORY['val_loss'].index(min(HISTORY['val_loss'])) + 1
ax1.axvline(best_ep, color='gray', linestyle=':', alpha=0.6, label=f'Best epoch ({best_ep})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('📉 Multi-Task Loss Curve', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(eps, HISTORY['train_f1'], 'o-', color='#2ecc71', label='Train wF1', linewidth=2)
ax2.plot(eps, HISTORY['val_f1'],   's--',color='#9b59b6', label='Val wF1',   linewidth=2)

best_ep2 = HISTORY['val_f1'].index(max(HISTORY['val_f1'])) + 1
ax2.axvline(best_ep2, color='gray', linestyle=':', alpha=0.6, label=f'Best epoch ({best_ep2})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Weighted F1')
ax2.set_title('📈 Multi-Task Avg Weighted-F1', fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle('CrisisMMD Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()

plt.savefig(str(cfg.PLOTS_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved.')

free_memory()

## 16. Main Test Results

Reloads the best checkpoint (highest validation F1) and computes final test-set metrics for every task.

> **Status: Existing results from prior executed run; not regenerated in this environment.**

In [ ]:
MODEL.load_state_dict(torch.load(cfg.CKPT_PATH, map_location=DEVICE))
MODEL.eval()
print(f'✅ Best checkpoint loaded from: {cfg.CKPT_PATH}')

all_preds = {k: [] for k in ['event','info','human','damage','verif']}
all_trues = {k: [] for k in ['event','info','human','damage','verif']}
all_probs = {k: [] for k in ['event','info','human','damage','verif']}
all_conf  = []
all_rt    = []
all_rv    = []

with torch.inference_mode():
    for batch in dl_test:
        ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
        qt   = batch['q_text'].to(DEVICE, non_blocking=True)
        qi   = batch['q_image'].to(DEVICE, non_blocking=True)

        out  = _tta_forward(MODEL, ids, mask, pix, qt, qi)
        out  = apply_calibration(out)

        for head in ['event','info','human','damage','verif']:
            logit_key = f'logits_{head}'
            if logit_key in out:
                all_preds[head].extend(out[logit_key].argmax(dim=-1).cpu().numpy())

        for head in ['event','info','human','damage','verif']:
            logit_key = f'logits_{head}'
            if logit_key in out:
                all_probs[head].append(F.softmax(out[logit_key], dim=-1).cpu().numpy())

        for head in ['event','info','human','damage','verif']:
            label_key = f'label_{head}'
            if label_key in batch:
                all_trues[head].extend(batch[label_key].numpy())

        probs = []
        for head in ['event', 'info', 'human', 'damage']:
            if f'logits_{head}' in out:
                probs.append(F.softmax(out[f'logits_{head}'], dim=-1).max(dim=-1)[0].cpu().numpy())

        conf = np.mean(probs, axis=0)
        all_conf.extend(conf.tolist())

        all_rt.extend(out['u_t'].squeeze(-1).cpu().numpy().tolist())
        all_rv.extend(out['u_v'].squeeze(-1).cpu().numpy().tolist())

for _k in list(all_probs.keys()):
    all_probs[_k] = np.concatenate(all_probs[_k], axis=0) if all_probs[_k] else None

HEAD_META = [
    ('event',  'Event Type',      LABEL_ENCODERS.get('event_type') if 'LABEL_ENCODERS' in globals() else None),
    ('info',   'Informative',     LABEL_ENCODERS.get('informative') if 'LABEL_ENCODERS' in globals() else None),
    ('human',  'Humanitarian',    LABEL_ENCODERS.get('humanitarian') if 'LABEL_ENCODERS' in globals() else None),
    ('damage', 'Damage Severity', LABEL_ENCODERS.get('damage') if 'LABEL_ENCODERS' in globals() else None),
]

TASK_REPORTS = {}
for key, title, le in HEAD_META:
    if not all_trues.get(key):
        continue

    try:
        labels = list(le.classes_) if le else None
    except AttributeError:
        labels = None

    report = classification_report(
        all_trues[key], all_preds[key],
        target_names=labels, zero_division=0
    )
    wf1 = f1_score(all_trues[key], all_preds[key], average='weighted', zero_division=0)
    TASK_REPORTS[key] = {'report': report, 'wf1': wf1, 'title': title, 'le': le}

    print('\n' + '=' * 60)
    print(f'  📋 {title}  —  Weighted F1: {wf1:.4f}')
    print('=' * 60)
    print(report)

print(f'\n📊 Mean Confidence Score  : {np.mean(all_conf):.4f}  ± {np.std(all_conf):.4f}')
print(f'📊 Mean Text Reliability  : {np.mean(all_rt):.4f}  ± {np.std(all_rt):.4f}')
print(f'📊 Mean Image Reliability : {np.mean(all_rv):.4f}  ± {np.std(all_rv):.4f}')

free_memory()

### 16.1 Comprehensive Evaluation Metrics
Reports Accuracy, Precision/Recall/F1 (macro & weighted), Cohen's Kappa, and MCC per task — plus ROC-AUC and PR-AUC for the binary Verification head.


> **Status: Existing results from prior executed run; not regenerated in this environment.**

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize
import numpy as np
import pandas as pd

def compute_full_metrics(y_true, y_pred, task_name, y_prob=None):
    """Returns the standard classification metric set for one task.
    If y_prob (full softmax probability matrix, shape [N, n_classes]) is
    given, also computes macro ROC-AUC / PR-AUC (one-vs-rest for
    multi-class tasks, standard binary form for 2-class tasks) so no task
    is left with NaN in the summary table.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    row = {
        'Task'            : task_name,
        'N'               : len(y_true),
        'Accuracy'        : accuracy_score(y_true, y_pred),
        'Precision(macro)': precision_score(y_true, y_pred, average='macro',    zero_division=0),
        'Precision(wtd)'  : precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall(macro)'   : recall_score(y_true, y_pred, average='macro',    zero_division=0),
        'Recall(wtd)'     : recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1(macro)'       : f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'F1(weighted)'    : f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Cohen_Kappa'     : cohen_kappa_score(y_true, y_pred),
        'MCC'             : matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else float('nan'),
    }

    row['ROC_AUC'] = float('nan')
    row['PR_AUC']  = float('nan')
    if y_prob is not None and len(np.unique(y_true)) > 1:
        n_classes = y_prob.shape[1]
        try:
            if n_classes == 2:
                row['ROC_AUC'] = roc_auc_score(y_true, y_prob[:, 1])
                row['PR_AUC']  = average_precision_score(y_true, y_prob[:, 1])
            else:

                present = np.unique(y_true)
                row['ROC_AUC'] = roc_auc_score(
                    y_true, y_prob, average='macro', multi_class='ovr', labels=present
                )
                y_true_bin = label_binarize(y_true, classes=present)
                y_prob_present = y_prob[:, present]
                row['PR_AUC'] = average_precision_score(y_true_bin, y_prob_present, average='macro')
        except ValueError:

            pass

    return row

_metric_rows = []
for key, title, le in HEAD_META:
    if not all_trues.get(key):
        continue
    _metric_rows.append(compute_full_metrics(all_trues[key], all_preds[key], title, y_prob=all_probs.get(key)))

if 'verif' in all_preds and all_preds['verif'] and all_trues.get('verif'):
    _verif_title = 'Verification (real, cross-modal)' if globals().get('VERIF_IS_REAL', False) else 'Verification (PROXY)'
    verif_row = compute_full_metrics(
        all_trues['verif'], all_preds['verif'], _verif_title, y_prob=all_probs.get('verif')
    )
    _metric_rows.append(verif_row)

df_metrics = pd.DataFrame(_metric_rows).set_index('Task')
print('=' * 100)
print('  📊 COMPREHENSIVE EVALUATION METRICS — Held-Out Test Set')
print('=' * 100)
print(df_metrics.round(4).to_string())

df_metrics.round(4).to_csv(str(cfg.METRICS_DIR / 'evaluation_metrics_full.csv'))
print("\n✅ Saved full metrics table to evaluation_metrics_full.csv")

fig, ax = plt.subplots(figsize=(12, 6))
plot_cols = ['Accuracy', 'Precision(wtd)', 'Recall(wtd)', 'F1(weighted)']
x = np.arange(len(df_metrics.index))
width = 0.2
colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']
for i, col in enumerate(plot_cols):
    ax.bar(x + (i - 1.5) * width, df_metrics[col], width, label=col, color=colors[i])
ax.set_xticks(x)
ax.set_xticklabels(df_metrics.index, rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('📊 Comprehensive Evaluation Metrics by Task', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'evaluation_metrics_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
_verif_is_proxy = not globals().get('VERIF_IS_REAL', False)
if _verif_is_proxy:
    print("⚠️  No independent image-side informativeness column was found in this dataset "
          "build — the 'Verification' row below falls back to the (informative == "
          "'not_informative') PROXY label. Treat it as an approximate sanity check, "
          "not a genuine task metric.")
else:
    print("✅ Using the REAL cross-modal Verification label: agreement between the "
          "independent text-side and image-side informativeness annotations "
          "(built in Section 2 as 'verif_idx'). This is a genuine dataset-native "
          "signal, not algebraically derived from a single column.")

### 16.1.1 Correction Note: Evaluation Bugs Addressed
**Bug 1 — Ablation table looked wrong.** The old loop compared "full model + calibration + TTA + gating" against "ablated model, no extras." Fixed by splitting into **Table A** (structural-only, all extras off) and **Table B** (post-hoc extras' own contribution).

**Bug 2 — Verification metrics looked broken.** `verif_true` was a proxy label (`label_info != 0`), not a real verification label, so it didn't match what the head was trained on. Fixed: Verification row now only uses real ground-truth labels if available, otherwise flagged as a proxy estimate.


In [ ]:
import pandas as pd

def render_metrics_table(df_metrics, proxy_flag=True):
    df = df_metrics.copy()
    pct_cols = [c for c in df.columns if c not in ('N',)]
    df_display = df.copy()
    for c in pct_cols:
        df_display[c] = df_display[c].map(lambda v: f"{v:.4f}" if pd.notnull(v) else "—")

    def flag(row_name):
        tags = []
        if proxy_flag and row_name == 'Verification' and _verif_is_proxy:
            tags.append('proxy label')
        try:
            kappa = df.loc[row_name, 'Cohen_Kappa']
            if pd.notnull(kappa) and kappa < 0:
                tags.append('worse than chance — investigate')
        except Exception:
            pass
        return ', '.join(tags)

    df_display['⚠ Notes'] = [flag(r) for r in df_display.index]

    print('=' * 100)
    print('  📊 COMPREHENSIVE EVALUATION METRICS — Held-Out Test Set')
    print('=' * 100)
    print(df_display.to_string())
    print('=' * 100)

    flagged = df_display[df_display['⚠ Notes'] != '']
    if not flagged.empty:
        print("\nRows needing attention before reporting these numbers as final:")
        for name, note in flagged['⚠ Notes'].items():
            print(f"  • {name}: {note}")

    try:
        from IPython.display import display
        styled = df_display.style.set_caption("Comprehensive Evaluation Metrics — Held-Out Test Set") \
            .set_properties(**{'text-align': 'center'}) \
            .applymap(lambda v: 'background-color:#fdecea' if v not in ('', '—') and 'proxy' in str(v) or 'chance' in str(v) else '',
                      subset=['⚠ Notes'])
        display(styled)
    except Exception:
        pass

    return df_display

df_metrics_display = render_metrics_table(df_metrics, proxy_flag=True)

## 17. Confusion Matrices

Per-task confusion matrices rendered as heatmaps for Event Type, Informativeness, Humanitarian Need, and Damage Severity.

In [ ]:
CMAP = LinearSegmentedColormap.from_list('crisis', ['#ffffff', '#1a237e'])

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flat

for ax, (key, title, le) in zip(axes, HEAD_META):
    if not all_trues.get(key):
        ax.axis('off')
        continue
    labels = list(le.classes_) if le else None
    cm     = confusion_matrix(all_trues[key], all_preds[key])
    cm_n   = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1e-8)
    wf1    = TASK_REPORTS.get(key, {}).get('wf1', 0)

    sns.heatmap(
        cm_n, annot=cm, fmt='d',
        xticklabels=labels, yticklabels=labels,
        cmap=CMAP, vmin=0, vmax=1,
        linewidths=0.5, linecolor='white',
        ax=ax, cbar_kws={'label': 'Recall', 'shrink': 0.8},
    )
    ax.set_title(f'{title}  (wF1={wf1:.3f})', fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True',      fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('🔲 Confusion Matrices — CrisisMMD Test Set', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved.')

free_memory()

## 18. Uncertainty Analysis

Compares E-REM's predicted uncertainty ($u_t$, $u_v$) between correctly- and incorrectly-classified test samples — a well-calibrated model should show higher uncertainty on its mistakes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rt   = np.asarray(all_rt, dtype=np.float32)
rv   = np.asarray(all_rv, dtype=np.float32)
conf = np.asarray(all_conf, dtype=np.float32)

mask = np.isfinite(rt) & np.isfinite(rv) & np.isfinite(conf)
rt, rv, conf = rt[mask], rv[mask], conf[mask]

rt = np.clip(rt, 0, 1)
rv = np.clip(rv, 0, 1)
conf = np.clip(conf, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(26, 6))

bins = np.linspace(0, 1, 30)

axes[0].hist(
    rt,
    bins=bins,
    alpha=0.6,
    color="#3498db",
    density=True,
    edgecolor="black",
    label="Text $R_t$"
)

axes[0].hist(
    rv,
    bins=bins,
    alpha=0.6,
    color="#e74c3c",
    density=True,
    edgecolor="black",
    label="Image $R_v$"
)

axes[0].axvline(rt.mean(), color="#2980b9",
                linestyle="--",
                linewidth=2,
                label=f"Mean R_t = {rt.mean():.3f}")

axes[0].axvline(rv.mean(), color="#c0392b",
                linestyle="--",
                linewidth=2,
                label=f"Mean R_v = {rv.mean():.3f}")

axes[0].set_xlim(0,1)
axes[0].set_xlabel("Reliability")
axes[0].set_ylabel("Density")
axes[0].set_title("Reliability Distribution")
axes[0].grid(alpha=0.3)
axes[0].legend()

scatter = axes[1].scatter(
    rt,
    rv,
    c=conf,
    cmap="viridis",
    s=18,
    alpha=0.7,
    edgecolors="none"
)

axes[1].plot([0,1],[0,1],'k--',alpha=0.3)

axes[1].set_xlim(0,1)
axes[1].set_ylim(0,1)

axes[1].set_xlabel("Text Reliability $R_t$")
axes[1].set_ylabel("Image Reliability $R_v$")
axes[1].set_title("Reliability Correlation")
axes[1].grid(alpha=0.3)

cbar = fig.colorbar(scatter, ax=axes[1])
cbar.set_label("Confidence")

if "human" in all_trues and len(all_trues["human"]) > 0:

    labels = np.asarray(all_trues["human"])[mask]

    le = LABEL_ENCODERS.get("humanitarian")

    if le is not None:
        class_names = le.classes_
    else:
        class_names = np.unique(labels)

    means = []
    stds = []

    for c in range(len(class_names)):
        idx = labels == c

        if idx.sum() > 0:
            means.append(conf[idx].mean())
            stds.append(conf[idx].std())
        else:
            means.append(0)
            stds.append(0)

    colors = plt.cm.Set2(np.linspace(0,1,len(class_names)))

    bars = axes[2].bar(
        class_names,
        means,
        yerr=stds,
        capsize=5,
        color=colors,
        edgecolor="black"
    )

    for b, m in zip(bars, means):
        axes[2].text(
            b.get_x()+b.get_width()/2,
            b.get_height()+0.02,
            f"{m:.3f}",
            ha="center",
            fontsize=9
        )

    axes[2].set_ylim(0,1.05)
    axes[2].set_ylabel("Mean Confidence")
    axes[2].set_xlabel("Humanitarian Class")
    axes[2].set_title("Confidence by Class")
    axes[2].grid(axis="y", alpha=0.3)

else:
    axes[2].axis("off")

if 'damage' in all_trues and len(all_trues['damage']) > 0:
    correct = np.asarray(all_preds['damage'])[mask] == np.asarray(all_trues['damage'])[mask]
    if correct.sum() > 0 and (~correct).sum() > 0:
        rt_c, rt_ic = rt[correct].mean(), rt[~correct].mean()
        rv_c, rv_ic = rv[correct].mean(), rv[~correct].mean()

        bar_w = 0.35
        x = np.arange(2)
        axes[3].bar(x - bar_w/2, [rt_c, rt_ic], width=bar_w, label='R_t (Text)', color='#3498db')
        axes[3].bar(x + bar_w/2, [rv_c, rv_ic], width=bar_w, label='R_v (Image)', color='#e74c3c')
        axes[3].set_xticks(x)
        axes[3].set_xticklabels(['Correct', 'Incorrect'])
        axes[3].set_ylabel('Mean Reliability Score')
        axes[3].set_title('Reliability vs Prediction Correctness')
        axes[3].legend()
        axes[3].grid(axis='y', alpha=0.3)
else:
    axes[3].axis("off")

plt.suptitle(
    f"Reliability Analysis\n"
    f"Mean R_t={rt.mean():.3f} | "
    f"Mean R_v={rv.mean():.3f} | "
    f"Mean Confidence={conf.mean():.3f}",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(rect=[0,0,1,0.94])

plt.savefig(
    cfg.OUT_DIR / "reliability_analysis.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Mean Text Reliability :", rt.mean())
print("Mean Image Reliability:", rv.mean())
print("Mean Confidence       :", conf.mean())
print("Saved to:", cfg.OUT_DIR / "reliability_analysis.png")

free_memory()

### 18.1 Mean Reliability Score (95% CI)
Raw `R = 1 - u` isn't calibrated by default. This section fits one temperature per modality on DEV only, then reports the calibrated Mean Reliability Score on test with a bootstrap 95% CI.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

@torch.inference_mode()
def collect_u_and_correctness(model, loader, task='human'):
    """Collect per-modality uncertainty (u_t,u_v) and task correctness for one split."""
    base = model.module if isinstance(model, nn.DataParallel) else model
    prev_mode = getattr(base, 'ablation_mode', 'full')
    base.ablation_mode = 'full'
    model.eval()
    u_t_all, u_v_all, correct_all = [], [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE)
        qt   = batch['q_text'].to(DEVICE)
        qi   = batch['q_image'].to(DEVICE)
        out  = model(ids, mask, pix, qt, qi)
        label_key = f'label_{task}'
        if label_key not in batch or f'logits_{task}' not in out:
            base.ablation_mode = prev_mode
            return None
        preds = out[f'logits_{task}'].argmax(dim=-1).cpu().numpy()
        trues = batch[label_key].numpy()
        u_t_all.append(out['u_t'].squeeze(-1).cpu().numpy())
        u_v_all.append(out['u_v'].squeeze(-1).cpu().numpy())
        correct_all.append((preds == trues).astype(np.float32))
    base.ablation_mode = prev_mode
    return (np.concatenate(u_t_all), np.concatenate(u_v_all), np.concatenate(correct_all))

def fit_temperature(u, correct, n_steps=200, lr=0.05):
    """Fit a single scalar temperature T minimizing NLL of R=sigmoid((1-2u)/T)
    against binary correctness, on a held-out (dev) split. Returns T (float)."""
    u_t = torch.tensor(u, dtype=torch.float32)
    y_t = torch.tensor(correct, dtype=torch.float32)
    logit_raw = (1.0 - 2.0 * u_t)
    T = torch.ones(1, requires_grad=True)
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=n_steps)
    def closure():
        opt.zero_grad()
        p = torch.sigmoid(logit_raw / T.clamp(min=1e-2))
        loss = F.binary_cross_entropy(p, y_t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(T.detach().clamp(min=1e-2))

def bootstrap_mean_ci(x, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    n = len(x)
    boots = np.array([x[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [(1-ci)/2*100, (1+ci)/2*100])
    return x.mean(), lo, hi

_RELIAB_TASK = 'human' if 'human' in all_trues and len(all_trues.get('human', [])) > 0 else 'damage'

dev_stats = collect_u_and_correctness(MODEL, dl_dev, task=_RELIAB_TASK) if 'dl_dev' in globals() else None

if dev_stats is not None:
    u_t_dev, u_v_dev, correct_dev = dev_stats
    T_t = fit_temperature(u_t_dev, correct_dev)
    T_v = fit_temperature(u_v_dev, correct_dev)
else:
    print('⚠️  dl_dev not available -- falling back to T=1.0 (no calibration correction applied).')
    T_t, T_v = 1.0, 1.0

rt_arr = np.clip(np.asarray(all_rt, dtype=np.float32), 0, 1)
rv_arr = np.clip(np.asarray(all_rv, dtype=np.float32), 0, 1)
u_t_test = 1.0 - rt_arr
u_v_test = 1.0 - rv_arr

R_t_cal = 1.0 / (1.0 + np.exp(-(1.0 - 2.0*u_t_test) / T_t))
R_v_cal = 1.0 / (1.0 + np.exp(-(1.0 - 2.0*u_v_test) / T_v))
R_mean_cal = (R_t_cal + R_v_cal) / 2.0

raw_mean = ((rt_arr + rv_arr) / 2.0)
m_raw, lo_raw, hi_raw   = bootstrap_mean_ci(raw_mean)
m_cal, lo_cal, hi_cal   = bootstrap_mean_ci(R_mean_cal)

print('── Mean Reliability Score (test set) ──')
print(f'Fitted temperatures on DEV -> T_t={T_t:.3f}  T_v={T_v:.3f}  (T≈1.0 means no correction was needed)')
print(f'Raw (uncalibrated)   : {m_raw:.4f}   95% CI [{lo_raw:.4f}, {hi_raw:.4f}]')
print(f'Temperature-calibrated: {m_cal:.4f}   95% CI [{lo_cal:.4f}, {hi_cal:.4f}]')
print()
print('Interpretation: the calibrated figure is only meaningfully higher than the raw one if the CIs')
print('barely overlap or don\'t overlap at all -- if they overlap substantially, report both and say so.')

MEAN_RELIABILITY = {
    'task_used_for_fit': _RELIAB_TASK,
    'T_text': T_t, 'T_image': T_v,
    'raw_mean': float(m_raw), 'raw_ci': (float(lo_raw), float(hi_raw)),
    'calibrated_mean': float(m_cal), 'calibrated_ci': (float(lo_cal), float(hi_cal)),
}

## 19. Explainability with Grad-CAM

Generates Grad-CAM heatmaps over the CLIP vision tower for any of the six output heads, highlighting which image regions drove each prediction.

> **Status: Existing results from prior executed run; not regenerated in this environment.**

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F

class CLIPGradCAM:
    """
    Robust Grad-CAM for CLIP Vision Transformer.
    Targets resblocks[-2] so spatial patch tokens receive non-zero attention gradients.
    """

    def __init__(self, model):
        self.model = model.module if isinstance(model, torch.nn.DataParallel) else model
        self.gradients = None
        self.activations = None
        target_layer = None

        if hasattr(self.model.vis_encoder, "encoder"):
            enc = self.model.vis_encoder.encoder
            if hasattr(enc, "transformer") and hasattr(enc.transformer, "resblocks"):

                target_layer = enc.transformer.resblocks[-2]
            elif hasattr(enc, "layers"):
                target_layer = enc.layers[-2]

        if target_layer is None:
            raise RuntimeError("Could not locate CLIP transformer block for Grad-CAM.")

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activations = out

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def generate(
        self,
        pixel_values,
        q_text,
        q_image,
        input_ids,
        attention_mask,
        target_head="logits_human",
        method="gradcam"
    ):

        with torch.set_grad_enabled(True):
            self.model.eval()
            self.model.zero_grad(set_to_none=True)

            device = next(self.model.parameters()).device

            def move(x, dtype=None):
                if not torch.is_tensor(x):
                    x = torch.tensor(x)
                x = x.to(device)
                if dtype is not None:
                    x = x.to(dtype)
                return x

            pixel_values = move(pixel_values, torch.float32)
            pixel_values.requires_grad_(True)

            q_text = move(q_text, torch.float32)
            q_image = move(q_image, torch.float32)
            input_ids = move(input_ids, torch.long)
            attention_mask = move(attention_mask, torch.long)

            outputs = self.model(
                input_ids,
                attention_mask,
                pixel_values,
                q_text,
                q_image
            )

            logits = outputs[target_head]
            pred_idx = logits.argmax(dim=-1).item()
            score = logits[0, pred_idx]

            score.backward()

            if self.activations is None or self.gradients is None:
                return np.zeros((224, 224), dtype=np.float32), pred_idx

            acts = self.activations.detach()
            grads = self.gradients.detach()

            if acts.shape[1] == pixel_values.shape[0]:

                acts = acts[1:, 0, :]
                grads = grads[1:, 0, :]
            else:

                acts = acts[0, 1:, :]
                grads = grads[0, 1:, :]

            acts = acts.cpu().numpy()
            grads = grads.cpu().numpy()

            if method == "gradcam++":

                grad2 = grads ** 2
                grad3 = grads ** 3
                sum_act_grad3 = (acts * grad3).sum(axis=0, keepdims=True)
                alpha_denom = 2.0 * grad2 + sum_act_grad3
                alpha_denom = np.where(np.abs(alpha_denom) > 1e-8, alpha_denom, 1e-8)
                alpha = grad2 / alpha_denom
                weights = (alpha * np.maximum(grads, 0)).sum(axis=0)
                cam = acts @ weights
            elif method == "eigencam":

                acts_centered = acts - acts.mean(axis=0, keepdims=True)
                try:
                    _, _, Vt = np.linalg.svd(acts_centered, full_matrices=False)
                    cam = acts_centered @ Vt[0]
                except np.linalg.LinAlgError:
                    cam = acts.mean(axis=1)
            else:
                weights = grads.mean(axis=0)
                cam = acts @ weights

            cam = np.maximum(cam, 0)

            num_tokens = cam.shape[0]
            grid = int(np.sqrt(num_tokens))

            if grid * grid != num_tokens:
                return np.zeros((224, 224), dtype=np.float32), pred_idx

            cam = cam.reshape(grid, grid)

            if cam.max() - cam.min() > 1e-8:
                cam = (cam - cam.min()) / (cam.max() - cam.min())
            else:
                cam = np.zeros_like(cam)

            cam = cv2.resize(
                cam,
                (pixel_values.shape[-1], pixel_values.shape[-2]),
                interpolation=cv2.INTER_CUBIC
            )

            return cam, pred_idx

def overlay_cam(img, cam):
    cam_uint8 = np.uint8(cam * 255)
    heatmap = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    img_np = np.array(img, dtype=np.uint8)

    overlay = cv2.addWeighted(img_np, 0.55, heatmap, 0.45, 0)
    return overlay

free_memory()

### 19.1 Grad-CAM Case Studies

Runs Grad-CAM on a handful of correctly- and incorrectly-classified humanitarian-task test samples, so the visual explanations can be compared side-by-side between hits and misses.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader

gcam = CLIPGradCAM(MODEL)
device = next(MODEL.parameters()).device

le_hum = LABEL_ENCODERS.get("humanitarian")
hum_names = list(le_hum.classes_) if le_hum else ["class_0", "class_1", "class_2"]

_valid_mask = df_test["image_path"].notna().values
valid_df = df_test.loc[_valid_mask].copy().reset_index(drop=True)

valid_df["pred_human"] = np.asarray(all_preds["human"])[_valid_mask]
valid_df["true_human"] = np.asarray(all_trues["human"])[_valid_mask]

correct_df = valid_df[valid_df.pred_human == valid_df.true_human]
incorrect_df = valid_df[valid_df.pred_human != valid_df.true_human]

correct_sample = correct_df.sample(min(4, len(correct_df)), random_state=42) if len(correct_df) else correct_df
incorrect_sample = incorrect_df.sample(min(4, len(incorrect_df)), random_state=42) if len(incorrect_df) else incorrect_df

sample_df = pd.concat([correct_sample, incorrect_sample]).reset_index(drop=True)
num_samples = len(sample_df)

if num_samples == 0:
    print("No samples found.")
else:
    dataset = CrisisMMDDataset(
        sample_df,
        TOKENIZER,
        VAL_TRANSFORMS,
        is_train=False,
    )

    loader = DataLoader(
        dataset,
        batch_size=num_samples,
        shuffle=False,
    )

    batch = next(iter(loader))

    batch = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    fig, axes = plt.subplots(
        3,
        num_samples,
        figsize=(4 * num_samples, 12)
    )

    if num_samples == 1:
        axes = np.expand_dims(axes, 1)

    fig.suptitle("Grad-CAM Visual Explanations (Danger Red Heatmaps)", fontsize=16, fontweight="bold")

    for i in range(num_samples):
        img_path = sample_df.iloc[i]["image_path"]

        try:
            orig = Image.open(img_path).convert("RGB").resize((224, 224))
        except Exception:
            orig = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

        single_batch = {k: v[i:i+1] for k, v in batch.items()}

        try:
            cam, pred_idx = gcam.generate(
                pixel_values=single_batch["pixel_values"],
                q_text=single_batch["q_text"],
                q_image=single_batch["q_image"],
                input_ids=single_batch["input_ids"],
                attention_mask=single_batch["attention_mask"],
            )
        except Exception as e:
            print(f"Sample {i} failed: {e}")
            cam = np.zeros((224, 224), dtype=np.float32)
            pred_idx = int(sample_df.iloc[i]["pred_human"])

        axes[0, i].imshow(orig)
        true_idx = int(sample_df.iloc[i]["true_human"])
        axes[0, i].set_title(f"True\n{hum_names[true_idx]}", fontsize=10)
        axes[0, i].axis("off")

        heatmap = np.uint8(cam * 255)
        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        axes[1, i].imshow(heatmap)
        axes[1, i].set_title("Grad-CAM Heatmap", fontsize=10)
        axes[1, i].axis("off")

        overlay = overlay_cam(orig, cam)
        axes[2, i].imshow(overlay)

        color = "forestgreen" if sample_df.iloc[i]["pred_human"] == sample_df.iloc[i]["true_human"] else "crimson"
        axes[2, i].set_title(f"Pred\n{hum_names[pred_idx]}", fontsize=10, color=color, fontweight="bold")
        axes[2, i].axis("off")

    plt.tight_layout()

    out_path = cfg.OUT_DIR / "gradcam_explanations.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved vibrant heatmap plot to {out_path} ✅")

free_memory()

### 19.2 Grad-CAM Faithfulness Metrics
Quantifies whether Grad-CAM heatmaps actually drive predictions (Humanitarian head, subsampled test set):
- **Average Drop %** — confidence drop using only the CAM region (lower = better)
- **Average Increase %** — how often confidence rises on the CAM-weighted image (higher = better)
- **Deletion AUC** — confidence collapse as salient pixels are removed (lower = better)
- **Insertion AUC** — confidence recovery as salient pixels are revealed (higher = better)


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

N_EVAL_SAMPLES = getattr(cfg, 'GRADCAM_N_EVAL_SAMPLES', 100)
N_STEPS        = 8
GCAM_TARGET_HEAD = 'logits_human'
GCAM_METHODS = getattr(cfg, 'GRADCAM_METHODS', ['gradcam'])

if 'gcam' not in dir():
    gcam = CLIPGradCAM(MODEL)

_eval_pool = valid_df if 'valid_df' in dir() else df_test[df_test['image_path'].apply(lambda p: Path(p).exists())]
eval_df = _eval_pool.sample(min(N_EVAL_SAMPLES, len(_eval_pool)), random_state=13).reset_index(drop=True)

_eval_dataset = CrisisMMDDataset(eval_df, TOKENIZER, VAL_TRANSFORMS, is_train=False)
_eval_loader  = DataLoader(_eval_dataset, batch_size=len(eval_df), shuffle=False)
_eval_batch   = next(iter(_eval_loader))
_eval_batch   = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in _eval_batch.items()}

@torch.inference_mode()
def _target_prob(pixel_values, q_text, q_image, input_ids, attention_mask, target_idx):
    out = MODEL(input_ids, attention_mask, pixel_values, q_text, q_image)
    return F.softmax(out[GCAM_TARGET_HEAD], dim=-1)[0, target_idx].item()

def _perturb(pixel_values, cam, fraction, mode, baseline):
    """mode='delete': replace the top-`fraction` most-salient pixels with `baseline`.
    mode='insert': start from `baseline`, reveal the top-`fraction` most-salient pixels."""
    _, _, H, W = pixel_values.shape
    n_total = H * W
    n_pix = int(round(fraction * n_total))
    flat_order = np.argsort(-cam.flatten())
    mask = np.zeros(n_total, dtype=bool)
    mask[flat_order[:n_pix]] = True
    mask_t = torch.tensor(mask.reshape(H, W), device=pixel_values.device).unsqueeze(0).unsqueeze(0)
    if mode == 'delete':
        return torch.where(mask_t, baseline, pixel_values)
    else:
        return torch.where(mask_t, pixel_values, baseline)

drop_pct, increase_flags, deletion_aucs, insertion_aucs = [], [], [], []
deletion_curves, insertion_curves = [], []
fractions = np.linspace(0.0, 1.0, N_STEPS + 1)

print(f'Evaluating Grad-CAM faithfulness on {len(eval_df)} test samples '
      f'({N_STEPS} perturbation steps each) …')

for i in tqdm(range(len(eval_df)), desc='Grad-CAM faithfulness'):
    single = {k: v[i:i+1] for k, v in _eval_batch.items()}
    pix = single['pixel_values']

    try:

        cam, pred_idx = gcam.generate(
            pixel_values=pix, q_text=single['q_text'], q_image=single['q_image'],
            input_ids=single['input_ids'], attention_mask=single['attention_mask'],
            target_head=GCAM_TARGET_HEAD, method=GCAM_METHODS[0],
        )
    except Exception as e:
        print(f'  Sample {i} Grad-CAM generation failed ({e}) -- skipped.')
        continue

    cam_t = torch.tensor(cam, device=pix.device, dtype=pix.dtype)
    baseline = torch.zeros_like(pix)

    p_orig = _target_prob(pix, single['q_text'], single['q_image'],
                           single['input_ids'], single['attention_mask'], pred_idx)

    soft_mask = cam_t.unsqueeze(0).unsqueeze(0)
    pix_weighted = pix * soft_mask + baseline * (1 - soft_mask)
    p_weighted = _target_prob(pix_weighted, single['q_text'], single['q_image'],
                               single['input_ids'], single['attention_mask'], pred_idx)
    drop_pct.append(max(0.0, p_orig - p_weighted) / max(p_orig, 1e-8) * 100.0)
    increase_flags.append(p_weighted > p_orig)

    del_probs = []
    for f in fractions:
        pix_f = _perturb(pix, cam, f, mode='delete', baseline=baseline)
        del_probs.append(_target_prob(pix_f, single['q_text'], single['q_image'],
                                       single['input_ids'], single['attention_mask'], pred_idx))
    deletion_aucs.append(float(np.trapz(del_probs, fractions)))
    deletion_curves.append(del_probs)

    ins_probs = []
    for f in fractions:
        pix_f = _perturb(pix, cam, f, mode='insert', baseline=baseline)
        ins_probs.append(_target_prob(pix_f, single['q_text'], single['q_image'],
                                       single['input_ids'], single['attention_mask'], pred_idx))
    insertion_aucs.append(float(np.trapz(ins_probs, fractions)))
    insertion_curves.append(ins_probs)

gradcam_metrics = {
    'Average Drop %'          : float(np.mean(drop_pct)) if drop_pct else float('nan'),
    'Average Increase %'      : float(np.mean(increase_flags)) * 100.0 if increase_flags else float('nan'),
    'Deletion AUC (lower=better)'  : float(np.mean(deletion_aucs)) if deletion_aucs else float('nan'),
    'Insertion AUC (higher=better)': float(np.mean(insertion_aucs)) if insertion_aucs else float('nan'),
    'N samples evaluated'     : len(deletion_aucs),
}

print('\n' + '=' * 60)
print(f'  📊 Grad-CAM Faithfulness Metrics — Humanitarian head (N={gradcam_metrics["N samples evaluated"]})')
print('=' * 60)
for k, v in gradcam_metrics.items():
    if k == 'N samples evaluated':
        print(f'  {k:32s}: {v}')
    else:
        print(f'  {k:32s}: {v:.4f}')

pd.DataFrame([gradcam_metrics]).to_csv(str(cfg.METRICS_DIR / 'gradcam_faithfulness_metrics.csv'), index=False)
print("\n✅ Saved Grad-CAM faithfulness metrics to gradcam_faithfulness_metrics.csv")

fig, ax = plt.subplots(figsize=(7, 5))
if deletion_curves and insertion_curves:
    del_arr = np.array(deletion_curves)
    ins_arr = np.array(insertion_curves)
    del_mean, del_std = del_arr.mean(axis=0), del_arr.std(axis=0)
    ins_mean, ins_std = ins_arr.mean(axis=0), ins_arr.std(axis=0)

    ax.plot(fractions, del_mean, 'o-', color='#c0392b', label=f'Deletion (AUC={np.mean(deletion_aucs):.3f}, lower=better)')
    ax.fill_between(fractions, del_mean - del_std, del_mean + del_std, color='#c0392b', alpha=0.15)

    ax.plot(fractions, ins_mean, 's-', color='#27ae60', label=f'Insertion (AUC={np.mean(insertion_aucs):.3f}, higher=better)')
    ax.fill_between(fractions, ins_mean - ins_std, ins_mean + ins_std, color='#27ae60', alpha=0.15)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Fraction of pixels perturbed (most salient first)')
ax.set_ylabel('Target-class probability')
ax.set_title('Grad-CAM Deletion / Insertion Faithfulness', fontsize=12, fontweight='bold')
ax.legend(loc='center right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'gradcam_faithfulness_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

if len(GCAM_METHODS) > 1:
    method_drop = {}
    for _method in GCAM_METHODS:
        _drops = []
        for i in range(len(eval_df)):
            single = {k: v[i:i+1] for k, v in _eval_batch.items()}
            pix = single['pixel_values']
            try:
                cam_m, pred_idx_m = gcam.generate(
                    pixel_values=pix, q_text=single['q_text'], q_image=single['q_image'],
                    input_ids=single['input_ids'], attention_mask=single['attention_mask'],
                    target_head=GCAM_TARGET_HEAD, method=_method,
                )
            except Exception:
                continue
            cam_t_m = torch.tensor(cam_m, device=pix.device, dtype=pix.dtype)
            baseline_m = torch.zeros_like(pix)
            p_orig_m = _target_prob(pix, single['q_text'], single['q_image'],
                                     single['input_ids'], single['attention_mask'], pred_idx_m)
            soft_mask_m = cam_t_m.unsqueeze(0).unsqueeze(0)
            pix_w_m = pix * soft_mask_m + baseline_m * (1 - soft_mask_m)
            p_w_m = _target_prob(pix_w_m, single['q_text'], single['q_image'],
                                  single['input_ids'], single['attention_mask'], pred_idx_m)
            _drops.append(max(0.0, p_orig_m - p_w_m) / max(p_orig_m, 1e-8) * 100.0)
        method_drop[_method] = float(np.mean(_drops)) if _drops else float('nan')

    print('\n' + '=' * 60)
    print(f'  Explainability method comparison (Average Drop %, lower=better), N={len(eval_df)}')
    print('=' * 60)
    for m, d in method_drop.items():
        print(f'  {m:12s}: {d:.2f}%')
    pd.DataFrame([method_drop]).to_csv(str(cfg.METRICS_DIR / 'gradcam_method_comparison.csv'), index=False)

free_memory()

## 20. Representation Visualization (t-SNE)

Projects the shared disaster representation $H_{disaster}$ (and/or $H'_t$, $H'_v$) into 2-D via t-SNE, colored by task label, to qualitatively inspect cluster separability.

> **Status: Existing results from prior executed run; not regenerated in this environment.**

In [ ]:
from sklearn.manifold import TSNE

print('Collecting H_disaster embeddings …')
embeddings, labels_dm = [], []
MODEL.eval()
with torch.inference_mode():
    for batch in dl_test:
        ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
        qt   = batch['q_text'].to(DEVICE, non_blocking=True)
        qi   = batch['q_image'].to(DEVICE, non_blocking=True)
        out  = MODEL(ids, mask, pix, qt, qi)
        embeddings.append(out['h_disaster'].cpu().numpy())
        labels_dm.extend(batch['label_damage'].numpy())

X     = np.vstack(embeddings)
Y     = np.array(labels_dm)
le_dm = LABEL_ENCODERS.get('damage')
dm_names = list(le_dm.classes_) if le_dm else [str(i) for i in range(cfg.N_DAMAGE)]

print(f'Running t-SNE on {X.shape[0]} embeddings (dim={X.shape[1]}) …')
tsne   = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000)
X_2d   = tsne.fit_transform(X)

COLS = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

Y_event = np.array(all_trues['event'])
le_event = LABEL_ENCODERS.get('event_type')
event_names = list(le_event.classes_) if le_event else [str(i) for i in range(cfg.N_EVENT)]

for c, name in enumerate(event_names):
    mask_ = Y_event == c
    axes[0].scatter(X_2d[mask_, 0], X_2d[mask_, 1],
               c=[COLS[c % len(COLS)]], label=name, alpha=0.65, s=18, edgecolors='none')
axes[0].set_title('🔵 t-SNE — colored by Event Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('t-SNE dim 1'); axes[0].set_ylabel('t-SNE dim 2')
axes[0].legend(title='Event Type', fontsize=10)
axes[0].grid(alpha=0.2)

for c, name in enumerate(dm_names):
    mask_ = Y == c
    axes[1].scatter(X_2d[mask_, 0], X_2d[mask_, 1],
               c=[COLS[c % len(COLS)]], label=name, alpha=0.65, s=18, edgecolors='none')
axes[1].set_title('🔵 t-SNE — colored by Damage Severity', fontsize=13, fontweight='bold')
axes[1].set_xlabel('t-SNE dim 1'); axes[1].set_ylabel('t-SNE dim 2')
axes[1].legend(title='Damage Severity', fontsize=10)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'tsne_embeddings_multitask.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ t-SNE visualization saved.')

free_memory()

## 21. Ablation Study
Disables each component at inference time and reports per-task weighted-F1.

**Caveat:** this is a forward-pass bypass ablation on one trained model, a valid lower bound but not equivalent to retraining from scratch. Section 18.2 implements the genuinely-retrained version (opt-in via `RUN_GENUINE_ABLATION`).

Class-balanced loss and manifold mixup are training-time only (not toggle-able post-hoc); logit calibration and TTA are toggle-able and shown as their own ablation rows.


> **Status: mixed — see per-row labels below.** Structural rows are **inference-time module bypass** on the single existing checkpoint (seed 42), from the prior executed run; not regenerated here. No row in this section was retrained.

### 21.1 Ablation Fairness Audit

**Confirmed by inspecting `CrisisMultiModal.forward` (`no_bcmf` branch):** the existing
`w/o UG-CMA` ablation row replaces UG-CMA's output with a naive `h_t_prime + h_v_prime` sum:

```python
if mode == 'no_bcmf':
    m_fuse = h_t_prime + h_v_prime
else:
    m_fuse, p_t, p_v = self.bcmf(h_t_prime, h_v_prime, u_t, u_v)
```

Per the task brief, this is **not sufficient evidence for UG-CMA's contribution on its own** —
a large drop when replacing a fusion module with unweighted addition is expected regardless of
whether that specific module (cross-attention *with* uncertainty gating) is doing anything
beyond "some form of learned cross-modal mixing." The scientifically fair comparison is:

> E-REM + UASG + **conventional (uncertainty-blind) cross-attention** vs. E-REM + UASG + **UG-CMA**

with everything else identical. This isolates what the *uncertainty-guided* part of UG-CMA adds,
not just "having any fusion module at all."

**Status:** a conventional cross-attention fusion module does not exist in the current notebook,
so this comparison could not be run in this session (also no GPU/dataset access). A minimal,
architecture-consistent scaffold is added below as a `NOT EXECUTED` cell — a standard
multi-head cross-attention block with the same input/output shape as `self.bcmf` (UG-CMA), but
without uncertainty-based gating/weighting. This is *not* a new architectural block in the main
model (it never replaces UG-CMA in the deployed model) — it is only an extra `ablation_mode`
branch for a controlled comparison, per your instruction not to add new architecture to the
main pipeline.

The existing `w/o UG-CMA` (naive sum) row is kept in Table A below, now clearly re-labeled as
a **lower-bound sanity check**, not the primary evidence for UG-CMA's value. The
paired-bootstrap significance test in Section 18.2 already correctly shows a large, significant
wF1 drop for `w/o UG-CMA` (p<0.001 on both humanitarian and damage) — that result is genuine,
just needs the conventional-cross-attention comparison to be a complete claim.


In [ ]:
# NOT EXECUTED IN THIS SESSION -- scaffold only, added for the conventional-cross-attention
# ablation comparison requested for journal rigor. No GPU/torch/dataset in the audit
# environment, so this has not been run or validated end-to-end. Review before use.
#
# This does NOT modify CrisisMultiModal or the deployed model -- it defines a standalone
# module with the same interface as self.bcmf (UG-CMA) so it can be swapped in for exactly
# one extra ablation_mode branch ('conv_cross_attn'), then removed/kept as an experiment.

class ConventionalCrossAttention(nn.Module):
    """Standard bidirectional multi-head cross-attention fusion, WITHOUT uncertainty gating.
    Same I/O contract as the UG-CMA module (self.bcmf): takes (h_t_prime, h_v_prime, u_t, u_v)
    and returns (m_fuse, p_t, p_v), but ignores u_t/u_v entirely -- isolating the effect of
    uncertainty-guided attention weighting specifically, not fusion-vs-no-fusion in general.
    """
    def __init__(self, dim, n_heads=8, dropout=0.1):
        super().__init__()
        self.t2v_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.v2t_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm_t = nn.LayerNorm(dim)
        self.norm_v = nn.LayerNorm(dim)
        self.out_proj = nn.Linear(dim * 2, dim)

    def forward(self, h_t_prime, h_v_prime, u_t=None, u_v=None):
        t = h_t_prime.unsqueeze(1)  # (B,1,D) -- pooled features treated as length-1 sequence
        v = h_v_prime.unsqueeze(1)
        t_attn, _ = self.t2v_attn(t, v, v)
        v_attn, _ = self.v2t_attn(v, t, t)
        t_fused = self.norm_t(h_t_prime + t_attn.squeeze(1))
        v_fused = self.norm_v(h_v_prime + v_attn.squeeze(1))
        m_fuse = self.out_proj(torch.cat([t_fused, v_fused], dim=-1))
        # p_t/p_v (per-modality gate weights) don't exist in an ungated module; return
        # uniform 0.5/0.5 so downstream code expecting these values doesn't break.
        p_t = torch.full((h_t_prime.size(0), 1), 0.5, device=h_t_prime.device)
        p_v = torch.full((h_v_prime.size(0), 1), 0.5, device=h_v_prime.device)
        return m_fuse, p_t, p_v

# To use: (1) instantiate `MODEL.module.conv_cross_attn = ConventionalCrossAttention(cfg.FUSE_DIM).to(DEVICE)`
# on the trained model (inference-time swap, matching how other ablation_mode branches work),
# (2) add an `elif mode == \'conv_cross_attn\':` branch in `forward` that calls it instead of
# `self.bcmf`, (3) re-run `eval_ablation(..., mode=\'conv_cross_attn\')` and the paired-bootstrap
# significance test against the full model. NOTE: a freshly-initialized attention block has
# random weights, so for a *fair* comparison it should be fine-tuned briefly (not just
# inference-swapped) -- unlike the other ablation_mode branches, which remove/replace a
# component with a fixed, non-parametric operation. This is why it is a scaffold, not a
# drop-in inference ablation.


### 21.2 Correction Note: Hierarchical Gating
**Bug**: gating overwrote predictions with the majority class on any predicted `not_informative` sample, no confidence check — dropping humanitarian wF1 from 0.7064 to 0.4684.

**Fix**: gating is now confidence-thresholded and off by default. Old and thresholded variants are both kept as labeled rows in Table B.

**Note**: E-REM/UASG rows in Table A only reflect their real contribution if the loaded checkpoint was trained after the Section 6/7/9 supervision-loss fix.


### 21.3 Ablation Methodology and Multi-Seed Status

**What kind of ablation is this?** Inspecting the code: `eval_ablation()` sets
`model.ablation_mode` and does a **single forward pass per configuration on the already-trained
full model** (module bypass / replacement at inference time, e.g. `no_bcmf`, `no_erem`,
`text_only`). This is an **inference-time structural ablation**, not a retrained-from-scratch
ablation. The word "genuine" in the original print statement refers to "real forward passes on
real held-out data" (as opposed to hand-typed numbers), not "retrained" — the code confirms no
`.fit()`/training loop runs here. This is labeled explicitly here per the task brief's
instruction not to call inference-time removal a retrained ablation.

`cfg.RUN_GENUINE_ABLATION = True` and `cfg.ABLATION_RETRAIN_EPOCHS = 15` exist in the config,
suggesting a retrained-ablation path may have been intended, but no retraining loop for
ablation variants is present in the executed cells or their outputs. If retrained ablations are
needed for the journal submission, that code does not currently exist and must be written and
run separately (out of scope for this session — no GPU/dataset access).

**Multi-seed check:** `cfg.ABLATION_SEEDS = [42, 43, 44]` is configured, but a text search of
this entire notebook's source and saved outputs found **no occurrence of seed 43 or seed 44**
anywhere. Only a single run (seed 42, implicit in `train_test_split(..., random_state=42)` and
default global seeding) was actually executed. **All results in this notebook are single-seed.**
Any mean ± std across seeds would be fabricated and is not reported. If multi-seed robustness
is required for the journal (strongly recommended for a fair comparison table), it needs to be
executed in the original GPU environment — 3 full training runs × (full model + key ablations)
at ~15–45 epochs each, which was out of reach for this static-review session.


In [ ]:
import numpy as np

MAJORITY_IDX = {
    'event': int(df_train['event_type_idx'].mode()[0])   if 'event_type_idx'   in df_train.columns else 0,
    'human': int(df_train['humanitarian_idx'].mode()[0]) if 'humanitarian_idx' in df_train.columns else 0,
    'damage': int(df_train['damage_idx'].mode()[0])      if 'damage_idx'       in df_train.columns else 0,
}
_le_info_ab = LABEL_ENCODERS.get('informative')
NOT_INFO_IDX = list(_le_info_ab.classes_).index('not_informative') if _le_info_ab is not None and 'not_informative' in _le_info_ab.classes_ else None

GATE_CONF_THRESH_DEFAULT = None

def eval_ablation(model, loader, mode='full', use_calib=False, use_tta=False, use_gating=False, gate_conf_thresh=GATE_CONF_THRESH_DEFAULT):
    """Evaluate the model with one component disabled/bypassed at inference.

    gate_conf_thresh: if use_gating=True, only overwrite a task prediction
    with the majority-class fallback when P(not_informative) from the info
    head exceeds this threshold. If None, gating is effectively disabled
    (regardless of use_gating) -- this is the recommended/default behavior,
    since the ablation shows unconditional gating hurts performance.
    """
    base = model.module if isinstance(model, nn.DataParallel) else model
    prev_mode = getattr(base, 'ablation_mode', 'full')
    base.ablation_mode = mode
    model.eval()

    tasks = ['event', 'info', 'human', 'damage']
    all_p = {t: [] for t in tasks}
    all_t = {t: [] for t in tasks}

    with torch.inference_mode():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
            mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
            pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
            qt   = batch['q_text'].to(DEVICE, non_blocking=True)
            qi   = batch['q_image'].to(DEVICE, non_blocking=True)

            out1 = model(ids, mask, pix, qt, qi)
            if use_tta:
                out2 = model(ids, mask, torch.flip(pix, dims=[-1]), qt, qi)
                logits = {}
                for k in ['logits_event', 'logits_info', 'logits_human', 'logits_damage']:
                    p1 = F.softmax(out1[k], dim=-1)
                    p2 = F.softmax(out2[k], dim=-1)
                    logits[k] = torch.log(((p1 + p2) / 2.0).clamp_min(1e-8))
            else:
                logits = {k: out1[k] for k in ['logits_event', 'logits_info', 'logits_human', 'logits_damage']}

            if use_calib:
                logits['logits_human']  = logits['logits_human']  + CALIB_BIAS['human'].to(DEVICE)
                logits['logits_damage'] = logits['logits_damage'] + CALIB_BIAS['damage'].to(DEVICE)

            info_probs = F.softmax(logits['logits_info'], dim=-1).cpu().numpy()
            preds = {task: logits[f'logits_{task}'].argmax(dim=-1).cpu().numpy() for task in tasks}

            if use_gating and NOT_INFO_IDX is not None and gate_conf_thresh is not None:
                not_info_conf = info_probs[:, NOT_INFO_IDX]
                gated_mask = (preds['info'] == NOT_INFO_IDX) & (not_info_conf >= gate_conf_thresh)
                for task in ('event', 'human', 'damage'):
                    preds[task] = np.where(gated_mask, MAJORITY_IDX[task], preds[task])

            for task in tasks:
                all_p[task].extend(preds[task])
                all_t[task].extend(batch[f'label_{task}'].numpy())

    base.ablation_mode = prev_mode
    per_task = {t: f1_score(all_t[t], all_p[t], average='weighted', zero_division=0) for t in tasks}
    per_task['avg'] = sum(per_task.values()) / len(per_task)
    return per_task

print('Running ablation study (per-task, genuine forward passes on the held-out test set) …')

_df_test_raw = df_test.copy()
if 'text_raw' in _df_test_raw.columns:
    _df_test_raw['text'] = _df_test_raw['text_raw']
ds_test_raw = CrisisMMDDataset(_df_test_raw, TOKENIZER, VAL_TRANSFORMS, is_train=False)
dl_test_raw = DataLoader(ds_test_raw, batch_size=cfg.BATCH_SIZE, shuffle=False)

STRUCTURAL_ABLATION_MODES = {
    'Full Model (Ours)'        : ('full',       False, False, False, dl_test),
    'w/o E-REM'                : ('no_erem',    False, False, False, dl_test),
    'w/o UASG'                 : ('no_rasg',    False, False, False, dl_test),
    'w/o UG-CMA'                : ('no_bcmf',    False, False, False, dl_test),
    'Text Only'                : ('text_only',  False, False, False, dl_test),
    'Image Only'               : ('image_only', False, False, False, dl_test),
}

POST_HOC_ABLATION_MODES = {
    'Full Model (Ours, deployed config)'          : ('full', True,  True,  False, dl_test, None),
    'Full − no calibration'                       : ('full', False, True,  False, dl_test, None),
    'Full − no TTA'                                : ('full', True,  False, False, dl_test, None),
    'Full − no calib/TTA'                          : ('full', False, False, False, dl_test, None),
    'Full − no text cleaning'                      : ('full', True,  True,  False, dl_test_raw, None),
    'Full + gating (unconditional, NOT recommended)': ('full', True,  True,  True,  dl_test, 0.0),
    'Full + gating (confidence >= 0.95)'           : ('full', True,  True,  True,  dl_test, 0.95),
}

print("── (A) Structural ablation (main paper table): every row uses the SAME")
print("    calib=False / TTA=False / gating=False setting, so only the")
print("    architectural component differs between rows. ──")
ablation_results   = {}
ablation_per_task  = {}
for name, (mode, use_calib, use_tta, use_gating, loader) in STRUCTURAL_ABLATION_MODES.items():
    per_task = eval_ablation(MODEL, loader, mode=mode, use_calib=use_calib, use_tta=use_tta, use_gating=use_gating)
    ablation_per_task[name] = per_task
    ablation_results[name]  = per_task['avg']
    print(f"  {name:22s} → avg wF1={per_task['avg']:.4f}  |  "
          f"event={per_task['event']:.4f}  info={per_task['info']:.4f}  "
          f"human={per_task['human']:.4f}  damage={per_task['damage']:.4f}")

_best_name = max(ablation_results, key=ablation_results.get)
print(f"\n🏆 Best structural configuration by avg wF1: '{_best_name}'"
      + ("  ✅ (matches 'Full Model (Ours)' as intended)" if _best_name == 'Full Model (Ours)' else
         "  ⚠️  'Full Model (Ours)' was NOT the best under IDENTICAL eval settings -- if E-REM/UASG margins "
         "stay within noise after confirming the checkpoint includes the E-REM retrain (see markdown note "
         "above Section 18), report this honestly as a limited/negative structural contribution rather than "
         "forcing the table to agree with the model name."))

print("\n── (B) Post-hoc technique ablation (secondary table): isolates the")
print("    contribution of calibration / TTA / text-cleaning / gating on top")
print("    of the structurally-complete full model. Not used for the")
print("    structural best-config check above. ──")
posthoc_per_task = {}
for name, (mode, use_calib, use_tta, use_gating, loader, gate_thresh) in POST_HOC_ABLATION_MODES.items():
    per_task = eval_ablation(MODEL, loader, mode=mode, use_calib=use_calib, use_tta=use_tta, use_gating=use_gating, gate_conf_thresh=gate_thresh)
    posthoc_per_task[name] = per_task
    print(f"  {name:48s} → avg wF1={per_task['avg']:.4f}  |  "
          f"event={per_task['event']:.4f}  info={per_task['info']:.4f}  "
          f"human={per_task['human']:.4f}  damage={per_task['damage']:.4f}")

for name, pt in ablation_per_task.items():
    pt['group'] = 'structural'
for name, pt in posthoc_per_task.items():
    pt['group'] = 'post_hoc'
ablation_per_task.update(posthoc_per_task)

import pandas as pd
df_ablation = pd.DataFrame([
    {'Configuration': name, 'Group': pt.get('group', 'structural'), 'Weighted_F1': pt['avg'],
     'Event_F1': pt['event'], 'Info_F1': pt['info'],
     'Humanitarian_F1': pt['human'], 'Damage_F1': pt['damage']}
    for name, pt in ablation_per_task.items()
])
df_ablation.to_csv(str(cfg.METRICS_DIR / 'ablation_results.csv'), index=False)
print('\n✅ Saved per-task ablation results to ablation_results.csv')

import numpy as np
fig, ax = plt.subplots(figsize=(14, 6))
configs = list(STRUCTURAL_ABLATION_MODES.keys())
task_plot = ['human', 'damage', 'avg']
task_labels = {'human': 'Humanitarian', 'damage': 'Damage Severity', 'avg': '4-Task Avg'}
colors = {'human': '#e67e22', 'damage': '#c0392b', 'avg': '#7f8c8d'}
x = np.arange(len(configs))
width = 0.25
for i, t in enumerate(task_plot):
    vals = [ablation_per_task[c][t] for c in configs]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=task_labels[t], color=colors[t], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=7, rotation=90)
ax.axhline(0.85, color='#2ecc71', linestyle='--', linewidth=1.5, label='Target 0.85')
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Weighted F1')
ax.set_ylim(0, 1.12)
ax.set_title('🔬 Ablation Study — Humanitarian & Damage Shown Explicitly (not averaged away)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'ablation_study.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Ablation study chart saved (per-task, real numbers).')

free_memory()

### 21.4 Diagnostic: E-REM Uncertainty Spread
Measures the real spread of `u_t`/`u_v` on the test set after training, to check whether the flat E-REM ablation row is explained by a collapsed uncertainty signal (fixable) or is a genuine negative result (report honestly either way).

In [ ]:
import numpy as np

@torch.inference_mode()
def erem_uncertainty_stats(model, loader, max_batches=None):
    base = model.module if isinstance(model, nn.DataParallel) else model
    prev_mode = getattr(base, 'ablation_mode', 'full')
    base.ablation_mode = 'full'
    model.eval()
    u_t_all, u_v_all = [], []
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE)
        qt   = batch['q_text'].to(DEVICE)
        qi   = batch['q_image'].to(DEVICE)
        out  = model(ids, mask, pix, qt, qi)
        u_t_all.append(out['u_t'].cpu().numpy().ravel())
        u_v_all.append(out['u_v'].cpu().numpy().ravel())
    base.ablation_mode = prev_mode
    return np.concatenate(u_t_all), np.concatenate(u_v_all)

u_t_vals, u_v_vals = erem_uncertainty_stats(MODEL, dl_test)

print("── E-REM uncertainty spread on the held-out test set ──")
print(f"u_t : mean={u_t_vals.mean():.4f}  std={u_t_vals.std():.4f}  min={u_t_vals.min():.4f}  max={u_t_vals.max():.4f}")
print(f"u_v : mean={u_v_vals.mean():.4f}  std={u_v_vals.std():.4f}  min={u_v_vals.min():.4f}  max={u_v_vals.max():.4f}")

if u_t_vals.std() < 0.02 and u_v_vals.std() < 0.02:
    print("\n⚠️  Both u_t and u_v have very low variance across the test set -- E-REM's output is "
          "close to a constant, which would explain a flat ablation regardless of how UASG/UG-CMA "
          "consume it. This is a genuine finding to report, not something to route around.")
else:
    print("\n✅ u_t/u_v vary meaningfully across samples -- if the E-REM ablation row is still flat, "
          "the cause is more likely elsewhere (e.g. downstream heads not relying much on the gated "
          "representation's fine-grained scale), which is worth a separate look.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(u_t_vals, bins=40, color='#3498db', edgecolor='white')
axes[0].set_title('u_t distribution (text uncertainty)')
axes[1].hist(u_v_vals, bins=40, color='#e67e22', edgecolor='white')
axes[1].set_title('u_v distribution (image uncertainty)')
plt.tight_layout()
plt.savefig(str(cfg.PLOTS_DIR / 'erem_uncertainty_spread.png'), dpi=150, bbox_inches='tight')
plt.show()

## 22. Per-Class and Per-Task Performance

Consolidated bar-chart / table summary of weighted-F1 per task, for the full model vs. each ablation.

> **Status: mixed — see per-row labels below.** Structural rows are **inference-time module bypass** on the single existing checkpoint (seed 42), from the prior executed run; not regenerated here. No row in this section was retrained.

In [ ]:
for key, title, le in HEAD_META:
    if not all_trues.get(key):
        continue
    labels  = list(le.classes_) if le else None
    report  = classification_report(
        all_trues[key], all_preds[key],
        target_names=labels, zero_division=0
    )
    wf1 = f1_score(all_trues[key], all_preds[key], average='weighted', zero_division=0)
    TASK_REPORTS[key] = {'report': report, 'wf1': wf1, 'title': title, 'le': le}

task_names = []
task_wf1   = []
for key, meta in TASK_REPORTS.items():
    task_names.append(meta['title'])
    task_wf1.append(meta['wf1'])

if not task_names:
    print('⚠️ No task reports available. Skipping dashboard.')
else:
    TASK_COLORS = ['#3498db','#2ecc71','#e67e22','#9b59b6','#e74c3c']

    fig = plt.figure(figsize=(16, 9))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    ax_main = fig.add_subplot(gs[0, :])
    bars = ax_main.bar(task_names, task_wf1,
                       color=TASK_COLORS[:len(task_names)],
                       edgecolor='white', linewidth=1.5, width=0.55)
    ax_main.axhline(0.90, color='#e74c3c', linestyle='--', linewidth=1.5, label='Target 0.90')
    for bar, val in zip(bars, task_wf1):
        ax_main.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax_main.set_ylabel('Weighted F1',  fontsize=12)
    ax_main.set_title('🎯 Multi-Task Performance Summary (Test Set)', fontsize=14, fontweight='bold')
    ax_main.set_ylim(0, 1.12)
    ax_main.legend(fontsize=10)
    ax_main.grid(alpha=0.3, axis='y')

    ax_conf = fig.add_subplot(gs[1, 0])
    vals    = [np.mean(all_conf), 1 - np.mean(all_conf)]
    ax_conf.pie(vals, labels=[f'Conf\n{np.mean(all_conf):.2f}', f'Unc\n{1-np.mean(all_conf):.2f}'],colors=['#2ecc71','#ecf0f1'], startangle=90,wedgeprops={'edgecolor':'white','linewidth':2})
    ax_conf.set_title('🎲 Mean\nConfidence', fontsize=11, fontweight='bold')

    ax_rel = fig.add_subplot(gs[1, 1])
    ax_rel.bar(['R_t (Text)', 'R_v (Image)'],
               [np.mean(all_rt), np.mean(all_rv)],
               color=['#3498db','#e74c3c'],
               yerr=[np.std(all_rt), np.std(all_rv)], capsize=8,
               edgecolor='white')
    ax_rel.set_ylim(0, 1)
    ax_rel.set_title('📡 Mean Reliability\nScores', fontsize=11, fontweight='bold')
    ax_rel.set_ylabel('Score')
    ax_rel.grid(alpha=0.3, axis='y')

    ax_abl = fig.add_subplot(gs[1, 2])
    full_f1  = ablation_results.get('Full Model (Ours)', 0)
    deltas   = {k: full_f1 - v for k, v in ablation_results.items() if k != 'Full Model (Ours)'}
    cols_abl = ['#e74c3c' if d > 0 else '#2ecc71' for d in deltas.values()]
    ax_abl.barh(list(deltas.keys()), list(deltas.values()), color=cols_abl, edgecolor='white')
    ax_abl.axvline(0, color='black', linewidth=1)
    ax_abl.set_xlabel('F1 Drop (Full − Ablated)')
    ax_abl.set_title('🔬 Ablation\nImpact (ΔwF1)', fontsize=11, fontweight='bold')
    ax_abl.tick_params(axis='y', labelsize=8)
    ax_abl.grid(alpha=0.3, axis='x')

    plt.suptitle('📊 CrisisMMD — Complete Performance Dashboard', fontsize=15, fontweight='bold', y=1.01)
    plt.savefig(str(cfg.PLOTS_DIR / 'performance_dashboard.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Performance dashboard saved.')

free_memory()

## 23. Statistical Significance
Runs a paired bootstrap over the test set to get a p-value/CI for each component's ΔF1 on the humanitarian and damage heads, so real effects can be distinguished from noise.


> **Status: Existing result from prior executed run (paired bootstrap over the existing seed-42 test-set predictions); not regenerated in this environment.**

In [ ]:
import numpy as np

def paired_bootstrap_delta_f1(preds_full, preds_ablated, trues, n_boot=2000, seed=0):
    """Paired bootstrap over sample indices: resample the SAME indices for both
    configurations so within-sample correlation is preserved, then compute the
    weighted-F1 gap on each resample to get a distribution over Delta F1."""
    from sklearn.metrics import f1_score
    rng = np.random.default_rng(seed)
    preds_full = np.asarray(preds_full); preds_ablated = np.asarray(preds_ablated); trues = np.asarray(trues)
    n = len(trues)
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        f1_full = f1_score(trues[idx], preds_full[idx], average='weighted', zero_division=0)
        f1_abl  = f1_score(trues[idx], preds_ablated[idx], average='weighted', zero_division=0)
        deltas[b] = f1_full - f1_abl
    lo, hi = np.percentile(deltas, [2.5, 97.5])

    p_value = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())
    p_value = min(p_value, 1.0)
    return deltas.mean(), lo, hi, p_value

@torch.inference_mode()
def collect_ablation_preds(model, loader, mode='full'):
    """Runs a single structural-ablation forward pass and returns real
    per-sample predictions + ground truths for every task.

    eval_ablation() in Section 18 intentionally only returns aggregated
    weighted-F1 scores (that's all the ablation dashboard needs), so it can't
    be reused here -- the paired bootstrap needs the raw per-sample arrays.
    This is a separate, self-contained collector for that purpose.
    """
    base = model.module if isinstance(model, nn.DataParallel) else model
    prev_mode = getattr(base, 'ablation_mode', 'full')
    base.ablation_mode = mode
    model.eval()

    tasks = ['event', 'info', 'human', 'damage']
    all_p = {t: [] for t in tasks}
    all_t = {t: [] for t in tasks}

    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        pix  = batch['pixel_values'].to(DEVICE, non_blocking=True)
        qt   = batch['q_text'].to(DEVICE, non_blocking=True)
        qi   = batch['q_image'].to(DEVICE, non_blocking=True)

        out = model(ids, mask, pix, qt, qi)
        for t in tasks:
            all_p[t].extend(out[f'logits_{t}'].argmax(dim=-1).cpu().numpy())
            all_t[t].extend(batch[f'label_{t}'].numpy())

    base.ablation_mode = prev_mode
    return {t: (np.array(all_p[t]), np.array(all_t[t])) for t in tasks}

_ablation_components = {
    'w/o E-REM'  : 'no_erem',
    'w/o UASG'   : 'no_rasg',
    'w/o UG-CMA' : 'no_bcmf',
}

print('Collecting per-sample predictions for the full model and each structural ablation (dl_test is not shuffled, so sample order lines up across runs) …')
_full_preds    = collect_ablation_preds(MODEL, dl_test, mode='full')
_ablated_preds = {label: collect_ablation_preds(MODEL, dl_test, mode=mode)
                  for label, mode in _ablation_components.items()}

_sig_rows = []
for task in ['human', 'damage']:
    full_preds, full_trues = _full_preds[task]
    for label in _ablation_components:
        abl_preds, abl_trues = _ablated_preds[label][task]

        if not np.array_equal(full_trues, abl_trues):
            print(f'⚠️  Ground-truth order mismatch for {label}/{task} -- skipping '
                  f'(check that dl_test is built with shuffle=False).')
            continue

        mean_d, lo, hi, p = paired_bootstrap_delta_f1(full_preds, abl_preds, full_trues)
        _sig_rows.append({
            'task': task,
            'component': label,
            'delta_f1_mean': round(float(mean_d), 4),
            'ci_95_lo': round(float(lo), 4),
            'ci_95_hi': round(float(hi), 4),
            'p_value': round(float(p), 4),
            'significant_at_0.05': bool(p < 0.05),
        })

if _sig_rows:
    df_sig = pd.DataFrame(_sig_rows)
    print(df_sig.to_string(index=False))
else:
    print('No significance rows computed -- see warnings above.')

## 24. Single-Sample Inference
Runs the trained model on one (text, image) pair; applies hierarchical gating, E-REM-calibrated confidence, and Grad-CAM heatmaps.

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import torch.nn.functional as F

@torch.inference_mode()
def predict_single(image_path: str, tweet_text: str, model=MODEL) -> dict:
    """Run end-to-end prediction for one (image, text) pair."""

    tweet_text = clean_text(tweet_text) if 'clean_text' in dir() else tweet_text
    enc = TOKENIZER(tweet_text, padding='max_length', truncation=True,
                    max_length=cfg.MAX_LEN, return_tensors='pt')
    ids  = enc['input_ids'].to(DEVICE, non_blocking=True)
    mask = enc['attention_mask'].to(DEVICE, non_blocking=True)

    try:
        img = Image.open(image_path).convert('RGB')
    except Exception:
        img = Image.fromarray(np.zeros((224,224,3), dtype=np.uint8))
    pix = VAL_TRANSFORMS(img).unsqueeze(0).to(DEVICE, non_blocking=True)

    cc, wc, ent = extract_text_quality(tweet_text)
    bs, br, co, ns = extract_image_quality(cv2.imread(image_path))
    qt = torch.tensor([[cc, wc, ent]], dtype=torch.float32).to(DEVICE, non_blocking=True)
    qi = torch.tensor([[bs, br, co, ns]], dtype=torch.float32).to(DEVICE, non_blocking=True)

    model.eval()
    out = _tta_forward(model, ids, mask, pix, qt, qi)
    out = apply_calibration(out)

    def decode(le, idx):
        return le.classes_[idx] if le and idx < len(le.classes_) else str(idx)

    def calibrate_confidence(uncertainty, u_t, u_v, beta=1.0, w_head=0.5, w_mod=0.5):
        sigma2   = uncertainty.squeeze().clamp_min(0.0)
        modal_u  = 0.5 * (u_t.squeeze() + u_v.squeeze())
        fused    = w_head * sigma2 + w_mod * modal_u
        conf     = torch.exp(-beta * fused)
        return float(conf.clamp(0.0, 1.0).cpu().item())

    conf = calibrate_confidence(out['uncertainty'], out['u_t'], out['u_v'])

    def top3(logits, le):
        probs = F.softmax(logits, dim=-1)[0]
        topk = torch.topk(probs, min(3, len(probs)))
        return [f"{decode(le, idx.item())} ({prob.item()*100:.1f}%)" for prob, idx in zip(topk.values, topk.indices)]

    event_out = top3(out['logits_event'], LABEL_ENCODERS.get('event_type'))
    info_out  = top3(out['logits_info'], LABEL_ENCODERS.get('informative'))
    human_out = top3(out['logits_human'], LABEL_ENCODERS.get('humanitarian'))
    damage_out = top3(out['logits_damage'], LABEL_ENCODERS.get('damage'))
    verif_out  = 'Authenticated' if out['logits_verif'].argmax(-1).item() == 0 else 'Potential Fabrication'

    le_info = LABEL_ENCODERS.get('informative')
    info_pred_idx = out['logits_info'].argmax(-1).item()
    info_pred_label = decode(le_info, info_pred_idx).lower()

    is_not_informative = 'not_informative' in info_pred_label or info_pred_label == 'not informative'

    if is_not_informative:
        event_out  = ['N/A']
        human_out  = ['N/A']
        damage_out = ['N/A']
        verif_out  = 'N/A'

    return {
        'event'        : event_out,
        'informative'  : info_out,
        'humanitarian' : human_out,
        'damage'       : damage_out,
        'verification' : verif_out,
        'confidence'   : round(min(max(conf, 0), 1), 4),

        'r_t'          : round(float((1.0 - out['u_t']).clamp(0, 1).item()), 4),
        'r_v'          : round(float((1.0 - out['u_v']).clamp(0, 1).item()), 4),
        'u_t'          : round(float(out['u_t'].item()), 4),
        'u_v'          : round(float(out['u_v'].item()), 4),
        'gated'        : is_not_informative,
    }

DEMO_PER_CATEGORY = 5

valid_rows = df_test[df_test['image_path'].apply(lambda p: Path(p).exists())].copy()

EVENT_DEMO_GROUPS = [
    ('wildfire',  'fire'),
    ('earthquake','earthquake'),
    ('hurricane', 'hurricane'),
    ('flood',     'flood'),
]

demo_parts = []
for display_name, event_value in EVENT_DEMO_GROUPS:
    if 'event_type' not in valid_rows.columns:
        break
    group = valid_rows[valid_rows['event_type'] == event_value]
    n_take = min(DEMO_PER_CATEGORY, len(group))
    if n_take == 0:
        print(f'⚠️  No valid (image-on-disk) test samples found for category "{display_name}" ({event_value}) -- skipping.')
        continue
    if n_take < DEMO_PER_CATEGORY:
        print(f'⚠️  Only {n_take}/{DEMO_PER_CATEGORY} valid samples available for "{display_name}" -- showing all of them.')
    picked = group.sample(n=n_take, random_state=7).copy()
    picked['demo_category'] = display_name
    demo_parts.append(picked)

if 'informative' in valid_rows.columns:
    not_info_group = valid_rows[valid_rows['informative'] == 'not_informative']
    n_take = min(DEMO_PER_CATEGORY, len(not_info_group))
    if n_take == 0:
        print('⚠️  No valid (image-on-disk) not_informative test samples found -- skipping that group.')
    else:
        if n_take < DEMO_PER_CATEGORY:
            print(f'⚠️  Only {n_take}/{DEMO_PER_CATEGORY} valid not_informative samples available -- showing all of them.')
        picked_ni = not_info_group.sample(n=n_take, random_state=7).copy()
        picked_ni['demo_category'] = 'not_informative'
        demo_parts.append(picked_ni)

if demo_parts:
    demo_rows = pd.concat(demo_parts, axis=0)
else:

    demo_rows = valid_rows.head(DEMO_PER_CATEGORY * 5).copy()
    demo_rows['demo_category'] = 'unspecified'

N_DEMO = len(demo_rows)
print(f'\n📋 Section 19 demo set: {N_DEMO} samples '
      f'({demo_rows["demo_category"].value_counts().to_dict()})')

import matplotlib.pyplot as plt
import cv2
import gc

if 'gcam' not in dir():
    gcam = CLIPGradCAM(MODEL)

for i, (idx, demo_row) in enumerate(demo_rows.iterrows()):
    tweet_text = demo_row['text']
    image_path = demo_row['image_path']
    result = predict_single(image_path, tweet_text)

    print('\n' + '=' * 55)
    print(f'  🚨  CrisisMMD Single-Sample Inference Output — Sample {i+1}/{len(demo_rows)} '
          f'[category: {demo_row.get("demo_category", "unspecified")}]')
    print('=' * 55)
    print(f'  Tweet            : {str(demo_row["text"])[:80]}…')
    print(f'  Image            : {demo_row["image_path"]}')
    print('─' * 55)
    print(f'  🌍  Event Type   : {", ".join(result["event"])}')
    print(f'  📢  Informative  : {", ".join(result["informative"])}')
    print(f'  🆘  Humanitarian : {", ".join(result["humanitarian"])}')
    print(f'  💥  Damage       : {", ".join(result["damage"])}')
    print(f'  ✅  Verification : {result["verification"]}')
    print(f'  🎯  Confidence   : {result["confidence"] * 100:.1f}%')
    print(f'  📝  R_t (Text)   : {result["r_t"]}')
    print(f'  🖼️   R_v (Image) : {result["r_v"]}')
    print('=' * 55)

    try:
        orig = Image.open(image_path).convert('RGB').resize((224, 224))
    except Exception:
        orig = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

    enc = TOKENIZER(tweet_text, padding='max_length', truncation=True,
                     max_length=cfg.MAX_LEN, return_tensors='pt')
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    pix  = VAL_TRANSFORMS(orig).unsqueeze(0).to(DEVICE)
    cc, wc, ent = extract_text_quality(tweet_text)
    bs, br, co, ns = extract_image_quality(cv2.imread(image_path))
    qt = torch.tensor([[cc, wc, ent]], dtype=torch.float32).to(DEVICE)
    qi = torch.tensor([[bs, br, co, ns]], dtype=torch.float32).to(DEVICE)

    target_head = 'logits_event' if result.get('gated') else 'logits_human'
    target_le   = LABEL_ENCODERS.get('event_type') if result.get('gated') else LABEL_ENCODERS.get('humanitarian')
    try:
        cam, pred_idx = gcam.generate(
            pixel_values=pix, q_text=qt, q_image=qi,
            input_ids=ids, attention_mask=mask,
            target_head=target_head,
        )
        pred_class_name = (target_le.classes_[pred_idx]
                            if target_le is not None and pred_idx < len(target_le.classes_)
                            else str(pred_idx))
    except Exception as e:
        print(f'Grad-CAM failed for sample {i+1}: {e}')
        cam = np.zeros((224, 224), dtype=np.float32)
        pred_class_name = 'N/A'

    overlay = overlay_cam(orig, cam)
    head_label = 'event (gated)' if result.get('gated') else 'humanitarian'

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(orig)
    axes[0].set_title(f'Sample {i+1}: {str(tweet_text)[:40]}...', fontsize=8)
    axes[0].axis('off')

    axes[1].imshow(overlay)
    axes[1].set_title(f'Grad-CAM [{head_label}]\nTop class: {pred_class_name}', fontsize=8)
    axes[1].axis('off')

    fig.suptitle(f'Section 19 — Sample {i+1}/{len(demo_rows)} '
                 f'[{demo_row.get("demo_category", "unspecified")}] Grad-CAM Explanation',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    out_path = cfg.OUT_DIR / f'gradcam_section19_sample{i+1:02d}.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved Grad-CAM for sample {i+1} to {out_path} ✅')

free_memory()

### 24.1 Gated Evaluation Safety Net
`predict_single` returns `'N/A'` for gated samples, which would break `sklearn.metrics` if used directly. This cell remaps gated predictions to a fixed majority-class index (from TRAIN) so arrays stay consistent for `classification_report`. Same mapping used in Section 18's `use_gating` ablation.


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

if 'MAJORITY_IDX' not in dir():
    MAJORITY_IDX = {
        'event' : int(df_train['event_type_idx'].mode()[0])   if 'event_type_idx'   in df_train.columns else 0,
        'human' : int(df_train['humanitarian_idx'].mode()[0]) if 'humanitarian_idx' in df_train.columns else 0,
        'damage': int(df_train['damage_idx'].mode()[0])       if 'damage_idx'       in df_train.columns else 0,
    }
if 'NOT_INFO_IDX' not in dir():
    _le_info_sn = LABEL_ENCODERS.get('informative')
    NOT_INFO_IDX = (list(_le_info_sn.classes_).index('not_informative')
                    if _le_info_sn is not None and 'not_informative' in _le_info_sn.classes_ else None)

@torch.inference_mode()
def gated_batch_predict(model, loader, task='humanitarian', gate=True, gate_conf_thresh=None):
    """Runs the SAME hierarchical-gating rule as eval_ablation (Section 18).

    FIX (2026-07-21): this used to overwrite predictions for every sample
    whose predicted informativeness was 'not_informative', with no confidence
    check -- shown by the Section 18 ablation to tank humanitarian wF1 from
    0.7064 to 0.4684. Now mirrors eval_ablation's confidence-thresholded
    gating: gate_conf_thresh=None (default) means gating is effectively off,
    matching what the ablation shows is best. Pass e.g. 0.95 to only gate
    when the info head is highly confident.
    """
    model.eval()
    task_key    = {'event_type': 'event', 'humanitarian': 'human', 'damage': 'damage'}[task]
    default_idx = MAJORITY_IDX[task_key]

    preds, trues = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        pix  = batch['pixel_values'].to(DEVICE)
        qt   = batch['q_text'].to(DEVICE)
        qi   = batch['q_image'].to(DEVICE)
        out  = model(ids, mask, pix, qt, qi)

        info_probs = F.softmax(out['logits_info'], dim=-1).cpu().numpy()
        info_pred  = info_probs.argmax(axis=-1)
        task_pred  = out[f'logits_{task_key}'].argmax(dim=-1).cpu().numpy()

        if gate and NOT_INFO_IDX is not None and gate_conf_thresh is not None:
            not_info_conf = info_probs[:, NOT_INFO_IDX]
            gated_mask = (info_pred == NOT_INFO_IDX) & (not_info_conf >= gate_conf_thresh)
            task_pred = np.where(gated_mask, default_idx, task_pred)

        preds.extend(task_pred.tolist())
        trues.extend(batch[f'label_{task_key}'].numpy().tolist())

    return np.array(preds), np.array(trues)

print('Demonstrating the metric-mapping fix on the "humanitarian" head (gating OFF by default per Section 18 findings) …')
preds_g, trues_g = gated_batch_predict(MODEL, dl_test, task='humanitarian', gate=True, gate_conf_thresh=None)
target_names = list(LABEL_ENCODERS['humanitarian'].classes_)
print(classification_report(trues_g, preds_g, target_names=target_names, zero_division=0))
print('✅ No crash: gating is confidence-thresholded (off by default) so predictions are '
      'no longer blindly overwritten with the majority class '
      f"({target_names[MAJORITY_IDX['human']]}) -- see Section 18 for the measured effect of "
      "different gating thresholds.")

free_memory()

## 25. Final Experimental Summary

Lists every checkpoint, plot, and metrics file written to `cfg.OUT_DIR` during this run.

In [ ]:
import os
import gc
import torch
import numpy as np

print('Saved Artefacts:', cfg.OUT_DIR)
print('-' * 55)
out_dir = str(cfg.OUT_DIR)
if os.path.isdir(out_dir):
    for fname in sorted(os.listdir(out_dir)):
        fpath = os.path.join(out_dir, fname)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024
            print(f'  {fname:45s}  {size:8.1f} KB')
else:
    print('  Output directory is empty or does not exist yet.')

print('\nNotebook execution complete!')
print('\nFinal Performance Summary:')
print('-' * 55)
for key, meta in TASK_REPORTS.items():
    print(f'  {meta["title"]:25s} -> wF1 = {meta["wf1"]:.4f}')
print(f'  {"Mean Confidence":25s} -> {float(np.mean(all_conf)):.4f}')
print(f'  {"Best Val wF1":25s} -> {best_val_f1:.4f}')

gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass

### 25.1 Artifact Export

This cell persists everything Notebook 2 needs to resume the pipeline from Section 21 onward, using **plain, targeted files only** (`.pt`, `.pkl`, `.csv`, `.json`, `.npz`, `.npy`) — no `dill`, no whole-session serialization, no pickled `DataLoader`/iterator objects (which is what broke the previous approach: `_MultiProcessingDataLoaderIter cannot be pickled`).

Notebook 2 does **not** try to unpickle a saved `DataLoader`. Instead it rebuilds `ds_train`, `ds_dev`, `ds_test`, and `dl_test` itself (cheap, no training) from the saved dataframes, and reconstructs `MODEL` by re-defining the exact same architecture classes and loading the saved `best_model.pt` state dict — the standard, robust way to move a trained PyTorch model between notebooks.

In [ ]:
# ==== SECTION 20.5 — Save Artefacts for Notebook 2 (no dill / no full-session pickling) ====
import os, shutil, json, pickle
import numpy as np
import pandas as pd
import torch

RESULTS_DIR = 'results'
CKPT_SAVE_DIR    = os.path.join(RESULTS_DIR, 'checkpoints')
DATA_SAVE_DIR    = os.path.join(RESULTS_DIR, 'data')
METRICS_SAVE_DIR = os.path.join(RESULTS_DIR, 'metrics')
PRED_SAVE_DIR    = os.path.join(RESULTS_DIR, 'predictions')

for _d in (RESULTS_DIR, CKPT_SAVE_DIR, DATA_SAVE_DIR, METRICS_SAVE_DIR, PRED_SAVE_DIR):
    os.makedirs(_d, exist_ok=True)

def _safe_pickle(obj, path, label):
    """Pickle a plain data object (dict/list/DataFrame/ndarray/LabelEncoder, etc.) with the
    standard library `pickle` -- never a live model, DataLoader, or other object holding open
    iterators/threads/CUDA handles, which is exactly what made dump_session() fail before."""
    try:
        with open(path, 'wb') as f:
            pickle.dump(obj, f)
        print(f'✅ Saved {label} -> {path}')
    except Exception as e:
        print(f'⚠️  Could not pickle {label} ({e}) -- skipping.')

# ---- 1) Model checkpoint: STATE DICT ONLY (never the live nn.Module / optimizer / loader) ----
if os.path.exists(cfg.CKPT_PATH):
    shutil.copy(cfg.CKPT_PATH, os.path.join(CKPT_SAVE_DIR, 'best_model.pt'))
    print(f'✅ Copied best checkpoint (state_dict) -> {CKPT_SAVE_DIR}/best_model.pt')
else:
    print(f'⚠️  No checkpoint found at {cfg.CKPT_PATH} -- skipping explicit checkpoint copy.')

# ---- 2) Dataframes ----
for _name in ['df_train', 'df_dev', 'df_test']:
    if _name in globals():
        globals()[_name].to_pickle(os.path.join(DATA_SAVE_DIR, f'{_name}.pkl'))
        print(f'✅ Saved {_name} -> {DATA_SAVE_DIR}/{_name}.pkl')

# ---- 3) Label encoders (saved for reference; Notebook 2's cells don't require them) ----
if 'LABEL_ENCODERS' in globals():
    _safe_pickle(LABEL_ENCODERS, os.path.join(DATA_SAVE_DIR, 'label_encoders.pkl'), 'LABEL_ENCODERS')

# ---- 4) Training history (needed by the Section 22 dynamic-loss-weight plot) ----
if 'HISTORY' in globals():
    _safe_pickle(HISTORY, os.path.join(DATA_SAVE_DIR, 'history.pkl'), 'HISTORY')

if 'best_val_f1' in globals():
    with open(os.path.join(DATA_SAVE_DIR, 'best_val_f1.json'), 'w') as f:
        json.dump({'best_val_f1': float(best_val_f1)}, f)
    print(f'✅ Saved best_val_f1.json')

# ---- 5) Evaluation metrics / reports (saved for reference / inspection) ----
if 'df_metrics' in globals():
    df_metrics.to_csv(os.path.join(METRICS_SAVE_DIR, 'df_metrics.csv'))
    print(f'✅ Saved df_metrics.csv')
if 'TASK_REPORTS' in globals():
    _safe_pickle(TASK_REPORTS, os.path.join(METRICS_SAVE_DIR, 'task_reports.pkl'), 'TASK_REPORTS')
if 'HEAD_META' in globals():
    _safe_pickle(HEAD_META, os.path.join(METRICS_SAVE_DIR, 'head_meta.pkl'), 'HEAD_META')
if 'ablation_results' in globals():
    _safe_pickle(ablation_results, os.path.join(METRICS_SAVE_DIR, 'ablation_results.pkl'), 'ablation_results')

# ---- 6) Raw predictions / probabilities / uncertainty scores (test set; saved for reference -- ----
# ---- Notebook 2 recomputes test_results/test_hooks itself via run_evaluation_with_hooks) ----
_pred_arrays = {}
for _name in ['all_preds', 'all_trues']:
    if _name in globals():
        for k, v in globals()[_name].items():
            _pred_arrays[f'{_name}_{k}'] = np.array(v)
for _name in ['all_conf', 'all_rt', 'all_rv']:
    if _name in globals():
        _pred_arrays[_name] = np.array(globals()[_name])
if _pred_arrays:
    np.savez(os.path.join(PRED_SAVE_DIR, 'test_predictions.npz'), **_pred_arrays)
    print(f'✅ Saved test_predictions.npz')

if 'all_probs' in globals():
    for k, v in all_probs.items():
        if v is not None:
            np.save(os.path.join(PRED_SAVE_DIR, f'probs_{k}.npy'), v)
    print(f'✅ Saved probs_*.npy')

if 'test_hooks' in globals():
    _safe_pickle(test_hooks, os.path.join(PRED_SAVE_DIR, 'test_hooks.pkl'), 'test_hooks')
if 'test_results' in globals():
    _safe_pickle(test_results, os.path.join(PRED_SAVE_DIR, 'test_results.pkl'), 'test_results')

print('\n' + '=' * 70)
print('  All Notebook-1 artifacts saved under ./results/  (plain files only, no dill)')
print('  REQUIRED by Notebook 2:')
print('  -> results/checkpoints/best_model.pt')
print('  -> results/data/{df_train,df_dev,df_test}.pkl')
print('  -> results/data/history.pkl')
print('  Saved for reference only (not loaded by Notebook 2):')
print('  -> results/data/label_encoders.pkl, best_val_f1.json')
print('  -> results/metrics/{df_metrics.csv, task_reports.pkl, head_meta.pkl, ablation_results.pkl}')
print('  -> results/predictions/{test_predictions.npz, probs_*.npy, test_hooks.pkl, test_results.pkl}')
print('=' * 70)
print('\nOn Kaggle: click "Save Version" so ./results/ persists as this notebook\'s Output,')
print('then attach it as an input Dataset to Notebook 2.')


## 26. Experimental Reproduction Status

| Component | Status |
|---|---|
| Dataset | Existing CrisisMMD setup |
| Train/dev/test split | Preserved |
| Main architecture | Preserved |
| DeBERTa-v3 | Preserved |
| CLIP ViT-B/32 | Preserved |
| E-REM | Preserved/corrected |
| UASG | Preserved |
| UG-CMA | Preserved |
| Existing saved results | Preserved |
| New training in this environment | Not executed |
| Multi-seed experiments (seeds 43, 44) | Not executed |
| Conventional cross-attention ablation | Not executed (ready-to-run scaffold only) |
| Leakage checks | Not executed (ready-to-run code only) |

**Why:** this audit/upgrade session has no GPU, no CrisisMMD dataset, no DeBERTa-v3/CLIP
model weights, and no `torch`/`transformers` packages installed — only the original
`.ipynb` file was available. Static validation (syntax, imports, cross-cell variable
dependencies, attribute references) was completed on every cell; full model execution
requires the CrisisMMD dataset, model weights, and a compatible PyTorch/GPU environment.
Nothing above was estimated, simulated, or numerically approximated to stand in for a real run.

**How to read this notebook:** it is a **journal-ready experimental notebook scaffold with
preserved prior executed results and corrected/reproducible code for future regeneration** —
not a fully experimentally validated journal submission. The corrected uncertainty-loss
objective, the leakage checks, and the conventional-cross-attention ablation must all be run
on a GPU with the real dataset before any claim depending on them can be made.


## Journal Audit Report (final)

**1. Bugs found**
- `loss_uncert = torch.log(sigma2).mean()` in the dynamic multi-task loss (Section 8/12): no
  error term, so gradient descent pushes `sigma2` toward its lower clamp regardless of actual
  reliability — a degenerate heteroscedastic-uncertainty objective.
- `w/o UG-CMA` ablation replaces UG-CMA with a naive `h_t_prime + h_v_prime` sum, which alone
  cannot isolate the value of *uncertainty-guided* attention specifically (confirmed by reading
  the `no_bcmf` branch of `CrisisMultiModal.forward`).
- Two evaluation bugs were **already found and fixed by a previous author** before this session
  (documented in the original notebook's own Section 15 "Fix note" cell): an ablation-table
  extras/structure mixup, and a verification-label/proxy mismatch. Verified both fixes are
  correctly implemented by reading the code — no further action needed.
- Hierarchical gating bug (unconditional majority-class overwrite) was **already found and
  fixed by a previous author** (documented in the original notebook's Section 18 "Fix
  (2026-07-21)" cell, humanitarian wF1 0.7064→0.4684 under the bug). Verified the fix
  (confidence-thresholded, off by default) is correctly implemented.

**2. Bugs fixed (this session)**
- `loss_uncert` removed from backprop; E-REM's `loss_erem` (EDL loss, which already has a
  proper error + variance + KL term) kept as the sole uncertainty training signal. Diagnostic
  logging of `sigma2`/`loss_uncert` preserved but disconnected from gradients.

**3. Scientific issues found (not fixable without retraining)**
- The existing `outputs['uncertainty']`/`sigma2` values in every saved result in this notebook
  come from a checkpoint trained *with* the old degenerate loss — their reliability as an
  uncertainty signal is unverified. Note this is a **different tensor** from E-REM's `u_t`/`u_v`
  (which were checked separately and do show meaningful spread, std≈0.13–0.14, not collapsed).
- `w/o UG-CMA` ablation (naive sum) is a valid lower bound but not, on its own, proof that the
  *uncertainty-guided attention* mechanism specifically matters — a conventional (uncertainty-
  blind) cross-attention comparison is the scientifically complete claim and does not yet exist.
- Ablations are inference-time module bypass on one trained checkpoint, not retrained-from-
  scratch variants, despite config flags (`RUN_GENUINE_ABLATION`, `ABLATION_RETRAIN_EPOCHS`)
  suggesting retraining was intended.
- Multi-seed results do not exist (only seed 42 was run) despite `ABLATION_SEEDS=[42,43,44]`
  being configured.

**4. Changes made**
- Added split-integrity/leakage-check cell (Section 2.2, new).
- Fixed and documented the uncertainty-loss collapse bug (Section 8/12).
- Added ablation-fairness audit note + conventional-cross-attention scaffold (Section 18).
- Added ablation-methodology + multi-seed honesty note (Section 18).
- Added this methodology/limitations preamble and this final audit report.
- No dataset, split, labels, or main architecture were changed. DeBERTa-v3 and CLIP ViT-B/32
  are unchanged. No new architectural block was added to the main pipeline (the conventional
  cross-attention module is an ablation-only scaffold, never wired into the deployed model).

**5–7. New experiments / ablations / metrics**
- None were *executed* — no GPU/dataset access in this session. New code was *written and
  documented* (leakage checks, conventional cross-attention ablation) but is explicitly marked
  `NOT EXECUTED` and must be run by you before it can be cited as a result.

**8. New figures**
- None generated in this session (no execution capability). All figures reused from the
  original notebook's genuine prior run and are unchanged.

**9. Was the uncertainty loss changed?**
- Yes — removed from backprop, replaced by E-REM's existing EDL loss as sole uncertainty
  objective. Not yet re-validated by retraining (requires GPU/dataset access this session
  didn't have).

**10. Is the UG-CMA comparison now fair?**
- Not yet. The statistically significant `w/o UG-CMA` result (Section 18.2 bootstrap, p<0.001)
  is genuine but only shows "some fusion beats no fusion." The conventional-cross-attention
  scaffold added this session, once trained and evaluated, would complete this claim — it has
  not been run.

**11. Which experiments were actually retrained?**
- None, in this session (no execution capability). In the original notebook's prior run: only
  the single main model (seed 42) was trained; ablations were inference-time bypasses on that
  one checkpoint, not retrained.

**12. Number of seeds actually executed**
- 1 (seed 42). Despite `ABLATION_SEEDS=[42,43,44]` in config, no seed-43/44 run exists anywhere
  in the notebook's source or saved outputs (verified by exhaustive text search).

**13. Old results that remain valid**
- All existing Section 15 test-set metrics, confusion matrices, per-task F1, Grad-CAM,
  t-SNE, and the Section 18.2 statistical-significance bootstrap — these reflect the actual
  prior run and were not touched.
- The real (non-proxy) cross-modal verification label construction (Section 2) was traced and
  confirmed correct: built from independent text-side/image-side informativeness agreement,
  `VERIF_IS_REAL=True` in this run, not an algebraic proxy.

**14. Old results that need regeneration**
- Anything using `outputs['uncertainty']`/`sigma2` (as opposed to E-REM's `u_t`/`u_v`), once
  the loss fix is retrained, to confirm the fix actually changes its behavior.
- The full ablation + significance table, once a genuinely retrained-ablation and/or
  conventional-cross-attention variant exists, for a complete UG-CMA claim.

**15. Experiments not executed and why**
- Leakage checks, conventional-cross-attention ablation, any retraining, and all multi-seed
  runs: this audit session had no GPU, no `torch`, no network access to model-weight hosts, and
  no copy of the CrisisMMD dataset (only the `.ipynb` file was uploaded).

**16. Does the new notebook run end-to-end?**
- Structurally yes for everything reused from the original (same code, same dependencies) —
  but this was not verified by execution in this session. The three new/modified pieces
  (leakage-check cell, the loss fix, the cross-attention scaffold) are unexecuted and should be
  reviewed/run first, since untested code can contain ordinary bugs.

**17. Remaining limitations**
- No independent re-execution of anything in this notebook was possible in this session.
- The uncertainty-loss fix is a code-level correction backed by clear math, not a validated
  improvement — validate by retraining and comparing `sigma2` spread and calibration before/after.
- Fair UG-CMA attribution and multi-seed robustness remain open; both require GPU time this
  session did not have.


### Notebook Cleanup Report

- Markdown sections corrected: all 26 top-level/sub-level headers retitled to the requested
  numbered journal structure (Section Map cell documents the one execution-order exception).
- Model terminology standardized: DeBERTa-v3, CLIP ViT-B/32, E-REM, UASG, InfoNCE, UG-CMA,
  CrisisMMD spelled/capitalized consistently across markdown; full names given at first mention.
- Dataset terminology standardized: task names (Event Type, Informativeness, Humanitarian,
  Damage Severity, Cross-Modal Verification/Agreement) and metric names (Accuracy, Macro
  Precision, Macro Recall, Macro-F1, Weighted-F1, MCC, Cohen's Kappa, ROC-AUC, PR-AUC) used
  consistently in headings; only cosmetic (markdown) text was touched, not functional code.
- NameError risks checked: automated cross-cell name-definition audit run over all 55 code
  cells; every flag reviewed by hand and found to be a false positive of the (deliberately
  simple, scope-blind) checker -- function parameters (self, dim, dropout, ...), exception
  variables (e), and comprehension/loop variables. No genuine undefined-variable bug found
  beyond the uncertainty-loss fix already made in the previous pass.
- AttributeError risks checked: model/tokenizer/dataset/dataframe attribute usages spot-checked
  in the training loop, evaluation loop, checkpoint loading, ablation, calibration, Grad-CAM,
  and t-SNE cells; no incorrect attribute references found.
- Cross-cell dependencies checked: confirmed the notebook's variable definitions all precede
  their uses in execution order, with one documented exception in the Section Map (calibration
  bias must be computed before the ablation study's post-hoc table uses it).
- Device handling checked: hardware-configuration cell already branches correctly on
  torch.cuda.is_available(); no unconditional .cuda() / hard-coded torch.device("cuda")
  calls found anywhere in the notebook. Added the required CPU-unavailable message.
- CUDA-specific assumptions removed/guarded where found; device-configuration cell already
  branches on `torch.cuda.is_available()` and multi-GPU status is now explicitly reported.

**This session's additions (static, no GPU/dataset access -- same constraint as prior session)**
- Added `gpu_memory_report()`: prints allocated / reserved / total memory **per GPU**, since a
  2xT4 pair is two independent ~15 GB devices, not one pooled 32 GB device.
- Added an explicit "Batch / Multi-GPU Configuration" printout (per-device batch size, gradient
  accumulation steps, GPUs used, multi-GPU active flag, and the computed effective global batch
  size = BATCH_SIZE x GRAD_ACCUM_STEPS x N_GPUS). `BATCH_SIZE=32` and `GRAD_ACCUM_STEPS=2` were
  already configurable fields on `CFG` (not hardcoded magic numbers) -- this only makes the
  resulting effective batch size and multi-GPU status visible at run time, as required for a
  reproducible 2xT4 Kaggle run.
- Verified `nn.DataParallel` is already applied correctly (`if N_GPUS > 1: MODEL = nn.DataParallel(MODEL)`)
  and that all downstream code correctly unwraps `.module` where needed -- no change made, only
  confirmed by reading the code.
- Verified the Grad-CAM evaluation cell's `DataLoader(..., batch_size=len(eval_df))` loads a
  bounded, configurable number of raw images (`GRADCAM_N_EVAL_SAMPLES`, default 100) as plain
  tensors, then runs the actual forward/backward Grad-CAM computation one sample at a time
  (`single = {k: v[i:i+1] ...}`) -- this is not a full-evaluation-set-as-one-batch pattern and
  is not a realistic OOM risk on a T4; no change made.
- Full syntax check re-run over all 55 code cells after edits: no syntax errors.
- All edits in this pass are confined to two diagnostic-printing code cells (Section 1.2, 1.3)
  and this audit cell; no architecture, dataset, split, label, encoder, loss, or reported-result
  cell was modified.emoved: none were present beyond the already-conditional
  USE_AMP = N_GPUS > 0 and DataParallel (only invoked when N_GPUS > 1).
- Visualization cells checked: existing plotting cells already read from real result
  dictionaries/label encoders (spot-checked Sections 17, 18, 19, 20, 22); no hard-coded or
  placeholder class names or dummy data found.
- Existing results preserved: no saved output, figure, or metric from the original notebook
  was altered, deleted, or overwritten.
- Fabricated results: **NONE.**

**Cells that still require actual GPU/CrisisMMD-dataset execution before they can be cited
as results:**
- Section 2.2 -- Split Integrity and Leakage Checks
- Section 12.1 / Section 13 -- retraining with the corrected uncertainty-loss objective
- Section 21.1 -- the conventional cross-attention ablation scaffold (fine-tuning + evaluation)
- Any retrained (as opposed to inference-time-bypass) version of the Section 21 ablations
- Any multi-seed (43, 44) repetition of training, ablations, or significance testing
